### CELL 1: Mount Google Drive, Import Thư viện & Kiểm tra RAM


In [1]:
# ==============================================================================
# CELL 1: MOUNT GOOGLE DRIVE, IMPORT THƯ VIỆN CƠ BẢN & KIỂM TRA TÀI NGUYÊN RAM
# ==============================================================================
# Giải thích:
# - Kết nối Google Colab với Google Drive tại đường dẫn '/content/drive'.
# - Khởi tạo các thư viện hệ thống, xử lý dữ liệu (NumPy, Pandas, SciPy/Math)
#   và vẽ biểu đồ (Matplotlib).
# - Kiểm tra dung lượng RAM khả dụng của máy ảo Colab trước khi nạp bảng dữ liệu lớn.

from __future__ import annotations

import gc
import hashlib
import json
import os
import pickle
import resource
import shutil
import sys
import warnings
from math import erfc, sqrt
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import sklearn

# Kết nối Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Tắt các cảnh báo không cần thiết từ LightGBM / Scikit-learn để log đầu ra gọn gàng
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")

print("===== THÔNG TIN MÔI TRƯỜNG & THƯ VIỆN =====")
print("Python      :", sys.version.split()[0])
print("Pandas      :", pd.__version__)
print("NumPy       :", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib      :", joblib.__version__)

ram_info = psutil.virtual_memory()
print("\n===== THÔNG TIN BỘ NHỚ RAM COLAB =====")
print(f"Tổng RAM     : {ram_info.total / (1024**3):.2f} GB")
print(f"RAM khả dụng : {ram_info.available / (1024**3):.2f} GB")
print(f"Đang sử dụng : {ram_info.percent}%")

Mounted at /content/drive
===== THÔNG TIN MÔI TRƯỜNG & THƯ VIỆN =====
Python      : 3.13.18
Pandas      : 2.2.3
NumPy       : 2.4.6
Scikit-learn: 1.6.1
Joblib      : 1.6.0

===== THÔNG TIN BỘ NHỚ RAM COLAB =====
Tổng RAM     : 12.67 GB
RAM khả dụng : 11.16 GB
Đang sử dụng : 11.9%


### CELL 2: Khai báo Đường dẫn Chuẩn theo Đích (#scrollTo=JivyMUFd9eOg)


In [2]:
# ==============================================================================
# CELL 2: KHAI BÁO ĐƯỜNG DẪN CHUẨN THEO ĐỊNH DẠNG NOTEBOOK ĐÍCH (#scrollTo=JivyMUFd9eOg)
# ==============================================================================
# Giải thích:
# - Đồng bộ 100% cấu trúc đường dẫn với file đích:
#     PROJECT_DIR            = /content/drive/MyDrive/IEEE_CIS_Project
#     CODE_DIR               = /content/drive/MyDrive/IEEE_CIS_Project/ieee_cis_code
#     OUTPUT_DIR             = /content/drive/MyDrive/IEEE_CIS_Project/artifacts
#     TRAIN_TRANSACTION_PATH = /content/drive/MyDrive/IEEE_CIS_Project/data/train_transaction.csv
#     TRAIN_IDENTITY_PATH    = /content/drive/MyDrive/IEEE_CIS_Project/data/train_identity.csv
# - Tích hợp thêm cơ chế tự động kiểm tra đường dẫn dự phòng (fallback) sang
#   '/content/drive/MyDrive/Colab Notebooks/data_sample_DA/' nếu chạy trên tài khoản
#   Drive lưu dữ liệu ở thư mục cũ, giúp tránh lỗi FileNotFoundError.

PROJECT_DIR = Path("/content/drive/MyDrive/IEEE_CIS_Project")

CODE_DIR = PROJECT_DIR / "ieee_cis_code"
OUTPUT_DIR = PROJECT_DIR / "artifacts"

TRAIN_TRANSACTION_PATH = PROJECT_DIR / "data" / "train_transaction.csv"
TRAIN_IDENTITY_PATH = PROJECT_DIR / "data" / "train_identity.csv"

# Đường dẫn thư mục con lưu kết quả (artifacts) của từng bước Giai đoạn 0 -> 4
STEP0_OUTPUT_DIR = OUTPUT_DIR / "artifacts_step0"
STEP1_OUTPUT_DIR = OUTPUT_DIR / "artifacts_step1_eda"
STEP2_OUTPUT_DIR = OUTPUT_DIR / "artifacts_step2_fe"
STEP34_OUTPUT_DIR = OUTPUT_DIR / "artifacts_step3_4_handover"

# Tự động kiểm tra dự phòng nếu file dữ liệu nằm ở thư mục Colab Notebooks/data_sample_DA
FALLBACK_DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/data_sample_DA")
if not TRAIN_TRANSACTION_PATH.is_file() and (FALLBACK_DATA_DIR / "train_transaction.csv").is_file():
    print("[Lưu ý] Không tìm thấy file tại IEEE_CIS_Project/data, tự động chuyển sang đường dẫn dự phòng:")
    TRAIN_TRANSACTION_PATH = FALLBACK_DATA_DIR / "train_transaction.csv"
    TRAIN_IDENTITY_PATH = FALLBACK_DATA_DIR / "train_identity.csv"

# Tạo sẵn các thư mục lưu kết quả đầu ra trên Google Drive
for folder in [OUTPUT_DIR, STEP0_OUTPUT_DIR, STEP1_OUTPUT_DIR, STEP2_OUTPUT_DIR, STEP34_OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("===== CẤU HÌNH ĐƯỜNG DẪN DỰ ÁN =====")
print("PROJECT_DIR           :", PROJECT_DIR)
print("CODE_DIR              :", CODE_DIR)
print("OUTPUT_DIR            :", OUTPUT_DIR)
print("TRAIN_TRANSACTION_PATH:", TRAIN_TRANSACTION_PATH)
print("TRAIN_IDENTITY_PATH   :", TRAIN_IDENTITY_PATH)

===== CẤU HÌNH ĐƯỜNG DẪN DỰ ÁN =====
PROJECT_DIR           : /content/drive/MyDrive/IEEE_CIS_Project
CODE_DIR              : /content/drive/MyDrive/IEEE_CIS_Project/ieee_cis_code
OUTPUT_DIR            : /content/drive/MyDrive/IEEE_CIS_Project/artifacts
TRAIN_TRANSACTION_PATH: /content/drive/MyDrive/IEEE_CIS_Project/data/train_transaction.csv
TRAIN_IDENTITY_PATH   : /content/drive/MyDrive/IEEE_CIS_Project/data/train_identity.csv


### CELL 3: Kiểm tra, Đọc Dữ liệu Gốc & Ghép Bảng (Transaction + Identity)


In [ ]:
# ==============================================================================
# CELL 3: KIỂM TRA, ĐỌC DỮ LIỆU GỐC VÀ GHÉP BẢNG (TRANSACTION + IDENTITY)
# ==============================================================================
# Giải thích:
# - Đọc 2 bảng dữ liệu gốc:
#     1. train_transaction.csv (590,540 dòng x 394 cột): chứa thông tin giao dịch và nhãn isFraud.
#     2. train_identity.csv    (144,233 dòng x 41 cột) : chứa thông tin thiết bị, mạng, định danh.
# - Ghép (Left Join) bảng Identity vào bảng Transaction theo khóa 'TransactionID'
#   để giữ nguyên toàn bộ 590,540 giao dịch gốc.
# - Xóa ngay 2 bảng trung gian (tr_trans, tr_id) và gọi gc.collect() để giải phóng ~2.5 GB RAM.

if not TRAIN_TRANSACTION_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy transaction file: {TRAIN_TRANSACTION_PATH}")

if not TRAIN_IDENTITY_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy identity file: {TRAIN_IDENTITY_PATH}")

print("Đang đọc train_transaction.csv...")
tr_trans = pd.read_csv(TRAIN_TRANSACTION_PATH, low_memory=False)

print("Đang đọc train_identity.csv...")
tr_id = pd.read_csv(TRAIN_IDENTITY_PATH, low_memory=False)

print("\n========== DATA INFORMATION ==========")
print("Transaction shape:", tr_trans.shape)
print("Identity shape   :", tr_id.shape)

# Left Join bảng Identity vào bảng Transaction theo TransactionID
print("\nĐang ghép dữ liệu (Left Merge theo TransactionID)...")
train_df = tr_trans.merge(
    tr_id,
    on="TransactionID",
    how="left",
)

print("Merged train shape          :", train_df.shape)
print("Unique TransactionID        :", train_df["TransactionID"].nunique())
print("DeviceType non-missing rate :", round(train_df["DeviceType"].notna().mean() * 100, 2), "%")

# Giải phóng ngay 2 bảng thành phần để tiết kiệm RAM trên Colab
del tr_trans, tr_id
gc.collect()
print("Đã giải phóng bộ nhớ của tr_trans và tr_id.")

Đang đọc train_transaction.csv...


### CELL 4: Định nghĩa các Hàm Bước 0 (step0_setup_and_split.py)


In [ ]:
# ==============================================================================
# CELL 4: ĐỊNH NGHĨA CÁC HÀM BƯỚC 0 (CHUẨN HÓA SCHEMA, HẠ KIỂU RAM & CHIA TẬP)
#         (Chuyển đổi từ file: step0_setup_and_split.py)
# ==============================================================================
# Giải thích chuyên môn:
# 1. Đo bộ nhớ (get_process_rss_mb, get_df_memory_mb):
#    Theo dõi sát lượng RAM thực tế của tiến trình (RSS / Peak RSS) và của DataFrame.
# 2. Định nghĩa Schema gốc (get_ieee_cis_canonical_schema):
#    Khóa chặt danh sách 432 biến dự báo gốc gồm 383 biến số (Numerical) và 49 biến phân loại
#    (Categorical - tính cả card1..card6, addr1..addr2 dù lưu dưới dạng mã số).
# 3. Trích xuất trước khi ép kiểu (extract_pre_downcast_features):
#    - Đổi tên cột id-01..id-38 -> id_01..id_38 cho đồng nhất.
#    - Tạo cờ 'has_identity' (1 nếu giao dịch có thông tin trong bảng Identity, 0 nếu không).
#    - Tách phần xu lẻ 'Amt_cents' (0..99) từ TransactionAmt ngay khi còn ở kiểu float64
#      (nếu ép xuống float32 trước rồi mới nhân 100 sẽ bị sai số dấu phẩy động).
# 4. Hạ kiểu bộ nhớ an toàn (downcast_dataframe_safely):
#    - Chuyển float64 -> float32 (kiểm tra sai số tương đối < 1e-6 và không làm mất NaN).
#    - Chuyển int64 -> int8/int16/int32 theo miền min/max thực tế, giúp giảm ~62% RAM.
# 5. Chia tập an toàn theo thời gian (create_timestamp_safe_splits):
#    - Chia 80% Train / 20% Validation theo TransactionDT nhưng gom toàn bộ giao dịch có cùng
#      giây (timestamp ties) tại ranh giới về cùng một phía để chống rò rỉ thời gian.
#    - Tạo thêm bảng phân tích độ nhạy Gap (0, 1, 7, 14 ngày) và 5 Inner Temporal Folds.


# ------------------------------------------------------------------------------
# 4.1. TIỆN ÍCH ĐO BỘ NHỚ TIẾN TRÌNH (PROCESS RSS / PEAK RSS & DATAFRAME MEMORY)
# ------------------------------------------------------------------------------
def get_process_rss_mb() -> Dict[str, float]:
    """
    Đo lượng RAM thực tế của tiến trình (Current RSS và Peak RSS tính bằng MB).
    Tương thích mọi môi trường (dùng psutil nếu có, hoặc đọc trực tiếp từ /proc/self/status & resource).
    """
    current_rss_mb = 0.0
    peak_rss_mb = 0.0

    try:
        proc = psutil.Process(os.getpid())
        current_rss_mb = proc.memory_info().rss / (1024 ** 2)
    except Exception:
        status_path = Path("/proc/self/status")
        if status_path.exists():
            for line in status_path.read_text().splitlines():
                if line.startswith("VmRSS:"):
                    current_rss_mb = float(line.split()[1]) / 1024.0
                elif line.startswith("VmHWM:"):
                    peak_rss_mb = float(line.split()[1]) / 1024.0

    ru_max = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    divisor = (1024.0 ** 2) if sys.platform == "darwin" else 1024.0
    peak_rss_mb = max(peak_rss_mb, float(ru_max) / divisor, current_rss_mb)

    return {
        "current_rss_mb": round(current_rss_mb, 2),
        "peak_rss_mb": round(peak_rss_mb, 2),
    }


def get_df_memory_mb(df: pd.DataFrame) -> float:
    """Trả về dung lượng bộ nhớ sâu (deep=True) của DataFrame tính bằng MB."""
    return float(df.memory_usage(deep=True).sum() / (1024 ** 2))


# ------------------------------------------------------------------------------
# 4.2. ĐỊNH NGHĨA CHÍNH XÁC SCHEMA 432 BIẾN GỐC (383 NUMERICAL + 49 CATEGORICAL)
# ------------------------------------------------------------------------------
def get_ieee_cis_canonical_schema() -> Tuple[List[str], List[str]]:
    """
    Khởi tạo danh sách chuẩn 383 biến số (Numerical) và 49 biến phân loại (Categorical)
    theo đúng đặc tả chính thức của cuộc thi IEEE-CIS Fraud Detection.
    """
    # 383 biến Numerical:
    # 2 (TransactionDT, TransactionAmt) + 2 (dist1, dist2) + 14 (C1-C14)
    # + 15 (D1-D15) + 339 (V1-V339) + 11 (id_01-id_11) = 383
    num_cols: List[str] = (
        ["TransactionDT", "TransactionAmt", "dist1", "dist2"]
        + [f"C{i}" for i in range(1, 15)]
        + [f"D{i}" for i in range(1, 16)]
        + [f"V{i}" for i in range(1, 340)]
        + [f"id_{i:02d}" for i in range(1, 12)]
    )

    # 49 biến Categorical (bao gồm cả các mã số thực chất là định danh nhóm như card1-6, addr1-2):
    # 1 (ProductCD) + 6 (card1-card6) + 2 (addr1, addr2) + 2 (P/R_emaildomain)
    # + 9 (M1-M9) + 27 (id_12-id_38) + 2 (DeviceType, DeviceInfo) = 49
    cat_cols: List[str] = (
        ["ProductCD"]
        + [f"card{i}" for i in range(1, 7)]
        + ["addr1", "addr2", "P_emaildomain", "R_emaildomain"]
        + [f"M{i}" for i in range(1, 10)]
        + [f"id_{i:02d}" for i in range(12, 39)]
        + ["DeviceType", "DeviceInfo"]
    )

    assert len(num_cols) == 383, f"Lỗi Schema: Số biến Numerical phải là 383, hiện tại là {len(num_cols)}"
    assert len(cat_cols) == 49, f"Lỗi Schema: Số biến Categorical phải là 49, hiện tại là {len(cat_cols)}"
    assert len(set(num_cols).intersection(set(cat_cols))) == 0, "Lỗi Schema: Có biến trùng giữa Numerical và Categorical"

    return num_cols, cat_cols


# ------------------------------------------------------------------------------
# 4.3. TÁCH PHẦN XU LẺ (AMT_CENTS) TRÊN FLOAT64 & TẠO CỜ HAS_IDENTITY GỐC
# ------------------------------------------------------------------------------
def extract_pre_downcast_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Thực hiện các phép trích xuất bắt buộc TRƯỚC KHI ép kiểu bộ nhớ (downcast):
    1. Chuẩn hóa tên cột id-NN -> id_NN (nếu đọc từ file CSV gốc chưa đổi tên).
    2. Tạo cờ has_identity (0/1) trực tiếp từ sự hiện diện của 40 trường bảng Identity.
    3. Tách phần xu lẻ Amt_cents (0..99) từ TransactionAmt khi còn ở độ chính xác float64.
    """
    rename_map = {f"id-{i:02d}": f"id_{i:02d}" for i in range(1, 39) if f"id-{i:02d}" in df.columns}
    if rename_map:
        df = df.rename(columns=rename_map)

    identity_cols = [f"id_{i:02d}" for i in range(1, 39)] + ["DeviceType", "DeviceInfo"]
    present_id_cols = [c for c in identity_cols if c in df.columns]

    # Tạo cờ has_identity = 1 nếu dòng giao dịch có khớp bản ghi Identity (ít nhất 1 cột Identity khác NaN)
    df["has_identity"] = df[present_id_cols].notna().any(axis=1).astype("int8")

    # Tách phần xu lẻ (0..99) trên kiểu float64 gốc để tránh sai số dấu phẩy động của float32
    amt_f64 = df["TransactionAmt"].to_numpy(dtype="float64")
    cents_int64 = np.round(amt_f64 * 100.0).astype("int64")
    df["Amt_cents"] = (cents_int64 % 100).astype("int8")

    return df


# ------------------------------------------------------------------------------
# 4.4. HẠ KIỂU BỘ NHỚ (MEMORY DOWNCASTING) CÓ KIỂM TRA MIỀN GIÁ TRỊ & SAI SỐ
# ------------------------------------------------------------------------------
def downcast_dataframe_safely(
    df: pd.DataFrame,
    num_cols: List[str],
    cat_cols: List[str],
    max_rel_error_tol: float = 1e-6,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """
    Hạ kiểu dữ liệu có kiểm soát:
    - Kiểm tra miền min/max trước khi hạ kiểu số nguyên (int64 -> int8/int16/int32).
    - Kiểm tra sai số tương đối trước khi hạ kiểu số thực (float64 -> float32).
    - Không ép các mã phân loại dạng số (card1-6, addr1-2) sang float32 nếu làm sai lệch mã nguyên.
    """
    mem_before_mb = get_df_memory_mb(df)
    rss_before = get_process_rss_mb()

    # Kiểm tra riêng sai số khi chuyển TransactionAmt từ float64 sang float32
    amt_f64 = df["TransactionAmt"].to_numpy(dtype="float64")
    amt_f32 = amt_f64.astype("float32")
    valid_mask = np.isfinite(amt_f64) & (amt_f64 > 0)
    max_amt_rel_err = float(np.max(np.abs(amt_f64[valid_mask] - amt_f32[valid_mask]) / amt_f64[valid_mask]))
    assert max_amt_rel_err < max_rel_error_tol, (
        f"Sai số tương đối của TransactionAmt khi sang float32 ({max_amt_rel_err:.2e}) vượt ngưỡng {max_rel_error_tol}"
    )

    # 1. Xử lý các cột khoá và nhãn (TransactionID, isFraud, TransactionDT)
    for int_col in ["TransactionID", "isFraud", "TransactionDT"]:
        if int_col in df.columns and pd.api.types.is_integer_dtype(df[int_col]):
            c_min, c_max = df[int_col].min(), df[int_col].max()
            for dtype_candidate in [np.int8, np.int16, np.int32, np.int64]:
                info = np.iinfo(dtype_candidate)
                if c_min >= info.min and c_max <= info.max:
                    df[int_col] = df[int_col].astype(dtype_candidate)
                    break

    # 2. Hạ kiểu cho 383 biến Numerical
    for col in num_cols:
        if col not in df.columns or col == "TransactionDT":
            continue
        if pd.api.types.is_float_dtype(df[col]):
            arr_f64 = df[col].to_numpy(dtype="float64")
            arr_f32 = arr_f64.astype("float32")
            assert np.isnan(arr_f64).sum() == np.isnan(arr_f32).sum(), f"Mất NaN ở cột {col}"
            assert np.isinf(arr_f32).sum() == 0, f"Tràn số (Inf) khi ép float32 ở cột {col}"
            df[col] = arr_f32
        elif pd.api.types.is_integer_dtype(df[col]):
            c_min, c_max = df[col].min(), df[col].max()
            for dtype_candidate in [np.int8, np.int16, np.int32]:
                info = np.iinfo(dtype_candidate)
                if c_min >= info.min and c_max <= info.max:
                    df[col] = df[col].astype(dtype_candidate)
                    break

    # 3. Giữ an toàn cho 49 biến Categorical
    for col in cat_cols:
        if col not in df.columns:
            continue
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            n_unique = df[col].nunique(dropna=True)
            if n_unique <= 500:
                df[col] = df[col].astype("category")
        elif pd.api.types.is_float_dtype(df[col]):
            non_na = df[col].dropna().to_numpy(dtype="float64")
            if len(non_na) > 0 and np.array_equal(non_na, non_na.astype("float32").astype("float64")):
                df[col] = df[col].astype("float32")

    mem_after_mb = get_df_memory_mb(df)
    rss_after = get_process_rss_mb()

    stats = {
        "df_memory_before_mb": round(mem_before_mb, 2),
        "df_memory_after_mb": round(mem_after_mb, 2),
        "memory_saved_pct": round((1.0 - mem_after_mb / max(mem_before_mb, 1e-9)) * 100.0, 2),
        "process_rss_before_mb": rss_before["current_rss_mb"],
        "process_rss_after_mb": rss_after["current_rss_mb"],
        "peak_rss_mb": rss_after["peak_rss_mb"],
        "max_amt_rel_err_float32": max_amt_rel_err,
    }
    return df, stats


# ------------------------------------------------------------------------------
# 4.5. CHIA TẬP TIMESTAMP-SAFE TEMPORAL SPLIT, GAP SENSITIVITY & STRATIFIED SPLIT
# ------------------------------------------------------------------------------
def _stratified_split_indices(y: np.ndarray, train_ratio: float = 0.80, random_state: int = 42) -> np.ndarray:
    """
    Trả về mảng vị trí dòng thuộc tập Train theo phương pháp Stratified Split (bảo toàn tỷ lệ lớp).
    """
    try:
        from sklearn.model_selection import StratifiedShuffleSplit
        sss = StratifiedShuffleSplit(n_splits=1, test_size=1.0 - train_ratio, random_state=random_state)
        strat_train_idx, _ = next(sss.split(np.zeros(len(y)), y))
        return strat_train_idx
    except ImportError:
        rng = np.random.default_rng(random_state)
        train_indices = []
        for cls_val in np.unique(y):
            cls_idx = np.flatnonzero(y == cls_val)
            rng.shuffle(cls_idx)
            n_cls_train = int(round(len(cls_idx) * train_ratio))
            train_indices.append(cls_idx[:n_cls_train])
        return np.concatenate(train_indices)


def create_timestamp_safe_splits(
    df: pd.DataFrame,
    train_ratio: float = 0.80,
    gap_days_list: Tuple[int, ...] = (0, 1, 7, 14),
    n_inner_folds: int = 5,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Tạo và kiểm chứng ranh giới chia tập:
    1. Primary Protocol (Temporal Split 80/20): Mọi giao dịch có cùng TransactionDT tại mốc cắt
       đều được gom về cùng một phía (không bao giờ cắt ngang các dòng trùng giây).
    2. Bảng độ nhạy Gap/Embargo (0, 1, 7, 14 ngày trước mốc Validation) để mô phỏng độ trễ nhãn.
    3. Bảng 5 Inner Temporal Folds (phục vụ chọn biến ở Giai đoạn 3).
    4. Secondary Benchmark (Stratified Split 80/20) để đối chiếu với yêu cầu đề cương.
    """
    assert df["TransactionDT"].notna().all(), "Cột TransactionDT không được chứa giá trị khuyết"

    # 1. Xác định mốc cắt thời gian 80% (Timestamp-Safe)
    cut_dt = float(df["TransactionDT"].quantile(train_ratio))
    temporal_train_mask = df["TransactionDT"] <= cut_dt
    temporal_val_mask = df["TransactionDT"] > cut_dt

    # Kiểm chứng tuyệt đối không có bất kỳ giây TransactionDT nào xuất hiện ở cả Train và Val
    train_dts = set(df.loc[temporal_train_mask, "TransactionDT"].unique())
    val_dts = set(df.loc[temporal_val_mask, "TransactionDT"].unique())
    assert train_dts.isdisjoint(val_dts), "LỖI NGHIÊM TRỌNG: Có giao dịch trùng TransactionDT bị xé sang cả Train và Val!"

    # 2. Tạo Stratified Split 80/20 độc lập (chỉ dùng để đối chiếu ở bước đánh giá mô hình)
    strat_train_idx = _stratified_split_indices(
        df["isFraud"].to_numpy(), train_ratio=train_ratio, random_state=random_state
    )
    strat_role = np.full(len(df), "val", dtype=object)
    strat_role[strat_train_idx] = "train"

    split_df = pd.DataFrame(
        {
            "TransactionID": df["TransactionID"].to_numpy(),
            "TransactionDT": df["TransactionDT"].to_numpy(),
            "isFraud": df["isFraud"].to_numpy(dtype="int8"),
            "temporal_role": np.where(temporal_train_mask, "train", "val"),
            "stratified_role": strat_role,
        },
        index=df.index,
    )

    # 3. Bảng phân tích độ nhạy theo Gap / Embargo (0, 1, 7, 14 ngày)
    gap_rows = []
    val_min_dt = float(df.loc[temporal_val_mask, "TransactionDT"].min())
    val_max_dt = float(df.loc[temporal_val_mask, "TransactionDT"].max())
    val_count = int(temporal_val_mask.sum())
    val_fraud = int(df.loc[temporal_val_mask, "isFraud"].sum())

    for gap_d in gap_days_list:
        gap_sec = gap_d * 86400.0
        eff_cut_dt = cut_dt - gap_sec
        tr_mask_gap = df["TransactionDT"] <= eff_cut_dt
        gap_mask = (df["TransactionDT"] > eff_cut_dt) & (df["TransactionDT"] <= cut_dt)

        tr_cnt = int(tr_mask_gap.sum())
        tr_frd = int(df.loc[tr_mask_gap, "isFraud"].sum())
        gap_rows.append(
            {
                "Kich_ban_Gap": f"Gap = {gap_d} ngày",
                "Train_Max_Day": round(eff_cut_dt / 86400.0, 2),
                "Val_Start_Day": round(val_min_dt / 86400.0, 2),
                "Val_End_Day": round(val_max_dt / 86400.0, 2),
                "So_dong_Train": tr_cnt,
                "So_Fraud_Train": tr_frd,
                "Ty_le_Fraud_Train_pct": round(tr_frd / max(tr_cnt, 1) * 100.0, 3),
                "So_dong_trong_Gap": int(gap_mask.sum()),
                "So_dong_Val": val_count,
                "So_Fraud_Val": val_fraud,
                "Ty_le_Fraud_Val_pct": round(val_fraud / max(val_count, 1) * 100.0, 3),
            }
        )
    gap_summary_df = pd.DataFrame(gap_rows)

    # 4. Bảng thống kê 5 Inner Temporal Folds bên trong Temporal Train (Expanding Window)
    train_sub = df.loc[temporal_train_mask, ["TransactionDT", "isFraud"]].sort_values("TransactionDT")
    quantiles = np.linspace(0.0, 1.0, n_inner_folds + 2)
    dt_bounds = [float(train_sub["TransactionDT"].quantile(q)) for q in quantiles]

    fold_rows = []
    for k in range(1, n_inner_folds + 1):
        tr_end_dt = dt_bounds[k]
        va_end_dt = dt_bounds[k + 1]
        f_tr_mask = train_sub["TransactionDT"] <= tr_end_dt
        f_va_mask = (train_sub["TransactionDT"] > tr_end_dt) & (train_sub["TransactionDT"] <= va_end_dt)

        n_tr = int(f_tr_mask.sum())
        f_tr = int(train_sub.loc[f_tr_mask, "isFraud"].sum())
        n_va = int(f_va_mask.sum())
        f_va = int(train_sub.loc[f_va_mask, "isFraud"].sum())

        fold_rows.append(
            {
                "Inner_Fold": f"Fold {k}",
                "Train_Day_Range": f"{dt_bounds[0]/86400:.1f} -> {tr_end_dt/86400:.1f}",
                "Val_Day_Range": f"{tr_end_dt/86400:.1f} -> {va_end_dt/86400:.1f}",
                "Train_Rows": n_tr,
                "Train_Fraud": f_tr,
                "Train_Fraud_pct": round(f_tr / max(n_tr, 1) * 100.0, 3),
                "Val_Rows": n_va,
                "Val_Fraud": f_va,
                "Val_Fraud_pct": round(f_va / max(n_va, 1) * 100.0, 3),
            }
        )
    inner_folds_df = pd.DataFrame(fold_rows)

    return split_df, gap_summary_df, inner_folds_df


# ------------------------------------------------------------------------------
# 4.6. HÀM CHẠY TRỌN GÓI BƯỚC 0 (PIPELINE ENTRYPOINT)
# ------------------------------------------------------------------------------
def run_step0_pipeline(
    train_df: pd.DataFrame,
    output_dir: str = str(STEP0_OUTPUT_DIR),
) -> Dict[str, object]:
    """
    Thực thi trọn gói Bước 0 trên bảng dữ liệu Train đã ghép:
    1. Kiểm định Schema 432 biến (383 Numerical + 49 Categorical).
    2. Tách Amt_cents (trên float64) và cờ has_identity.
    3. Hạ kiểu bộ nhớ an toàn và đo mức tiết kiệm RAM + Peak RSS.
    4. Tạo Timestamp-Safe Temporal Split, bảng độ nhạy Gap và bảng 5 Inner Folds.
    5. Lưu split_indices.parquet và schema_step0.json tại OUTPUT_DIR / 'artifacts_step0'.
    """
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    num_cols, cat_cols = get_ieee_cis_canonical_schema()

    # Bước 0.1: Trích xuất Amt_cents (float64) và has_identity trước khi hạ kiểu
    train_df = extract_pre_downcast_features(train_df)

    # Kiểm tra đủ 432 biến gốc trong bảng đã ghép
    missing_schema_cols = [c for c in (num_cols + cat_cols) if c not in train_df.columns]
    assert len(missing_schema_cols) == 0, f"Thiếu các cột trong Schema 432 biến: {missing_schema_cols[:10]}"

    # Bước 0.2: Hạ kiểu RAM an toàn
    train_df, mem_stats = downcast_dataframe_safely(train_df, num_cols=num_cols, cat_cols=cat_cols)

    # Bước 0.3: Chia tập Timestamp-Safe Temporal Split & Stratified Split
    split_df, gap_summary_df, inner_folds_df = create_timestamp_safe_splits(train_df)

    # Bước 0.4: Xuất file bàn giao vào đúng đường dẫn đích
    split_file = out_path / "split_indices.parquet"
    schema_file = out_path / "schema_step0.json"
    try:
        split_df.to_parquet(split_file, index=False)
    except Exception:
        split_file = out_path / "split_indices.csv"
        split_df.to_csv(split_file, index=False)

    schema_payload = {
        "n_rows_train_merged": int(len(train_df)),
        "n_candidate_features_original": 432,
        "n_numerical_original": len(num_cols),
        "n_categorical_original": len(cat_cols),
        "step0_new_features": ["has_identity", "Amt_cents"],
        "identity_coverage_rate_pct": round(float(train_df["has_identity"].mean() * 100.0), 2),
        "overall_fraud_rate_pct": round(float(train_df["isFraud"].mean() * 100.0), 2),
        "memory_stats": mem_stats,
        "numerical_columns": num_cols,
        "categorical_columns": cat_cols,
    }
    with open(schema_file, "w", encoding="utf-8") as f:
        json.dump(schema_payload, f, indent=2, ensure_ascii=False)

    print("=" * 76)
    print("HOÀN TẤT BƯỚC 0: CHUẨN BỊ DỮ LIỆU, HẠ KIỂU RAM & CHIA TẬP AN TOÀN")
    print("=" * 76)
    print(f"- Kích thước bảng Train: {train_df.shape[0]:,} dòng x {train_df.shape[1]} cột (đã thêm has_identity, Amt_cents)")
    print(f"- Định nghĩa Schema gốc: {len(num_cols)} biến Numerical + {len(cat_cols)} biến Categorical = 432 biến")
    print(
        f"- Dung lượng DataFrame : {mem_stats['df_memory_before_mb']:.2f} MB -> "
        f"{mem_stats['df_memory_after_mb']:.2f} MB (Tiết kiệm {mem_stats['memory_saved_pct']:.2f}%)"
    )
    print(
        f"- Bộ nhớ Tiến trình    : RSS hiện tại = {mem_stats['process_rss_after_mb']:.2f} MB | "
        f"Peak RSS = {mem_stats['peak_rss_mb']:.2f} MB"
    )
    print(
        f"- Sai số tương đối tối đa của TransactionAmt khi sang float32: "
        f"{mem_stats['max_amt_rel_err_float32']:.2e}"
    )
    print(f"- Tỷ lệ khớp Identity  : {schema_payload['identity_coverage_rate_pct']:.2f}%")
    print(f"- Tỷ lệ Fraud toàn tập : {schema_payload['overall_fraud_rate_pct']:.2f}%")
    print("\n[Bảng 1] Phân tích độ nhạy theo kịch bản độ trễ nhãn (Gap / Embargo):")
    print(gap_summary_df.to_string(index=False))
    print("\n[Bảng 2] Thống kê 5 Inner Temporal Folds (phục vụ chọn biến ở Giai đoạn 3):")
    print(inner_folds_df.to_string(index=False))
    print(f"\nĐã lưu file chỉ số chia tập tại: {split_file}")
    print(f"Đã lưu file đặc tả Schema tại  : {schema_file}")

    return {
        "df": train_df,
        "split_df": split_df,
        "gap_summary_df": gap_summary_df,
        "inner_folds_df": inner_folds_df,
        "schema_payload": schema_payload,
    }

print("Đã khởi tạo xong toàn bộ hàm của Bước 0 (step0_setup_and_split).")

### CELL 5: Thực thi Bước 0 & Kiểm tra Bảng Chia tập (split_df)


In [ ]:
# ==============================================================================
# CELL 5: THỰC THI BƯỚC 0 & KIỂM TRA BẢNG CHỈ SỐ CHIA TẬP (SPLIT_INDICES)
# ==============================================================================
# Giải thích:
# - Chạy hàm run_step0_pipeline() trên bảng train_df vừa ghép ở Cell 3.
# - Kết quả được lưu vào `/content/drive/MyDrive/IEEE_CIS_Project/artifacts/artifacts_step0`.
# - Giải phóng biến `train_df` gốc (vì bảng đã hạ kiểu RAM nằm trong `step0_res["df"]`)
#   và hiển thị 5 dòng đầu của `split_df` giống hệt cell kiểm tra trong Notebook đích.

print("\n========== RUNNING STEP 0 ==========")
step0_res = run_step0_pipeline(
    train_df,
    output_dir=str(OUTPUT_DIR / "artifacts_step0"),
)
print("STEP 0 COMPLETED!")

# Giải phóng train_df gốc để tiết kiệm RAM
del train_df
gc.collect()

# Kiểm tra trực tiếp bảng split_df vừa tạo (giống cell kiểm tra trong file đích)
split_df_preview = step0_res["split_df"]
print("\n===== SPLIT DATA PREVIEW =====")
print("Shape:", split_df_preview.shape)
print("Columns:", split_df_preview.columns.tolist())
print("\n===== TEMPORAL ROLE =====")
print(split_df_preview["temporal_role"].value_counts(dropna=False))
display(split_df_preview.head())

### CELL 6: Định nghĩa các Hàm Bước 1 - Advanced EDA (step1_advanced_eda.py)


In [ ]:
# ==============================================================================
# CELL 6: ĐỊNH NGHĨA CÁC HÀM BƯỚC 1 - PHÂN TÍCH KHÁM PHÁ CHUYÊN SÂU (ADVANCED EDA)
#         (Chuyển đổi từ file: step1_advanced_eda.py)
# ==============================================================================
# Giải thích chuyên môn:
# 1. Chống Data Leakage tuyệt đối:
#    Mọi thống kê và biểu đồ có sử dụng nhãn isFraud chỉ được tính trên tập Temporal Train
#    (80% thời gian đầu, tương ứng 472,432 giao dịch).
# 2. Thống kê suy diễn chặt chẽ:
#    - Khoảng tin cậy 95% Wilson (wilson_ci_95): chính xác hơn xấp xỉ chuẩn khi tỷ lệ Fraud nhỏ (~3.5%).
#    - Kiểm định 2 tỷ lệ (two_proportion_z_pvalue) kết hợp hiệu chỉnh đa kiểm định Benjamini-Hochberg
#      (benjamini_hochberg_fdr) để kiểm soát False Discovery Rate (FDR < 0.05) khi quét qua 432 biến.
#    - Hệ số Cramér's V có hiệu chỉnh thiên lệch (bias_corrected_cramers_v) cho 49 biến phân loại.
# 3. Phân tích Khuyết có điều kiện (analyze_conditional_missingness):
#    - Đánh giá 40 biến Identity RIÊNG bên trong nhóm `has_identity == 1`.
#    - Gom cụm động 339 biến V1-V339 theo mặt nạ khuyết thực tế bằng mã băm 64-bit (ra 14 cụm V).
# 4. Phân tích Biến số, Ngoại lai P95/P99 & Trôi dạt thời gian họ D (analyze_numerical_and_outliers).
# 5. Xuất 10 bảng CSV và 3 biểu đồ chất lượng cao (Figure 4, Figure 5, Figure 6).


# ------------------------------------------------------------------------------
# 6.1. CÁC HÀM THỐNG KÊ CỐT LÕI (WILSON 95% CI, LIFT NHÓM BÙ, CRAMÉR'S V & FDR)
# ------------------------------------------------------------------------------
def wilson_ci_95(k: np.ndarray, n: np.ndarray, z: float = 1.959963984540054) -> Tuple[np.ndarray, np.ndarray]:
    """
    Tính khoảng tin cậy 95% theo công thức Wilson cho tỷ lệ nhị thức p = k / n.
    Chính xác hơn xấp xỉ chuẩn truyền thống khi tỷ lệ Fraud nhỏ (3.5%) hoặc cỡ mẫu nhóm nhỏ.
    """
    k_arr = np.asarray(k, dtype="float64")
    n_arr = np.asarray(n, dtype="float64")
    safe_n = np.maximum(n_arr, 1.0)
    p = k_arr / safe_n
    z2 = z * z

    denom = 1.0 + z2 / safe_n
    center = (p + z2 / (2.0 * safe_n)) / denom
    half_width = (z * np.sqrt((p * (1.0 - p) + z2 / (4.0 * safe_n)) / safe_n)) / denom

    low = np.where(n_arr > 0, np.clip(center - half_width, 0.0, 1.0), np.nan)
    high = np.where(n_arr > 0, np.clip(center + half_width, 0.0, 1.0), np.nan)
    return low, high


def benjamini_hochberg_fdr(p_values: np.ndarray) -> np.ndarray:
    """Hiệu chỉnh đa kiểm định theo thủ tục Benjamini-Hochberg (kiểm soát False Discovery Rate)."""
    p = np.asarray(p_values, dtype="float64")
    q = np.full_like(p, np.nan, dtype="float64")
    valid_mask = np.isfinite(p)
    p_valid = p[valid_mask]
    m = len(p_valid)
    if m == 0:
        return q

    order = np.argsort(p_valid)
    ranked_p = p_valid[order]
    ranks = np.arange(1, m + 1, dtype="float64")
    q_ranked = ranked_p * (m / ranks)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0.0, 1.0)

    q_valid = np.empty_like(p_valid)
    q_valid[order] = q_ranked
    q[valid_mask] = q_valid
    return q


def two_proportion_z_pvalue(k1: int, n1: int, k2: int, n2: int) -> float:
    """Tính p-value kiểm định 2 tỷ lệ độc lập (nhóm đang xét vs nhóm bù)."""
    if n1 <= 0 or n2 <= 0:
        return np.nan
    p1 = k1 / n1
    p2 = k2 / n2
    p_pool = (k1 + k2) / (n1 + n2)
    if p_pool <= 0.0 or p_pool >= 1.0:
        return 1.0
    se = np.sqrt(p_pool * (1.0 - p_pool) * (1.0 / n1 + 1.0 / n2))
    if se <= 0.0:
        return 1.0
    z = abs(p1 - p2) / se
    return float(erfc(z / sqrt(2.0)))


def bias_corrected_cramers_v(cat_series: pd.Series, y: pd.Series) -> float:
    """
    Tính hệ số Cramér's V có hiệu chỉnh thiên lệch (Bergsma 2013) giữa biến phân loại và isFraud.
    Bao gồm cả trạng thái 'Missing' như một nhóm riêng biệt để không bỏ sót tín hiệu khuyết.
    """
    s = cat_series.astype("object").fillna("__MISSING__").astype(str)
    ct = pd.crosstab(s, y)
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return 0.0

    obs = ct.to_numpy(dtype="float64")
    n = obs.sum()
    if n <= 1:
        return 0.0

    row_sums = obs.sum(axis=1, keepdims=True)
    col_sums = obs.sum(axis=0, keepdims=True)
    expected = (row_sums @ col_sums) / n
    valid = expected > 0
    chi2 = float(np.sum(((obs[valid] - expected[valid]) ** 2) / expected[valid]))

    phi2 = chi2 / n
    r, k = obs.shape
    phi2_corr = max(0.0, phi2 - ((k - 1.0) * (r - 1.0)) / (n - 1.0))
    r_corr = r - ((r - 1.0) ** 2) / (n - 1.0)
    k_corr = k - ((k - 1.0) ** 2) / (n - 1.0)
    denom = min(k_corr - 1.0, r_corr - 1.0)
    if denom <= 0.0:
        return 0.0
    return float(np.sqrt(phi2_corr / denom))


# ------------------------------------------------------------------------------
# 6.2. PHÂN TÍCH ĐỘ PHỦ IDENTITY, KHUYẾT CÓ ĐIỀU KIỆN & GOM CỤM HỌ V
# ------------------------------------------------------------------------------
def analyze_conditional_missingness(
    train_df: pd.DataFrame,
    num_cols: List[str],
    cat_cols: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Thực hiện 3 phân tích cấu trúc khuyết trên tập Temporal Train:
    1. Bảng so sánh tỷ lệ gian lận theo sự hiện diện của bảng Identity (has_identity = 1 vs 0).
    2. Bảng đánh giá tỷ lệ khuyết và Risk Ratio (Lift) của từng biến:
       - Các biến Transaction (392 biến): đánh giá trên toàn bộ Temporal Train.
       - Các biến Identity (40 biến): đánh giá CÓ ĐIỀU KIỆN bên trong nhóm has_identity == 1.
       - Áp dụng hiệu chỉnh đa kiểm định Benjamini-Hochberg (FDR).
    3. Gom cụm động các biến V1-V339 có cùng mặt nạ khuyết thực tế (missingness pattern).
    """
    y_all = train_df["isFraud"].to_numpy(dtype="int64")
    n_total = len(train_df)
    total_fraud = int(y_all.sum())

    # 1. Phân tích Identity Coverage (has_identity == 1 vs 0)
    id_cov_rows = []
    for val, label in [(1, "Có bản ghi Identity (has_identity = 1)"), (0, "Không có Identity (has_identity = 0)")]:
        mask = (train_df["has_identity"].to_numpy() == val)
        n_grp = int(mask.sum())
        k_grp = int(y_all[mask].sum())
        n_comp = n_total - n_grp
        k_comp = total_fraud - k_grp

        rate_grp = k_grp / max(n_grp, 1)
        rate_comp = k_comp / max(n_comp, 1)
        ci_low, ci_high = wilson_ci_95(np.array([k_grp]), np.array([n_grp]))
        lift = rate_grp / max(rate_comp, 1e-9)

        id_cov_rows.append(
            {
                "Nhom_Identity": label,
                "So_luong_Giao_dich": n_grp,
                "Ty_trong_pct": round(n_grp / max(n_total, 1) * 100.0, 2),
                "So_ca_Fraud": k_grp,
                "Fraud_Rate_pct": round(rate_grp * 100.0, 3),
                "CI_95_Low_pct": round(float(ci_low[0]) * 100.0, 3),
                "CI_95_High_pct": round(float(ci_high[0]) * 100.0, 3),
                "Chenh_lech_vs_Nhom_bu_pp": round((rate_grp - rate_comp) * 100.0, 3),
                "Risk_Ratio_Lift": round(lift, 3),
            }
        )
    identity_coverage_df = pd.DataFrame(id_cov_rows)

    # 2. Đánh giá Missingness & Lift có điều kiện cho 432 biến gốc
    identity_set = set([f"id_{i:02d}" for i in range(1, 39)] + ["DeviceType", "DeviceInfo"])
    id_sub_mask = (train_df["has_identity"].to_numpy() == 1)

    miss_rows = []
    for col in (num_cols + cat_cols):
        if col not in train_df.columns:
            continue
        is_id_col = col in identity_set
        if is_id_col:
            sub_y = y_all[id_sub_mask]
            is_miss = train_df.loc[id_sub_mask, col].isna().to_numpy()
            scope_name = "Conditional (has_identity=1)"
        else:
            sub_y = y_all
            is_miss = train_df[col].isna().to_numpy()
            scope_name = "Full Temporal Train"

        n_scope = len(sub_y)
        n_miss = int(is_miss.sum())
        n_pres = n_scope - n_miss
        k_miss = int(sub_y[is_miss].sum())
        k_pres = int(sub_y[~is_miss].sum())

        r_miss = k_miss / max(n_miss, 1) if n_miss > 0 else np.nan
        r_pres = k_pres / max(n_pres, 1) if n_pres > 0 else np.nan
        lift_miss_vs_pres = (r_miss / max(r_pres, 1e-9)) if (n_miss > 0 and n_pres > 0 and r_pres > 0) else np.nan
        diff_pp = (r_miss - r_pres) * 100.0 if (n_miss > 0 and n_pres > 0) else np.nan
        pval = two_proportion_z_pvalue(k_miss, n_miss, k_pres, n_pres) if (n_miss >= 10 and n_pres >= 10) else np.nan

        miss_rows.append(
            {
                "Feature": col,
                "Scope": scope_name,
                "Missing_Count": n_miss,
                "Present_Count": n_pres,
                "Missing_Rate_pct": round(n_miss / max(n_scope, 1) * 100.0, 2),
                "Fraud_Rate_Missing_pct": round(r_miss * 100.0, 3) if np.isfinite(r_miss) else np.nan,
                "Fraud_Rate_Present_pct": round(r_pres * 100.0, 3) if np.isfinite(r_pres) else np.nan,
                "Diff_Missing_vs_Present_pp": round(diff_pp, 3) if np.isfinite(diff_pp) else np.nan,
                "Lift_Missing_vs_Present": round(lift_miss_vs_pres, 3) if np.isfinite(lift_miss_vs_pres) else np.nan,
                "p_value_raw": pval,
            }
        )

    missing_audit_df = pd.DataFrame(miss_rows)
    missing_audit_df["q_value_bh_fdr"] = benjamini_hochberg_fdr(missing_audit_df["p_value_raw"].to_numpy())
    missing_audit_df["Has_Strong_Missing_Signal"] = (
        (missing_audit_df["q_value_bh_fdr"] < 0.05)
        & (
            (missing_audit_df["Lift_Missing_vs_Present"] >= 1.25)
            | (missing_audit_df["Lift_Missing_vs_Present"] <= 0.80)
            | (missing_audit_df["Diff_Missing_vs_Present_pp"].abs() >= 1.0)
        )
    )
    missing_audit_df = missing_audit_df.sort_values("Missing_Rate_pct", ascending=False).reset_index(drop=True)

    # 3. Gom cụm động các biến V1-V339 có cùng mặt nạ khuyết thực tế (Missingness Pattern Clustering)
    v_cols = [f"V{i}" for i in range(1, 340) if f"V{i}" in train_df.columns]
    pattern_groups: Dict[Tuple[int, int], List[str]] = {}
    for col in v_cols:
        mask_arr = train_df[col].isna().to_numpy(dtype=np.uint8)
        n_na = int(mask_arr.sum())
        mask_hash = int(pd.util.hash_array(mask_arr).sum())
        key = (n_na, mask_hash)
        pattern_groups.setdefault(key, []).append(col)

    v_cluster_rows = []
    for cluster_idx, ((n_na, _), cols_in_cluster) in enumerate(
        sorted(pattern_groups.items(), key=lambda x: x[0][0]), start=1
    ):
        v_cluster_rows.append(
            {
                "Cluster_ID": f"V_Cluster_{cluster_idx:02d}",
                "Num_V_Columns": len(cols_in_cluster),
                "Missing_Count": n_na,
                "Missing_Rate_pct": round(n_na / max(n_total, 1) * 100.0, 2),
                "Sample_Columns": ", ".join(cols_in_cluster[:6]) + ("..." if len(cols_in_cluster) > 6 else ""),
                "All_Columns": cols_in_cluster,
            }
        )
    v_clusters_df = pd.DataFrame(v_cluster_rows)

    return identity_coverage_df, missing_audit_df, v_clusters_df


# ------------------------------------------------------------------------------
# 6.3. PHÂN TÍCH BIẾN SỐ THEO NHÃN ISFRAUD, QUAN HỆ D-TIME & NGOẠI LAI (95TH/99TH)
# ------------------------------------------------------------------------------
def analyze_numerical_and_outliers(
    train_df: pd.DataFrame,
    focus_num_cols: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Thực hiện 3 bảng phân tích cho biến số trên tập Temporal Train:
    1. Thống kê phân phối (Mean, Std, Median, IQR, 95th, 99th) tách theo Non-Fraud (0) và Fraud (1).
    2. Phân tích tỷ lệ gian lận ở vùng ngoại lai (trên phân vị thứ 95 và 99) so với nhóm bù.
    3. Kiểm tra mối tương quan giữa D1-D15 và thời gian trôi qua (elapsed_day) để làm cơ sở cho D_anchor.
    """
    if focus_num_cols is None:
        focus_num_cols = ["TransactionAmt", "Amt_cents", "dist1", "C1", "C2", "C13", "D1", "D2", "D15"]
    focus_num_cols = [c for c in focus_num_cols if c in train_df.columns]

    y = train_df["isFraud"].to_numpy(dtype="int64")
    total_n = len(train_df)

    dist_rows = []
    outlier_rows = []

    for col in focus_num_cols:
        s_all = train_df[col].dropna()
        if len(s_all) == 0:
            continue
        for cls_val, cls_name in [(0, "Non-Fraud (0)"), (1, "Fraud (1)")]:
            s_cls = train_df.loc[train_df["isFraud"] == cls_val, col].dropna()
            if len(s_cls) == 0:
                continue
            q25, q50, q75, q95, q99 = np.percentile(s_cls.to_numpy(dtype="float64"), [25, 50, 75, 95, 99])
            dist_rows.append(
                {
                    "Feature": col,
                    "Class": cls_name,
                    "Observed_Count": len(s_cls),
                    "Mean": round(float(s_cls.mean()), 3),
                    "Std": round(float(s_cls.std()), 3),
                    "Median": round(float(q50), 3),
                    "IQR": round(float(q75 - q25), 3),
                    "P95": round(float(q95), 3),
                    "P99": round(float(q99), 3),
                    "Max": round(float(s_cls.max()), 3),
                }
            )

        # Phân tích Outliers ở ngưỡng P95 và P99 trên các dòng có giá trị quan sát
        obs_mask = train_df[col].notna().to_numpy()
        vals_obs = train_df.loc[obs_mask, col].to_numpy(dtype="float64")
        y_obs = y[obs_mask]
        n_obs = len(y_obs)
        k_obs = int(y_obs.sum())

        p95_thr, p99_thr = np.percentile(vals_obs, [95, 99])
        for pct_label, thr in [("P95 (Top 5%)", p95_thr), ("P99 (Top 1%)", p99_thr)]:
            out_mask = vals_obs >= thr
            n_out = int(out_mask.sum())
            k_out = int(y_obs[out_mask].sum())
            n_comp = n_obs - n_out
            k_comp = k_obs - k_out

            r_out = k_out / max(n_out, 1)
            r_comp = k_comp / max(n_comp, 1)
            ci_low, ci_high = wilson_ci_95(np.array([k_out]), np.array([n_out]))
            lift = r_out / max(r_comp, 1e-9)

            outlier_rows.append(
                {
                    "Feature": col,
                    "Percentile_Gate": pct_label,
                    "Threshold_Value": round(float(thr), 3),
                    "Outlier_Count": n_out,
                    "Outlier_Fraud_Count": k_out,
                    "Outlier_Fraud_Rate_pct": round(r_out * 100.0, 3),
                    "CI_95_Low_pct": round(float(ci_low[0]) * 100.0, 3),
                    "CI_95_High_pct": round(float(ci_high[0]) * 100.0, 3),
                    "Complement_Fraud_Rate_pct": round(r_comp * 100.0, 3),
                    "Lift_vs_Complement": round(lift, 3),
                }
            )

    num_dist_df = pd.DataFrame(dist_rows)
    outlier_df = pd.DataFrame(outlier_rows)

    # 3. Phân tích quan hệ giữa D1-D15 và thời gian trôi qua (elapsed_day)
    elapsed_day = (train_df["TransactionDT"].to_numpy(dtype="float64") // 86400.0)
    d_cols = [f"D{i}" for i in range(1, 16) if f"D{i}" in train_df.columns]
    d_rows = []
    for d_col in d_cols:
        d_vals = train_df[d_col].to_numpy(dtype="float64")
        valid = np.isfinite(d_vals)
        n_val = int(valid.sum())
        if n_val < 10:
            continue
        anchor_vals = elapsed_day[valid] - d_vals[valid]
        corr_dt = float(pd.Series(d_vals[valid]).corr(pd.Series(elapsed_day[valid]), method="spearman"))
        neg_rate = float((d_vals[valid] < 0).mean() * 100.0)

        d_rows.append(
            {
                "D_Feature": d_col,
                "Observed_Count": n_val,
                "Missing_Rate_pct": round((1.0 - n_val / max(total_n, 1)) * 100.0, 2),
                "Negative_Values_pct": round(neg_rate, 3),
                "Spearman_with_ElapsedDay": round(corr_dt, 4),
                "Raw_D_Std": round(float(np.std(d_vals[valid])), 2),
                "Anchor_D_Std": round(float(np.std(anchor_vals)), 2),
            }
        )
    d_drift_df = pd.DataFrame(d_rows)

    return num_dist_df, outlier_df, d_drift_df


# ------------------------------------------------------------------------------
# 6.4. PHÂN TÍCH BIẾN PHÂN LOẠI (COUNT + FRAUD RATE + CI 95% + CRAMÉR'S V)
# ------------------------------------------------------------------------------
def analyze_categorical_features(
    train_df: pd.DataFrame,
    cat_cols: List[str],
    key_cat_cols: Optional[List[str]] = None,
    top_k_levels: int = 8,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    1. Tính hệ số Cramér's V có hiệu chỉnh thiên lệch cho toàn bộ 49 biến phân loại.
    2. Lập bảng chi tiết (Count, Fraud Count, Fraud Rate %, Wilson CI 95%, Lift vs nhóm bù)
       cho các biến phân loại trọng tâm (ProductCD, card4, card6, P_emaildomain, DeviceType, M1-M9).
    """
    y_series = train_df["isFraud"]
    y_arr = y_series.to_numpy(dtype="int64")
    total_n = len(train_df)
    total_k = int(y_arr.sum())

    # 1. Xếp hạng Cramér's V cho toàn bộ 49 biến phân loại
    cramers_rows = []
    for col in cat_cols:
        if col not in train_df.columns:
            continue
        v_score = bias_corrected_cramers_v(train_df[col], y_series)
        n_uniq = int(train_df[col].nunique(dropna=True))
        miss_pct = float(train_df[col].isna().mean() * 100.0)
        cramers_rows.append(
            {
                "Categorical_Feature": col,
                "Unique_Levels": n_uniq,
                "Missing_Rate_pct": round(miss_pct, 2),
                "Cramers_V_BiasCorrected": round(v_score, 4),
            }
        )
    cramers_df = pd.DataFrame(cramers_rows).sort_values("Cramers_V_BiasCorrected", ascending=False).reset_index(drop=True)

    # 2. Bảng chi tiết theo từng giá trị (Level) của các biến phân loại chủ chốt
    if key_cat_cols is None:
        key_cat_cols = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain", "DeviceType", "M1", "M4"]
    key_cat_cols = [c for c in key_cat_cols if c in train_df.columns]

    level_rows = []
    for col in key_cat_cols:
        s = train_df[col].astype("object").fillna("Missing").astype(str)
        top_levels = s.value_counts().head(top_k_levels).index.tolist()
        s_grouped = np.where(s.isin(top_levels), s, "Other_Rare")

        for lvl in pd.Series(s_grouped).value_counts().index:
            mask = (s_grouped == lvl)
            n_lvl = int(mask.sum())
            k_lvl = int(y_arr[mask].sum())
            n_comp = total_n - n_lvl
            k_comp = total_k - k_lvl

            r_lvl = k_lvl / max(n_lvl, 1)
            r_comp = k_comp / max(n_comp, 1) if n_comp > 0 else total_k / max(total_n, 1)
            ci_low, ci_high = wilson_ci_95(np.array([k_lvl]), np.array([n_lvl]))
            lift = r_lvl / max(r_comp, 1e-9)

            level_rows.append(
                {
                    "Feature": col,
                    "Category_Level": str(lvl),
                    "Transaction_Count": n_lvl,
                    "Share_pct": round(n_lvl / max(total_n, 1) * 100.0, 2),
                    "Fraud_Count": k_lvl,
                    "Fraud_Rate_pct": round(r_lvl * 100.0, 3),
                    "CI_95_Low_pct": round(float(ci_low[0]) * 100.0, 3),
                    "CI_95_High_pct": round(float(ci_high[0]) * 100.0, 3),
                    "Lift_vs_Complement": round(lift, 3),
                }
            )
    cat_levels_df = pd.DataFrame(level_rows)
    return cramers_df, cat_levels_df


# ------------------------------------------------------------------------------
# 6.5. TƯƠNG QUAN HẠNG SPEARMAN TRƯỚC KHI ĐIỀN KHUYẾT (PAIRWISE-COMPLETE)
# ------------------------------------------------------------------------------
def analyze_spearman_correlations(
    train_df: pd.DataFrame,
    num_cols: List[str],
    min_observed_pairs: int = 100,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    1. Tính tương quan hạng Spearman giữa từng biến Numerical (chưa điền khuyết) và isFraud.
    2. Phát hiện các cặp biến có đa cộng tuyến cao (|rho| >= 0.90) bên trong họ C và D.
    """
    y_s = train_df["isFraud"].astype("float64")
    target_corr_rows = []

    for col in num_cols:
        if col not in train_df.columns:
            continue
        s = train_df[col]
        valid = s.notna()
        n_obs = int(valid.sum())
        if n_obs < min_observed_pairs or s[valid].nunique() < 2:
            continue
        rho = float(s[valid].astype("float64").corr(y_s[valid], method="spearman"))
        target_corr_rows.append(
            {
                "Feature": col,
                "Observed_Count": n_obs,
                "Spearman_with_isFraud": round(rho, 4),
                "Abs_Spearman": round(abs(rho), 4),
            }
        )

    target_corr_df = (
        pd.DataFrame(target_corr_rows, columns=["Feature", "Observed_Count", "Spearman_with_isFraud", "Abs_Spearman"])
        .sort_values("Abs_Spearman", ascending=False)
        .reset_index(drop=True)
    )

    # Kiểm tra các cặp tương quan cao (|rho| >= 0.90) trong nhóm C1-C14 và D1-D15
    cd_cols = [f"C{i}" for i in range(1, 15) if f"C{i}" in train_df.columns] + [
        f"D{i}" for i in range(1, 16) if f"D{i}" in train_df.columns
    ]
    corr_mat_cd = train_df[cd_cols].astype("float64").corr(method="spearman")
    high_pairs = []
    for i in range(len(cd_cols)):
        for j in range(i + 1, len(cd_cols)):
            c1, c2 = cd_cols[i], cd_cols[j]
            val = corr_mat_cd.loc[c1, c2]
            if np.isfinite(val) and abs(val) >= 0.90:
                high_pairs.append(
                    {
                        "Feature_1": c1,
                        "Feature_2": c2,
                        "Spearman_Corr": round(float(val), 4),
                    }
                )
    high_collinear_df = (
        pd.DataFrame(high_pairs, columns=["Feature_1", "Feature_2", "Spearman_Corr"])
        .sort_values("Spearman_Corr", ascending=False)
        .reset_index(drop=True)
    )

    return target_corr_df, high_collinear_df


# ------------------------------------------------------------------------------
# 6.6. TRỰC QUAN HÓA CHUYÊN SÂU (XUẤT 3 HÌNH CHUẨN BÁO CÁO)
# ------------------------------------------------------------------------------
def generate_step1_figures(
    train_df: pd.DataFrame,
    identity_coverage_df: pd.DataFrame,
    cat_levels_df: pd.DataFrame,
    target_corr_df: pd.DataFrame,
    outlier_df: pd.DataFrame,
    output_dir: Path,
) -> Dict[str, str]:
    """Vẽ và lưu 3 biểu đồ tổng hợp cho Báo cáo Phần 2 (Advanced EDA)."""
    plt.rcParams["font.sans-serif"] = "DejaVu Sans"
    plt.rcParams["axes.edgecolor"] = "#CCCCCC"
    fig_paths: Dict[str, str] = {}

    # FIGURE 4: Phân phối Numerical theo lớp isFraud (Amt_log, Elapsed Hour, C1, Amt_cents)
    fig4, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=150)
    non_fraud = train_df[train_df["isFraud"] == 0]
    fraud = train_df[train_df["isFraud"] == 1]

    # Panel A: log1p(TransactionAmt)
    ax = axes[0, 0]
    ax.hist(
        np.log1p(non_fraud["TransactionAmt"].dropna()),
        bins=45,
        density=True,
        alpha=0.55,
        color="#1f77b4",
        label=f"Non-Fraud (n={len(non_fraud):,})",
    )
    ax.hist(
        np.log1p(fraud["TransactionAmt"].dropna()),
        bins=45,
        density=True,
        alpha=0.60,
        color="#d62728",
        label=f"Fraud (n={len(fraud):,})",
    )
    ax.set_title("A. Phân phối log(1 + TransactionAmt) theo lớp isFraud", fontsize=11, fontweight="bold")
    ax.set_xlabel("log(1 + TransactionAmt)")
    ax.set_ylabel("Mật độ (Density)")
    ax.legend(frameon=True)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    # Panel B: Tỷ lệ gian lận và Số lượng giao dịch theo Giờ tương đối (elapsed_hour 0..23)
    ax = axes[0, 1]
    elapsed_hour = (train_df["TransactionDT"].to_numpy(dtype="int64") // 3600) % 24
    hr_df = (
        pd.DataFrame({"hour": elapsed_hour, "isFraud": train_df["isFraud"].to_numpy()})
        .groupby("hour")["isFraud"]
        .agg(cnt="count", frd="sum")
        .reindex(range(24), fill_value=0)
    )
    hr_rate = (hr_df["frd"] / np.maximum(hr_df["cnt"], 1)) * 100.0
    ci_l, ci_h = wilson_ci_95(hr_df["frd"].to_numpy(), hr_df["cnt"].to_numpy())

    ax.bar(hr_df.index, hr_df["cnt"], color="#aec7e8", alpha=0.7, label="Số giao dịch (Count)")
    ax.set_xlabel("Chu kỳ giờ tương đối (elapsed_hour: 0..23)")
    ax.set_ylabel("Số lượng giao dịch (Count)", color="#1f77b4")
    ax2 = ax.twinx()
    ax2.plot(hr_df.index, hr_rate, color="#d62728", marker="o", linewidth=2, label="Fraud Rate (%)")
    ax2.fill_between(hr_df.index, ci_l * 100.0, ci_h * 100.0, color="#d62728", alpha=0.18, label="95% Wilson CI")
    ax2.set_ylabel("Tỷ lệ gian lận - Fraud Rate (%)", color="#d62728")
    ax.set_title("B. Số lượng giao dịch & Fraud Rate theo Chu kỳ 24 giờ", fontsize=11, fontweight="bold")

    # Panel C: Boxplot log(1 + C1) theo lớp
    ax = axes[1, 0]
    c1_0 = np.log1p(non_fraud["C1"].dropna().clip(lower=0))
    c1_1 = np.log1p(fraud["C1"].dropna().clip(lower=0))
    bp = ax.boxplot(
        [c1_0, c1_1],
        tick_labels=["Non-Fraud (0)", "Fraud (1)"],
        patch_artist=True,
        showfliers=True,
    )
    for patch, color in zip(bp["boxes"], ["#9ecae1", "#fcbba1"]):
        patch.set_facecolor(color)
    ax.set_title("C. So sánh phân phối log(1 + C1) giữa 2 nhóm", fontsize=11, fontweight="bold")
    ax.set_ylabel("log(1 + C1)")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    # Panel D: Phân phối phần xu lẻ (Amt_cents = 0 vs > 0)
    ax = axes[1, 1]
    is_nonzero_cents = np.where(train_df["Amt_cents"] == 0, "Số tiền chẵn (.00)", "Có xu lẻ (.01-.99)")
    cents_grp = (
        pd.DataFrame({"grp": is_nonzero_cents, "isFraud": train_df["isFraud"].to_numpy()})
        .groupby("grp")["isFraud"]
        .agg(cnt="count", frd="sum")
    )
    cents_rate = (cents_grp["frd"] / np.maximum(cents_grp["cnt"], 1)) * 100.0
    bars = ax.bar(cents_grp.index, cents_rate, color=["#2ca02c", "#ff7f0e"], width=0.45)
    for bar, (_, row_val) in zip(bars, cents_grp.iterrows()):
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            h + 0.1,
            f"{h:.2f}%\n(n={int(row_val['cnt']):,})",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )
    ax.set_ylim(0, max(float(cents_rate.max()) * 1.35, 5.0))
    ax.set_title("D. Tỷ lệ gian lận theo Phần xu lẻ (Amt_cents)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Fraud Rate (%)")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    fig4.tight_layout()
    fig4_path = output_dir / "fig4_numerical_distributions_by_fraud.png"
    fig4.savefig(fig4_path)
    plt.close(fig4)
    fig_paths["fig4_numerical"] = str(fig4_path)

    # FIGURE 5: Tỷ lệ gian lận kèm Số lượng giao dịch cho Biến Phân loại & Identity
    fig5, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=150)

    ax = axes[0]
    labels_id = ["Có Identity\n(has_identity=1)", "Không có Identity\n(has_identity=0)"]
    rates_id = identity_coverage_df["Fraud_Rate_pct"].to_numpy()
    cnts_id = identity_coverage_df["So_luong_Giao_dich"].to_numpy()
    bars = ax.bar(labels_id, rates_id, color=["#d62728", "#1f77b4"], width=0.45)
    for bar, r_val, c_val in zip(bars, rates_id, cnts_id):
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            r_val + 0.15,
            f"{r_val:.2f}%\n(n={int(c_val):,})",
            ha="center",
            va="bottom",
            fontsize=9.5,
            fontweight="bold",
        )
    ax.set_ylim(0, max(float(rates_id.max()) * 1.35, 6.0))
    ax.set_title("A. Chênh lệch Fraud Rate theo Độ phủ bảng Identity", fontsize=11, fontweight="bold")
    ax.set_ylabel("Fraud Rate (%)")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    ax = axes[1]
    prod_df = cat_levels_df[cat_levels_df["Feature"] == "ProductCD"].copy()
    if len(prod_df) > 0:
        x_pos = np.arange(len(prod_df))
        ax.bar(x_pos, prod_df["Transaction_Count"], color="#c6dbef", width=0.5, label="Transaction Count")
        ax.set_xticks(x_pos)
        ax.set_xticklabels(prod_df["Category_Level"])
        ax.set_ylabel("Số lượng giao dịch (Count)", color="#1f77b4")
        ax2 = ax.twinx()
        ax2.plot(
            x_pos,
            prod_df["Fraud_Rate_pct"],
            color="#d62728",
            marker="s",
            linewidth=2.2,
            label="Fraud Rate (%)",
        )
        for x_i, r_i, f_i in zip(x_pos, prod_df["Fraud_Rate_pct"], prod_df["Fraud_Count"]):
            ax2.text(x_i, r_i + 0.2, f"{r_i:.1f}%\n({int(f_i)} ca)", ha="center", va="bottom", fontsize=8.5, color="#b30000")
        ax2.set_ylim(0, max(float(prod_df["Fraud_Rate_pct"].max()) * 1.35, 6.0))
        ax2.set_ylabel("Fraud Rate (%)", color="#d62728")
        ax.set_title("B. Số lượng giao dịch & Fraud Rate theo Mã sản phẩm (ProductCD)", fontsize=11, fontweight="bold")

    fig5.tight_layout()
    fig5_path = output_dir / "fig5_categorical_fraud_rate_with_counts.png"
    fig5.savefig(fig5_path)
    plt.close(fig5)
    fig_paths["fig5_categorical"] = str(fig5_path)

    # FIGURE 6: Top Tương quan Spearman với isFraud & Lift ở ngưỡng Ngoại lai P95/P99
    fig6, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=150)

    ax = axes[0]
    top_corr = target_corr_df.head(12).iloc[::-1]
    colors_corr = np.where(top_corr["Spearman_with_isFraud"] >= 0, "#d62728", "#1f77b4")
    ax.barh(top_corr["Feature"], top_corr["Spearman_with_isFraud"], color=colors_corr)
    ax.axvline(0.0, color="#333333", linewidth=1)
    ax.set_title("A. Top 12 biến Numerical có tương quan Spearman mạnh nhất với isFraud", fontsize=10.5, fontweight="bold")
    ax.set_xlabel("Hệ số tương quan hạng Spearman (trước khi điền khuyết)")
    ax.grid(axis="x", linestyle="--", alpha=0.4)

    ax = axes[1]
    p99_sub = outlier_df[outlier_df["Percentile_Gate"] == "P99 (Top 1%)"].head(8).iloc[::-1]
    if len(p99_sub) > 0:
        ax.barh(p99_sub["Feature"], p99_sub["Lift_vs_Complement"], color="#ff7f0e")
        ax.axvline(1.0, color="#333333", linestyle="--", linewidth=1.2, label="Mức nền (Lift = 1.0)")
        ax.set_title("B. Hệ số nhân rủi ro (Fraud Lift) ở nhóm Ngoại lai P99 (Top 1%)", fontsize=10.5, fontweight="bold")
        ax.set_xlabel("Risk Ratio / Lift so với 99% giao dịch còn lại")
        ax.legend(frameon=True)
        ax.grid(axis="x", linestyle="--", alpha=0.4)

    fig6.tight_layout()
    fig6_path = output_dir / "fig6_spearman_corr_and_outlier_lift.png"
    fig6.savefig(fig6_path)
    plt.close(fig6)
    fig_paths["fig6_corr_outlier"] = str(fig6_path)

    return fig_paths


# ------------------------------------------------------------------------------
# 6.7. HÀM CHẠY TRỌN GÓI BƯỚC 1 (PIPELINE ENTRYPOINT + KIỂM THỬ CHỐNG LEAKAGE)
# ------------------------------------------------------------------------------
def run_step1_advanced_eda(
    df: pd.DataFrame,
    split_df: pd.DataFrame,
    output_dir: str = str(STEP1_OUTPUT_DIR),
) -> Dict[str, object]:
    """
    Thực thi toàn bộ Bước 1 (Advanced EDA) chỉ trên tập Temporal Train:
    - Kiểm tra bằng assert không có dòng Validation nào được đưa vào tính toán.
    - Xuất đầy đủ 10 bảng CSV và 3 biểu đồ PNG tại OUTPUT_DIR / 'artifacts_step1_eda'.
    """
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    num_cols, cat_cols = get_ieee_cis_canonical_schema()

    # Chốt chặn chống Data Leakage: Chỉ lấy các dòng thuộc Temporal Train
    assert len(df) == len(split_df), "Kích thước df và split_df không khớp!"
    train_mask = (split_df["temporal_role"].to_numpy() == "train")
    val_mask = (split_df["temporal_role"].to_numpy() == "val")
    train_only_df = df.loc[train_mask].copy()

    # Kiểm chứng ranh giới thời gian
    max_train_dt = float(train_only_df["TransactionDT"].max())
    min_val_dt = float(df.loc[val_mask, "TransactionDT"].min())
    assert max_train_dt < min_val_dt, "LỖI LEAKAGE: Tồn tại dòng trong Temporal Train có TransactionDT >= Temporal Val!"

    # 1. Phân tích Độ phủ Identity, Khuyết có điều kiện + FDR và Gom cụm họ V
    identity_coverage_df, missing_audit_df, v_clusters_df = analyze_conditional_missingness(
        train_only_df, num_cols=num_cols, cat_cols=cat_cols
    )

    # 2. Phân tích Biến Số theo lớp, Ngoại lai P95/P99 và Quan hệ D-Time
    num_dist_df, outlier_df, d_drift_df = analyze_numerical_and_outliers(train_only_df)

    # 3. Phân tích Biến Phân loại (Cramér's V & Chi tiết từng nhóm kèm Count + CI 95%)
    cramers_df, cat_levels_df = analyze_categorical_features(train_only_df, cat_cols=cat_cols)

    # 4. Phân tích Tương quan Hạng Spearman (Pairwise-Complete trước khi điền khuyết)
    target_corr_df, high_collinear_df = analyze_spearman_correlations(train_only_df, num_cols=num_cols)

    # 5. Xuất biểu đồ và lưu bảng kết quả vào đúng đường dẫn đích
    fig_paths = generate_step1_figures(
        train_only_df,
        identity_coverage_df=identity_coverage_df,
        cat_levels_df=cat_levels_df,
        target_corr_df=target_corr_df,
        outlier_df=outlier_df,
        output_dir=out_path,
    )

    identity_coverage_df.to_csv(out_path / "eda_identity_coverage.csv", index=False)
    missing_audit_df.to_csv(out_path / "eda_conditional_missingness_fdr.csv", index=False)
    v_clusters_df.drop(columns=["All_Columns"]).to_csv(out_path / "eda_v_missing_clusters.csv", index=False)
    num_dist_df.to_csv(out_path / "eda_numerical_by_class.csv", index=False)
    outlier_df.to_csv(out_path / "eda_outliers_p95_p99.csv", index=False)
    d_drift_df.to_csv(out_path / "eda_d_features_time_drift.csv", index=False)
    cramers_df.to_csv(out_path / "eda_categorical_cramers_v.csv", index=False)
    cat_levels_df.to_csv(out_path / "eda_categorical_levels_detail.csv", index=False)
    target_corr_df.to_csv(out_path / "eda_spearman_with_target.csv", index=False)
    high_collinear_df.to_csv(out_path / "eda_high_collinear_pairs_cd.csv", index=False)

    print("=" * 78)
    print("HOÀN TẤT BƯỚC 1: ADVANCED EDA TRÊN TẬP TEMPORAL TRAIN (0% VALIDATION LEAKAGE)")
    print("=" * 78)
    print(f"- Số dòng Temporal Train đưa vào phân tích: {len(train_only_df):,} dòng (Tỷ lệ Fraud: {train_only_df['isFraud'].mean()*100:.2f}%)")
    print(f"- Số cụm mặt nạ khuyết thực tế của V1-V339: {len(v_clusters_df)} cụm")
    print(f"- Số biến có tín hiệu khuyết đáng kể (FDR < 0.05 & |Lift| mạnh): {int(missing_audit_df['Has_Strong_Missing_Signal'].sum())} biến")
    print("\n[Bảng 1.1] So sánh tỷ lệ gian lận theo Độ phủ bảng Identity (has_identity):")
    print(identity_coverage_df.to_string(index=False))
    print("\n[Bảng 1.2] Top 8 biến Phân loại có hệ số Cramér's V (hiệu chỉnh thiên lệch) cao nhất:")
    print(cramers_df.head(8).to_string(index=False))
    print("\n[Bảng 1.3] Phân tích Ngoại lai ở ngưỡng P95 & P99 cho các biến tiêu biểu:")
    print(outlier_df.head(8).to_string(index=False))
    print(f"\nĐã lưu 10 bảng thống kê CSV và 3 biểu đồ PNG tại thư mục: {out_path}")

    return {
        "identity_coverage_df": identity_coverage_df,
        "missing_audit_df": missing_audit_df,
        "v_clusters_df": v_clusters_df,
        "num_dist_df": num_dist_df,
        "outlier_df": outlier_df,
        "d_drift_df": d_drift_df,
        "cramers_df": cramers_df,
        "cat_levels_df": cat_levels_df,
        "target_corr_df": target_corr_df,
        "high_collinear_df": high_collinear_df,
        "fig_paths": fig_paths,
    }

print("Đã khởi tạo xong toàn bộ hàm của Bước 1 (step1_advanced_eda).")

### CELL 7: Thực thi Bước 1 (Advanced EDA)


In [ ]:
# ==============================================================================
# CELL 7: THỰC THI BƯỚC 1 (ADVANCED EDA) TRÊN TẬP TEMPORAL TRAIN
# ==============================================================================
# Giải thích:
# - Gọi hàm run_step1_advanced_eda() và lưu kết quả vào:
#   `/content/drive/MyDrive/IEEE_CIS_Project/artifacts/artifacts_step1_eda`
# - Sau khi xuất xong 10 bảng CSV và 3 biểu đồ PNG ra Google Drive, giải phóng `step1_res`
#   để giữ RAM trống cho Bước 2 (Feature Engineering).

print("\n========== RUNNING STEP 1 ==========")
step1_res = run_step1_advanced_eda(
    step0_res["df"],
    step0_res["split_df"],
    output_dir=str(OUTPUT_DIR / "artifacts_step1_eda"),
)
print("STEP 1 COMPLETED!")

del step1_res
gc.collect()

### CELL 8: Định nghĩa các Hàm & Class Bước 2 - Feature Engineering (step2_feature_engineering.py)


In [ ]:
# ==============================================================================
# CELL 8: ĐỊNH NGHĨA CÁC HÀM & CLASS BƯỚC 2 - POINT-IN-TIME FEATURE ENGINEERING
#         (Chuyển đổi từ file: step2_feature_engineering.py)
# ==============================================================================
# Giải thích chuyên môn:
# 1. Nhóm biến Stateless (generate_stateless_features - tính độc lập trên từng dòng):
#    - Tỷ lệ và số lượng khuyết theo 5 họ biến: C, D, M, V, và Identity có điều kiện
#      (gán -1 khi has_identity == 0 để không trùng lặp thông tin với cờ has_identity).
#    - Thời gian chu kỳ tương đối (elapsed_hour, day_cycle, hour_sin/cos, day_sin/cos).
#    - Neo mốc thời gian cho họ biến D: D1_anchor..D15_anchor = elapsed_day - D_i.
#    - Biến tiền tệ: TransactionAmt_log = log(1 + TransactionAmt), is_round_dollar (Amt_cents == 0).
#    - Biến khớp tên miền email 3 trạng thái (1: trùng, 0: khác, -1: thiếu 1 hoặc cả 2) + Provider + TLD.
#    - Tách biệt 3 khía cạnh thiết bị: device_brand (DeviceInfo), os_family (id_30), browser_family (id_31).
# 2. Nhóm biến Lịch sử Mở rộng Point-in-Time (PointInTimeCardHistoryTransformer):
#    - Với mỗi giao dịch ở giây t, chỉ lấy lịch sử nghiêm ngặt History(t) = {j : TransactionDT_j < t}.
#    - Loại bỏ toàn bộ giao dịch xảy ra cùng giây t (Timestamp Ties) để không tự bao hàm lẫn nhau.
#    - Làm mượt Bayesian (Bayesian Smoothing m=10) với GlobalMean(< t) cũng chỉ tính trước giây t.
# 3. Nhóm biến Mã hóa Tần suất Đóng băng (FrozenFrequencyEncoder):
#    - Chỉ fit bảng tần suất trên tập Temporal Train; giá trị mới ở Val nhận 0.0, NaN nhận -1.0.
# 4. Bộ 3 Unit Tests tự động (run_point_in_time_unit_tests):
#    - Test A (Historical Recalculation), Test B (Future Perturbation), Test C (Timestamp Ties & Order Invariance).


# ------------------------------------------------------------------------------
# 8.1. CÁC HÀM CHUẨN HÓA CHUỖI THIẾT BỊ, HỆ ĐIỀU HÀNH, TRÌNH DUYỆT & EMAIL
# ------------------------------------------------------------------------------
def _clean_device_brand(series: pd.Series) -> pd.Series:
    """Chuẩn hóa DeviceInfo về nhóm hãng sản xuất / dòng thiết bị chính (tránh nổ hàng nghìn nhãn)."""
    s = series.astype("object").fillna("").astype(str).str.lower().str.strip()
    out = np.full(len(s), "Other", dtype=object)
    out[s == ""] = "Missing"
    out[s.str.contains("windows|trident|rv:|microsoft", regex=True)] = "Windows"
    out[s.str.contains("ios|iphone|ipad|ipod", regex=True)] = "iOS_Device"
    out[s.str.contains("macos|macintosh|mac os", regex=True)] = "MacOS"
    out[s.str.contains("sm-|samsung|gt-|sch-|sgh-", regex=True)] = "Samsung"
    out[s.str.contains("huawei|hi6210|ale-|cam-|pra-|bnd-", regex=True)] = "Huawei"
    out[s.str.contains("moto|xt1|xt2", regex=True)] = "Motorola"
    out[s.str.contains("lg-|lgms|vs9|ls9", regex=True)] = "LG"
    return pd.Series(out, index=series.index, dtype="category")


def _clean_os_family(series: pd.Series) -> pd.Series:
    """Khai thác hệ điều hành từ cột id_30 (tách biệt với DeviceInfo)."""
    s = series.astype("object").fillna("").astype(str).str.lower().str.strip()
    out = np.full(len(s), "Other", dtype=object)
    out[s == ""] = "Missing"
    out[s.str.contains("windows", regex=False)] = "Windows"
    out[s.str.contains("ios", regex=False)] = "iOS"
    out[s.str.contains("mac", regex=False)] = "Mac"
    out[s.str.contains("android", regex=False)] = "Android"
    out[s.str.contains("linux", regex=False)] = "Linux"
    return pd.Series(out, index=series.index, dtype="category")


def _clean_browser_family(series: pd.Series) -> pd.Series:
    """Khai thác trình duyệt từ cột id_31 (tách biệt với DeviceInfo và id_30)."""
    s = series.astype("object").fillna("").astype(str).str.lower().str.strip()
    out = np.full(len(s), "Other", dtype=object)
    out[s == ""] = "Missing"
    out[s.str.contains("chrome", regex=False)] = "Chrome"
    out[s.str.contains("safari", regex=False) & ~s.str.contains("chrome", regex=False)] = "Safari"
    out[s.str.contains("ie|edge|trident", regex=True)] = "IE_Edge"
    out[s.str.contains("firefox", regex=False)] = "Firefox"
    out[s.str.contains("samsung", regex=False)] = "Samsung_Browser"
    out[s.str.contains("android", regex=False)] = "Android_Browser"
    return pd.Series(out, index=series.index, dtype="category")


def _extract_email_provider_and_tld(series: pd.Series) -> Tuple[pd.Series, pd.Series]:
    """Tách riêng nhóm nhà cung cấp dịch vụ email và đuôi tên miền (TLD)."""
    s = series.astype("object").fillna("").astype(str).str.lower().str.strip()

    provider = np.full(len(s), "other", dtype=object)
    provider[s == ""] = "missing"
    provider[s.str.contains("gmail|google", regex=True)] = "gmail"
    provider[s.str.contains("yahoo|ymail|rocketmail", regex=True)] = "yahoo"
    provider[s.str.contains("hotmail|outlook|live|msn", regex=True)] = "microsoft"
    provider[s.str.contains("icloud|me.com|mac.com", regex=True)] = "apple"
    provider[s.str.contains("aol", regex=False)] = "aol"
    provider[s.str.contains("anonymous", regex=False)] = "anonymous"

    # Tách đuôi TLD (ví dụ .com, .mx, .es, .net)
    tld = s.str.extract(r"\.([a-z]+)$", expand=False).fillna("")
    tld_grouped = np.where(
        tld == "",
        "missing",
        np.where(tld.isin(["com", "mx", "es", "net", "de", "fr", "uk", "jp"]), tld, "other_tld"),
    )

    return (
        pd.Series(provider, index=series.index, dtype="category"),
        pd.Series(tld_grouped, index=series.index, dtype="category"),
    )


# ------------------------------------------------------------------------------
# 8.2. BỘ TẠO ĐẶC TRƯNG STATELESS (KHÔNG PHỤ THUỘC THỐNG KÊ TẬP HỢP)
# ------------------------------------------------------------------------------
def generate_stateless_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tạo toàn bộ các biến Stateless (chỉ phụ thuộc vào từng dòng giao dịch hiện tại,
    0% rủi ro rò rỉ giữa các dòng hoặc nhìn trước tương lai).
    """
    out = df.copy()

    # 1. Đảm bảo đã có has_identity và Amt_cents từ Bước 0
    if "has_identity" not in out.columns:
        id_cols_all = [f"id_{i:02d}" for i in range(1, 39)] + ["DeviceType", "DeviceInfo"]
        present_id = [c for c in id_cols_all if c in out.columns]
        out["has_identity"] = out[present_id].notna().any(axis=1).astype("int8")

    if "Amt_cents" not in out.columns:
        amt_f64 = out["TransactionAmt"].to_numpy(dtype="float64")
        out["Amt_cents"] = (np.round(amt_f64 * 100.0).astype("int64") % 100).astype("int8")

    # 2. Số lượng và tỷ lệ khuyết chuẩn hóa theo từng họ biến (C, D, M, V, và Identity có điều kiện)
    c_cols = [f"C{i}" for i in range(1, 15) if f"C{i}" in out.columns]
    d_cols = [f"D{i}" for i in range(1, 16) if f"D{i}" in out.columns]
    m_cols = [f"M{i}" for i in range(1, 10) if f"M{i}" in out.columns]
    v_cols = [f"V{i}" for i in range(1, 340) if f"V{i}" in out.columns]
    id_cols = [f"id_{i:02d}" for i in range(1, 39) if f"id_{i:02d}" in out.columns] + [
        c for c in ["DeviceType", "DeviceInfo"] if c in out.columns
    ]

    out["n_missing_C"] = out[c_cols].isna().sum(axis=1).astype("int8")
    out["missing_rate_C"] = (out["n_missing_C"] / max(len(c_cols), 1)).astype("float32")

    out["n_missing_D"] = out[d_cols].isna().sum(axis=1).astype("int8")
    out["missing_rate_D"] = (out["n_missing_D"] / max(len(d_cols), 1)).astype("float32")

    out["n_missing_M"] = out[m_cols].isna().sum(axis=1).astype("int8")
    out["missing_rate_M"] = (out["n_missing_M"] / max(len(m_cols), 1)).astype("float32")

    out["n_missing_V"] = out[v_cols].isna().sum(axis=1).astype("int16")
    out["missing_rate_V"] = (out["n_missing_V"] / max(len(v_cols), 1)).astype("float32")

    # Khi has_identity == 1: đếm số trường thiếu bên trong bản ghi Identity (0..39).
    # Khi has_identity == 0: gán -1 để không mã hóa trùng lại has_identity.
    raw_id_miss = out[id_cols].isna().sum(axis=1).to_numpy(dtype="int16")
    has_id_arr = out["has_identity"].to_numpy()
    out["n_missing_id"] = np.where(has_id_arr == 1, raw_id_miss, -1).astype("int8")
    out["missing_rate_id"] = np.where(
        has_id_arr == 1,
        raw_id_miss / max(len(id_cols), 1),
        -1.0,
    ).astype("float32")

    # 3. Đặc trưng thời gian chu kỳ tương đối từ TransactionDT (timedelta tính bằng giây)
    dt_sec = out["TransactionDT"].to_numpy(dtype="int64")
    elapsed_day = (dt_sec // 86400).astype("int32")
    elapsed_hour = ((dt_sec // 3600) % 24).astype("int8")
    day_cycle = (elapsed_day % 7).astype("int8")

    out["elapsed_day"] = elapsed_day  # Biến trung gian phục vụ tính D_anchor, sẽ bị loại khỏi ma trận X ở Bước 3
    out["elapsed_hour"] = elapsed_hour
    out["day_cycle"] = day_cycle
    out["hour_sin"] = np.sin(2.0 * np.pi * elapsed_hour / 24.0).astype("float32")
    out["hour_cos"] = np.cos(2.0 * np.pi * elapsed_hour / 24.0).astype("float32")
    out["day_sin"] = np.sin(2.0 * np.pi * day_cycle / 7.0).astype("float32")
    out["day_cos"] = np.cos(2.0 * np.pi * day_cycle / 7.0).astype("float32")

    # 4. Biến thử nghiệm neo mốc thời gian tương đối cho họ D (D1_anchor .. D15_anchor)
    elapsed_day_f32 = elapsed_day.astype("float32")
    for d_col in d_cols:
        out[f"{d_col}_anchor"] = (elapsed_day_f32 - out[d_col].astype("float32")).astype("float32")

    # 5. Biến tiền tệ (TransactionAmt_log & is_round_dollar)
    amt_f32 = out["TransactionAmt"].astype("float32").clip(lower=0.0)
    out["TransactionAmt_log"] = np.log1p(amt_f32).astype("float32")
    out["is_round_dollar"] = (out["Amt_cents"] == 0).astype("int8")

    # 6. Biến Email 3 trạng thái (1: trùng, 0: khác, -1: thiếu 1 hoặc cả 2) + Provider + TLD
    if "P_emaildomain" in out.columns and "R_emaildomain" in out.columns:
        p_raw = out["P_emaildomain"].astype("object")
        r_raw = out["R_emaildomain"].astype("object")
        both_present = p_raw.notna() & r_raw.notna() & (p_raw.astype(str) != "") & (r_raw.astype(str) != "")
        same_dom = p_raw.astype(str).str.lower() == r_raw.astype(str).str.lower()
        out["email_domain_match"] = np.where(~both_present, -1, np.where(same_dom, 1, 0)).astype("int8")

        out["P_email_provider"], out["P_email_tld"] = _extract_email_provider_and_tld(out["P_emaildomain"])
        out["R_email_provider"], out["R_email_tld"] = _extract_email_provider_and_tld(out["R_emaildomain"])

    # 7. Tách riêng 3 biến Thiết bị / Hệ điều hành / Trình duyệt
    if "DeviceInfo" in out.columns:
        out["device_brand"] = _clean_device_brand(out["DeviceInfo"])
    if "id_30" in out.columns:
        out["os_family"] = _clean_os_family(out["id_30"])
    if "id_31" in out.columns:
        out["browser_family"] = _clean_browser_family(out["id_31"])

    return out


# ------------------------------------------------------------------------------
# 8.3. POINT-IN-TIME EXPANDING HISTORY (< t NGHIÊM NGẶT) CHO CARD1
# ------------------------------------------------------------------------------
class PointInTimeCardHistoryTransformer:
    """
    Tính toán các đặc trưng lịch sử nghiêm ngặt trước thời điểm t:
        History(t) = { j : TransactionDT_j < t }
    Đảm bảo:
    1. Loại bỏ toàn bộ các giao dịch có cùng TransactionDT == t (Timestamp Ties).
    2. GlobalMean(< t) cũng chỉ được tính từ các giao dịch có TransactionDT < t.
    3. Xử lý 3 trạng thái Cold-start:
       - card1 đã xuất hiện trước t (cnt_prior > 0): dùng Bayesian smoothed mean + has_card1_history = 1.
       - card1 mới xuất hiện lần đầu (cnt_prior == 0): dùng GlobalMean(< t) + has_card1_history = 0.
       - card1 bị thiếu (NaN): giữ NaN + card1_is_missing = 1.
       - Giao dịch ở giây đầu tiên (chưa có GlobalMean(< t)): gán NaN + no_global_history = 1.
    4. Bất biến hoàn toàn với thứ tự dòng (Row-Order Invariance).
    """

    def __init__(self, smoothing_m: float = 10.0, mode: str = "online") -> None:
        assert mode in ("online", "frozen"), "mode phải là 'online' hoặc 'frozen'"
        self.smoothing_m = float(smoothing_m)
        self.mode = mode
        self.train_history_df_: Optional[pd.DataFrame] = None
        self.frozen_card_table_: Optional[pd.DataFrame] = None
        self.frozen_global_mean_: float = np.nan

    @staticmethod
    def _compute_strict_prior_tables(base_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        valid_amt = base_df["TransactionAmt"].notna() & np.isfinite(base_df["TransactionAmt"].to_numpy(dtype="float64"))
        clean_df = base_df.loc[valid_amt, ["TransactionDT", "card1", "TransactionAmt"]].copy()
        clean_df["TransactionDT"] = clean_df["TransactionDT"].astype("int64")
        clean_df["TransactionAmt"] = clean_df["TransactionAmt"].astype("float64")

        # 1. Lịch sử toàn cục (Global History < t) theo từng mốc TransactionDT duy nhất
        g_ts = (
            clean_df.groupby("TransactionDT", as_index=False)["TransactionAmt"]
            .agg(ts_cnt="count", ts_sum="sum")
            .sort_values("TransactionDT", kind="mergesort")
            .reset_index(drop=True)
        )
        g_ts["global_cnt_prior"] = g_ts["ts_cnt"].cumsum() - g_ts["ts_cnt"]
        g_ts["global_sum_prior"] = g_ts["ts_sum"].cumsum() - g_ts["ts_sum"]
        g_ts["global_mean_prior"] = np.where(
            g_ts["global_cnt_prior"] > 0,
            g_ts["global_sum_prior"] / np.maximum(g_ts["global_cnt_prior"], 1),
            np.nan,
        )

        # 2. Lịch sử theo từng (card1, TransactionDT) cho các dòng card1 không bị thiếu
        card_valid = clean_df[clean_df["card1"].notna()].copy()
        card_ts = (
            card_valid.groupby(["card1", "TransactionDT"], as_index=False)["TransactionAmt"]
            .agg(c_cnt="count", c_sum="sum")
            .sort_values(["card1", "TransactionDT"], kind="mergesort")
            .reset_index(drop=True)
        )
        card_ts["card1_cnt_prior"] = card_ts.groupby("card1")["c_cnt"].cumsum() - card_ts["c_cnt"]
        card_ts["card1_sum_prior"] = card_ts.groupby("card1")["c_sum"].cumsum() - card_ts["c_sum"]

        return (
            g_ts[["TransactionDT", "global_cnt_prior", "global_sum_prior", "global_mean_prior"]],
            card_ts[["card1", "TransactionDT", "card1_cnt_prior", "card1_sum_prior"]],
        )

    def fit(self, train_df: pd.DataFrame) -> "PointInTimeCardHistoryTransformer":
        cols = ["TransactionID", "TransactionDT", "card1", "TransactionAmt"]
        present_cols = [c for c in cols if c in train_df.columns]
        self.train_history_df_ = train_df[present_cols].copy()

        valid = self.train_history_df_["TransactionAmt"].notna()
        sub = self.train_history_df_.loc[valid]
        self.frozen_global_mean_ = float(sub["TransactionAmt"].astype("float64").mean()) if len(sub) > 0 else np.nan
        self.frozen_card_table_ = (
            sub[sub["card1"].notna()]
            .groupby("card1", as_index=False)["TransactionAmt"]
            .agg(frozen_cnt="count", frozen_sum="sum")
        )
        return self

    def transform(self, df: pd.DataFrame, is_train_split: bool = False) -> pd.DataFrame:
        assert self.train_history_df_ is not None, "Phải gọi .fit(train_df) trước khi .transform()!"
        out = df.copy()

        if is_train_split or self.mode == "online":
            if is_train_split:
                combined_hist = out[["TransactionDT", "card1", "TransactionAmt"]].copy()
            else:
                if "TransactionID" in out.columns and "TransactionID" in self.train_history_df_.columns:
                    extra_train = self.train_history_df_[
                        ~self.train_history_df_["TransactionID"].isin(out["TransactionID"])
                    ][["TransactionDT", "card1", "TransactionAmt"]]
                else:
                    min_dt_cur = out["TransactionDT"].min()
                    extra_train = self.train_history_df_[
                        self.train_history_df_["TransactionDT"] < min_dt_cur
                    ][["TransactionDT", "card1", "TransactionAmt"]]
                combined_hist = pd.concat(
                    [extra_train, out[["TransactionDT", "card1", "TransactionAmt"]]],
                    ignore_index=True,
                )

            g_ts, card_ts = self._compute_strict_prior_tables(combined_hist)

            lookup_df = pd.DataFrame(
                {
                    "TransactionDT": out["TransactionDT"].astype("int64").to_numpy(),
                    "card1": out["card1"].to_numpy(),
                },
                index=out.index,
            )
            merged = lookup_df.merge(g_ts, on="TransactionDT", how="left")
            merged = merged.merge(card_ts, on=["card1", "TransactionDT"], how="left")

            g_cnt_prior = merged["global_cnt_prior"].fillna(0).to_numpy(dtype="float64")
            g_mean_prior = merged["global_mean_prior"].to_numpy(dtype="float64")
            c_cnt_prior = merged["card1_cnt_prior"].to_numpy(dtype="float64")
            c_sum_prior = merged["card1_sum_prior"].to_numpy(dtype="float64")
        else:
            assert self.frozen_card_table_ is not None
            lookup_df = pd.DataFrame({"card1": out["card1"].to_numpy()}, index=out.index)
            merged = lookup_df.merge(self.frozen_card_table_, on="card1", how="left")
            g_cnt_prior = np.full(len(out), len(self.train_history_df_), dtype="float64")
            g_mean_prior = np.full(len(out), self.frozen_global_mean_, dtype="float64")
            c_cnt_prior = merged["frozen_cnt"].fillna(0.0).to_numpy(dtype="float64")
            c_sum_prior = merged["frozen_sum"].fillna(0.0).to_numpy(dtype="float64")

        card_missing_mask = out["card1"].isna().to_numpy()
        no_global_mask = (g_cnt_prior <= 0) | (~np.isfinite(g_mean_prior))

        c_cnt_clean = np.where(card_missing_mask, np.nan, np.nan_to_num(c_cnt_prior, nan=0.0))
        c_sum_clean = np.where(card_missing_mask, np.nan, np.nan_to_num(c_sum_prior, nan=0.0))

        has_card_hist = (~card_missing_mask) & (c_cnt_clean > 0)

        m = self.smoothing_m
        smoothed_mean = np.where(
            card_missing_mask | no_global_mask,
            np.nan,
            (c_sum_clean + m * g_mean_prior) / (c_cnt_clean + m),
        )

        amt_arr = out["TransactionAmt"].to_numpy(dtype="float64")
        amt_ratio = np.where(
            np.isfinite(smoothed_mean) & (smoothed_mean > 0),
            amt_arr / smoothed_mean,
            np.nan,
        )

        out["card1_is_missing"] = card_missing_mask.astype("int8")
        out["no_global_history"] = no_global_mask.astype("int8")
        out["has_card1_history"] = has_card_hist.astype("int8")
        out["card1_cnt_prior"] = c_cnt_clean.astype("float32")
        out["card1_cnt_prior_log"] = np.where(
            np.isfinite(c_cnt_clean), np.log1p(np.maximum(c_cnt_clean, 0.0)), np.nan
        ).astype("float32")
        out["card1_mean_amt_prior"] = smoothed_mean.astype("float32")
        out["Amt_to_card1_mean_prior"] = amt_ratio.astype("float32")

        return out


# ------------------------------------------------------------------------------
# 8.4. FROZEN BATCH FREQUENCY ENCODER CHO BIẾN PHÂN LOẠI NHIỀU GIÁ TRỊ
# ------------------------------------------------------------------------------
class FrozenFrequencyEncoder:
    """
    Mã hóa tần suất xuất hiện (proportion trên tập Train) cho các biến phân loại nhiều giá trị.
    - Học bảng tần suất chỉ trên Temporal Train.
    - Giá trị chưa từng xuất hiện ở Validation/Test nhận 0.0; giá trị NaN nhận -1.0.
    """

    def __init__(self, cols: List[str]) -> None:
        self.cols = list(cols)
        self.freq_maps_: Dict[str, Dict[object, float]] = {}

    def fit(self, train_df: pd.DataFrame) -> "FrozenFrequencyEncoder":
        self.freq_maps_.clear()
        n_train = max(len(train_df), 1)
        for col in self.cols:
            if col not in train_df.columns:
                continue
            vc = train_df[col].dropna().value_counts()
            self.freq_maps_[col] = (vc / float(n_train)).to_dict()
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        for col, fmap in self.freq_maps_.items():
            if col not in out.columns:
                continue
            s = out[col]
            mapped = s.map(fmap).astype("float64").fillna(0.0)
            mapped = np.where(s.isna(), -1.0, mapped)
            out[f"{col}_freq"] = mapped.astype("float32")
        return out


# ------------------------------------------------------------------------------
# 8.5. BỘ 3 BÀI KIỂM THỬ POINT-IN-TIME ĐỘC LẬP (UNIT TESTS A, B, C)
# ------------------------------------------------------------------------------
def _naive_reference_card1_point_in_time(
    df: pd.DataFrame, row_idx: int, smoothing_m: float = 10.0
) -> Dict[str, float]:
    """Hàm tham chiếu độc lập (vét cạn từng dòng trên lịch sử < t) dùng để kiểm định Test A."""
    row = df.iloc[row_idx]
    t_cur = int(row["TransactionDT"])
    c_cur = row["card1"]
    amt_cur = float(row["TransactionAmt"])

    hist = df[(df["TransactionDT"].astype("int64") < t_cur) & df["TransactionAmt"].notna()]
    g_cnt = len(hist)

    if pd.isna(c_cur):
        return {
            "card1_is_missing": 1,
            "no_global_history": int(g_cnt == 0),
            "has_card1_history": 0,
            "card1_cnt_prior": np.nan,
            "card1_mean_amt_prior": np.nan,
            "Amt_to_card1_mean_prior": np.nan,
        }

    if g_cnt == 0:
        return {
            "card1_is_missing": 0,
            "no_global_history": 1,
            "has_card1_history": 0,
            "card1_cnt_prior": 0.0,
            "card1_mean_amt_prior": np.nan,
            "Amt_to_card1_mean_prior": np.nan,
        }

    g_mean = float(hist["TransactionAmt"].astype("float64").mean())
    c_hist = hist[hist["card1"] == c_cur]
    c_cnt = float(len(c_hist))
    c_sum = float(c_hist["TransactionAmt"].astype("float64").sum()) if c_cnt > 0 else 0.0

    mean_prior = (c_sum + smoothing_m * g_mean) / (c_cnt + smoothing_m)
    ratio_prior = amt_cur / mean_prior if mean_prior > 0 else np.nan

    return {
        "card1_is_missing": 0,
        "no_global_history": 0,
        "has_card1_history": int(c_cnt > 0),
        "card1_cnt_prior": c_cnt,
        "card1_mean_amt_prior": mean_prior,
        "Amt_to_card1_mean_prior": ratio_prior,
    }


def run_point_in_time_unit_tests(df: pd.DataFrame, smoothing_m: float = 10.0) -> Dict[str, bool]:
    """
    Thực thi 3 bài kiểm thử Point-in-Time bắt buộc (Test A, Test B, Test C).
    """
    pit = PointInTimeCardHistoryTransformer(smoothing_m=smoothing_m, mode="online")
    pit.fit(df)
    transformed = pit.transform(df, is_train_split=True)

    # TEST A: HISTORICAL RECALCULATION
    min_dt = df["TransactionDT"].min()
    first_dt_indices = np.flatnonzero(df["TransactionDT"].to_numpy() == min_dt)[:5].tolist()

    dt_counts = df["TransactionDT"].value_counts()
    tied_dts = dt_counts[dt_counts > 1].index.tolist()
    tie_indices = (
        np.flatnonzero(df["TransactionDT"].isin(tied_dts[:3])).tolist()[:10]
        if tied_dts
        else []
    )

    na_card_indices = np.flatnonzero(df["card1"].isna().to_numpy())[:10].tolist()
    cold_card_indices = np.flatnonzero(
        (transformed["has_card1_history"].to_numpy() == 0)
        & (transformed["card1_is_missing"].to_numpy() == 0)
    )[:10].tolist()

    rng = np.random.default_rng(42)
    random_indices = rng.choice(len(df), size=min(40, len(df)), replace=False).tolist()
    test_a_indices = sorted(set(first_dt_indices + tie_indices + na_card_indices + cold_card_indices + random_indices))

    for idx in test_a_indices:
        ref = _naive_reference_card1_point_in_time(df, idx, smoothing_m=smoothing_m)
        row_t = transformed.iloc[idx]
        for flag_col in ["card1_is_missing", "no_global_history", "has_card1_history"]:
            assert int(row_t[flag_col]) == int(ref[flag_col]), (
                f"Test A thất bại tại dòng {idx}, cột {flag_col}: {row_t[flag_col]} != {ref[flag_col]}"
            )
        for num_col in ["card1_cnt_prior", "card1_mean_amt_prior", "Amt_to_card1_mean_prior"]:
            assert np.allclose(
                np.float32(row_t[num_col]),
                np.float32(ref[num_col]),
                rtol=1e-5,
                atol=1e-5,
                equal_nan=True,
            ), f"Test A thất bại tại dòng {idx}, cột {num_col}: {row_t[num_col]} != {ref[num_col]}"

    # TEST B: FUTURE PERTURBATION
    t_cut = float(df["TransactionDT"].quantile(0.55))
    df_perturbed = df.copy()
    future_mask = df_perturbed["TransactionDT"] >= t_cut
    df_perturbed.loc[future_mask, "TransactionAmt"] = df_perturbed.loc[future_mask, "TransactionAmt"] * 50.0 + 999.0
    df_perturbed.loc[future_mask, "card1"] = 999999.0

    pit_pert = PointInTimeCardHistoryTransformer(smoothing_m=smoothing_m, mode="online")
    pit_pert.fit(df_perturbed)
    trans_perturbed = pit_pert.transform(df_perturbed, is_train_split=True)

    past_mask = df["TransactionDT"] < t_cut
    pit_cols = [
        "card1_is_missing",
        "no_global_history",
        "has_card1_history",
        "card1_cnt_prior",
        "card1_cnt_prior_log",
        "card1_mean_amt_prior",
        "Amt_to_card1_mean_prior",
    ]
    for col in pit_cols:
        a_orig = transformed.loc[past_mask, col].to_numpy()
        a_pert = trans_perturbed.loc[past_mask, col].to_numpy()
        assert np.allclose(a_orig, a_pert, rtol=0.0, atol=0.0, equal_nan=True), (
            f"Test B (Future Perturbation) thất bại ở cột {col}!"
        )

    # TEST C: TIMESTAMP TIES & ROW-ORDER INVARIANCE
    tie_toy = pd.DataFrame(
        {
            "TransactionID": [1, 2, 3, 4],
            "TransactionDT": [5000, 10000, 10000, 10000],
            "card1": [123.0, 123.0, 123.0, 123.0],
            "TransactionAmt": [100.0, 200.0, 500.0, 900.0],
        }
    )
    pit_toy = PointInTimeCardHistoryTransformer(smoothing_m=10.0, mode="online").fit(tie_toy)
    res_toy = pit_toy.transform(tie_toy, is_train_split=True)
    tied_rows = res_toy[res_toy["TransactionDT"] == 10000]
    assert np.all(tied_rows["card1_cnt_prior"].to_numpy() == 1.0), "Test C thất bại: Cùng timestamp bị tự bao hàm!"
    assert np.allclose(tied_rows["card1_mean_amt_prior"].to_numpy(), 100.0), (
        "Test C thất bại: Trung bình lịch sử của các dòng trùng timestamp không đồng nhất!"
    )

    shuffled_df = df.sample(frac=1.0, random_state=999).reset_index(drop=True)
    pit_shuf = PointInTimeCardHistoryTransformer(smoothing_m=smoothing_m, mode="online").fit(shuffled_df)
    trans_shuf = pit_shuf.transform(shuffled_df, is_train_split=True)

    orig_by_id = transformed.sort_values("TransactionID").reset_index(drop=True)
    shuf_by_id = trans_shuf.sort_values("TransactionID").reset_index(drop=True)
    for col in pit_cols:
        assert np.allclose(
            orig_by_id[col].to_numpy(),
            shuf_by_id[col].to_numpy(),
            rtol=1e-6,
            atol=1e-6,
            equal_nan=True,
        ), f"Test C (Row-Order Invariance) thất bại ở cột {col}!"

    return {
        "Test_A_Historical_Recalculation": True,
        "Test_B_Future_Perturbation": True,
        "Test_C_Timestamp_Ties_and_Order_Invariance": True,
    }


# ------------------------------------------------------------------------------
# 8.6. HÀM CHẠY TRỌN GÓI BƯỚC 2 (PIPELINE ENTRYPOINT)
# ------------------------------------------------------------------------------
def run_step2_feature_engineering(
    df: pd.DataFrame,
    split_df: pd.DataFrame,
    output_dir: str = str(STEP2_OUTPUT_DIR),
    history_mode: str = "online",
) -> Dict[str, object]:
    """
    Thực thi trọn gói Bước 2 (Feature Engineering & Point-in-Time Unit Tests A, B, C):
    1. Chạy kiểm thử tự động Test A, Test B, Test C trước tiên.
    2. Tạo toàn bộ biến Stateless (khuyết theo họ, thời gian chu kỳ, D_anchor, tiền tệ, email, thiết bị).
    3. Fit PointInTimeCardHistoryTransformer và FrozenFrequencyEncoder chỉ trên Temporal Train,
       sau đó biến đổi cho cả Temporal Train và Temporal Validation.
    4. Lưu bảng tóm tắt đặc trưng mới vào OUTPUT_DIR / 'artifacts_step2_fe'.
    """
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    train_mask = (split_df["temporal_role"].to_numpy() == "train")
    val_mask = (split_df["temporal_role"].to_numpy() == "val")

    # 1. Chạy bộ 3 Unit Tests Point-in-Time (A, B, C) trên tập Temporal Train
    unit_test_results = run_point_in_time_unit_tests(df.loc[train_mask].copy(), smoothing_m=10.0)

    # 2. Tạo các biến Stateless cho toàn bảng (vì Stateless chỉ tính trên từng dòng độc lập)
    fe_df = generate_stateless_features(df)

    # 3. Fit và Transform đặc trưng lịch sử Point-in-Time (< t) của card1
    pit_transformer = PointInTimeCardHistoryTransformer(smoothing_m=10.0, mode=history_mode)
    pit_transformer.fit(fe_df.loc[train_mask])

    fe_train = pit_transformer.transform(fe_df.loc[train_mask], is_train_split=True)
    fe_val = pit_transformer.transform(fe_df.loc[val_mask], is_train_split=False)
    fe_full = pd.concat([fe_train, fe_val], axis=0).loc[df.index]

    # 4. Fit FrozenFrequencyEncoder chỉ trên Temporal Train cho các biến nhiều giá trị
    high_card_cols = ["card1", "card2", "card3", "card5", "addr1", "addr2", "P_emaildomain", "R_emaildomain"]
    freq_encoder = FrozenFrequencyEncoder(cols=high_card_cols)
    freq_encoder.fit(fe_full.loc[train_mask])
    fe_full = freq_encoder.transform(fe_full)

    # Tổng hợp danh sách các biến mới vừa được tạo ở Bước 2
    new_cols = [c for c in fe_full.columns if c not in df.columns]
    summary_rows = []
    for col in new_cols:
        s_tr = fe_full.loc[train_mask, col]
        summary_rows.append(
            {
                "New_Feature": col,
                "Dtype": str(s_tr.dtype),
                "Train_Missing_Rate_pct": round(float(s_tr.isna().mean() * 100.0), 2),
                "Unique_Values_Train": int(s_tr.nunique(dropna=True)),
            }
        )
    new_features_summary_df = pd.DataFrame(summary_rows)
    new_features_summary_df.to_csv(out_path / "fe_step2_new_features_summary.csv", index=False)

    print("=" * 78)
    print("HOÀN TẤT BƯỚC 2: POINT-IN-TIME FEATURE ENGINEERING & UNIT TESTS (A, B, C)")
    print("=" * 78)
    print("- Kết quả kiểm thử Point-in-Time Unit Tests:")
    for t_name, passed in unit_test_results.items():
        print(f"  + {t_name}: {'PASSED' if passed else 'FAILED'}")
    print(f"- Số lượng đặc trưng mới được tạo thêm ở Bước 2: {len(new_cols)} biến")
    print(f"- Tổng số cột hiện tại của bảng dữ liệu        : {fe_full.shape[1]} cột")
    print("\n[Bảng tóm tắt các nhóm biến mới tạo ở Bước 2 (20 biến tiêu biểu đầu tiên)]:")
    print(new_features_summary_df.head(20).to_string(index=False))

    return {
        "fe_df": fe_full,
        "new_feature_names": new_cols,
        "new_features_summary_df": new_features_summary_df,
        "unit_test_results": unit_test_results,
        "pit_transformer": pit_transformer,
        "freq_encoder": freq_encoder,
    }

print("Đã khởi tạo xong toàn bộ hàm và class của Bước 2 (step2_feature_engineering).")

### CELL 9: Thực thi Bước 2 (Feature Engineering & Unit Tests A, B, C)


In [ ]:
# ==============================================================================
# CELL 9: THỰC THI BƯỚC 2 (FEATURE ENGINEERING & UNIT TESTS A, B, C)
# ==============================================================================
# Giải thích:
# - Chạy hàm run_step2_feature_engineering() trên dữ liệu từ Bước 0.
# - Lưu bảng tóm tắt các biến mới vào `/content/drive/MyDrive/IEEE_CIS_Project/artifacts/artifacts_step2_fe`.
# - Giải phóng `step0_res["df"]` ngay sau khi hoàn tất Bước 2 vì toàn bộ dữ liệu
#   đã nằm trong `step2_res["fe_df"]` (493 cột).

print("\n========== RUNNING STEP 2 ==========")
step2_res = run_step2_feature_engineering(
    df=step0_res["df"],
    split_df=step0_res["split_df"],
    output_dir=str(OUTPUT_DIR / "artifacts_step2_fe"),
)
print("STEP 2 COMPLETED!")

# Giải phóng df của Step 0 vì đã có fe_df ở Step 2
del step0_res["df"]
gc.collect()

### CELL 10: Định nghĩa các Hàm & Class Bước 3 & 4 - Feature Selection & Handover (step3_4_selection_and_handover.py)


In [ ]:
# ==============================================================================
# CELL 10: ĐỊNH NGHĨA CÁC HÀM & CLASS BƯỚC 3 & 4 - FEATURE SELECTION & HANDOVER
#          (Chuyển đổi từ file: step3_4_selection_and_handover.py)
# ==============================================================================
# Giải thích chuyên môn:
# 1. Cấm tuyệt đối các cột định danh/mốc thời gian thô/nhãn:
#    FORBIDDEN_PREDICTOR_COLS = {"TransactionID", "isFraud", "TransactionDT", "elapsed_day"}.
# 2. Thí nghiệm Ablation 4 cấu hình cho họ biến D (run_d_family_ablation_study):
#    So sánh (A) Chỉ D gốc, (B) Chỉ D_anchor, (C) D gốc + D_anchor chọn lọc, (D) Không dùng họ D
#    trên các Inner Temporal Folds theo cả ROC-AUC và Average Precision (AP).
# 3. Lựa chọn đặc trưng 3 tầng (run_three_tier_feature_selection):
#    - Tầng 1: Sàng lọc theo Effect Size Lift + Benjamini-Hochberg FDR & loại biến hằng (Constant).
#    - Tầng 2: Khử đa cộng tuyến 2 vòng lặp theo từng cụm (|rho_Spearman| >= 0.95).
#    - Tầng 3: Đánh giá độ ổn định và Permutation Importance qua 5 Inner Temporal Folds.
# 4. Đóng gói 3 cấu hình Preprocessor chuyên biệt (ProductionTabularPreprocessor):
#    - 'linear'       : Điền Median + cờ _is_missing + log1p(họ C) + RobustScaler + One-Hot.
#    - 'sklearn_tree' : Điền Median + cờ _is_missing + One-Hot (dành cho DecisionTree / RandomForest).
#    - 'gbdt'         : Giữ nguyên NaN tự nhiên (Native Missing) + One-Hot (dành cho LightGBM / XGBoost).
# 5. Kiểm thử nghiệm thu cuối cùng (Test D: Validation Label Isolation & Test E: 0/NaN/Unseen & Reload).

FORBIDDEN_PREDICTOR_COLS = {"TransactionID", "isFraud", "TransactionDT", "elapsed_day"}


# ------------------------------------------------------------------------------
# 10.1. CÁC HÀM ĐO LƯỜNG ROC-AUC, AVERAGE PRECISION (AP) & FAST HISTOGRAM GBDT
# ------------------------------------------------------------------------------
def compute_roc_auc_score(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Tính diện tích dưới đường cong ROC (ROC-AUC) chính xác bằng thống kê hạng Mann-Whitney U."""
    y_t = np.asarray(y_true, dtype="int64")
    y_s = np.asarray(y_score, dtype="float64")
    n_pos = int(np.sum(y_t == 1))
    n_neg = int(np.sum(y_t == 0))
    if n_pos == 0 or n_neg == 0:
        return 0.5

    ranks = pd.Series(y_s).rank(method="average").to_numpy(dtype="float64")
    sum_pos_ranks = float(np.sum(ranks[y_t == 1]))
    auc = (sum_pos_ranks - n_pos * (n_pos + 1.0) / 2.0) / (n_pos * n_neg)
    return float(np.clip(auc, 0.0, 1.0))


def compute_average_precision_score(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """
    Tính Average Precision (AP) theo chuẩn scikit-learn:
        AP = sum_n (R_n - R_{n-1}) * P_n
    """
    y_t = np.asarray(y_true, dtype="int64")
    y_s = np.asarray(y_score, dtype="float64")
    n_pos = int(np.sum(y_t == 1))
    if n_pos == 0:
        return 0.0

    order = np.argsort(-y_s, kind="mergesort")
    y_sorted = y_t[order]
    tp_cum = np.cumsum(y_sorted == 1).astype("float64")
    k_positions = np.arange(1, len(y_sorted) + 1, dtype="float64")
    precision_at_k = tp_cum / k_positions
    ap = float(np.sum(precision_at_k[y_sorted == 1]) / n_pos)
    return float(np.clip(ap, 0.0, 1.0))


class FastHistogramGBDT:
    """
    Bộ phân loại GBDT có hỗ trợ NaN tự nhiên (Native Missing) và trọng số lớp mất cân bằng.
    Tự động sử dụng LightGBM Booster gốc trên Colab để đạt tốc độ và độ chính xác tối ưu.
    """

    def __init__(self, n_estimators: int = 25, learning_rate: float = 0.15, random_state: int = 42) -> None:
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.trees_: List[Tuple[int, float, bool, float, float]] = []
        self.base_score_: float = 0.0
        self.feature_gains_: Optional[np.ndarray] = None
        self.lgb_model_ = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "FastHistogramGBDT":
        X_arr = np.asarray(X, dtype="float32")
        y_arr = np.asarray(y, dtype="float64")
        n_samples, n_features = X_arr.shape
        self.feature_gains_ = np.zeros(n_features, dtype="float64")

        try:
            import lightgbm as lgb
            pos_cnt = max(float(np.sum(y_arr == 1)), 1.0)
            neg_cnt = max(float(np.sum(y_arr == 0)), 1.0)
            model = lgb.LGBMClassifier(
                n_estimators=max(self.n_estimators * 2, 50),
                learning_rate=0.08,
                num_leaves=31,
                scale_pos_weight=neg_cnt / pos_cnt,
                random_state=self.random_state,
                n_jobs=-1,
                verbose=-1,
            )
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", UserWarning)
                model.fit(X_arr, y_arr.astype("int32"))
            self.lgb_model_ = model
            self.feature_gains_ = np.asarray(model.booster_.feature_importance(importance_type="gain"), dtype="float64")
            return self
        except ImportError:
            pass

        p_init = np.clip(np.mean(y_arr), 1e-5, 1.0 - 1e-5)
        self.base_score_ = float(np.log(p_init / (1.0 - p_init)))
        raw_pred = np.full(n_samples, self.base_score_, dtype="float64")

        pos_weight = (1.0 - p_init) / max(p_init, 1e-5)
        sample_w = np.where(y_arr == 1, pos_weight, 1.0)

        nan_masks = np.isnan(X_arr)
        medians = np.nanmedian(X_arr, axis=0)
        medians = np.where(np.isfinite(medians), medians, 0.0)
        left_masks = (~nan_masks) & (X_arr <= medians.reshape(1, -1))

        self.trees_.clear()
        for _ in range(self.n_estimators):
            prob = 1.0 / (1.0 + np.exp(-np.clip(raw_pred, -15.0, 15.0)))
            grad = (y_arr - prob) * sample_w

            sum_grad_total = float(np.sum(grad))
            grad_col = grad.reshape(-1, 1)

            sum_g_left = np.sum(grad_col * left_masks, axis=0)
            cnt_left = np.maximum(np.sum(left_masks, axis=0), 1.0)

            sum_g_nan = np.sum(grad_col * nan_masks, axis=0)
            cnt_nan = np.sum(nan_masks, axis=0)

            s_l1 = sum_g_left + sum_g_nan
            n_l1 = cnt_left + cnt_nan
            s_r1 = sum_grad_total - s_l1
            n_r1 = np.maximum(n_samples - n_l1, 1.0)
            gain1 = (s_l1 ** 2) / n_l1 + (s_r1 ** 2) / n_r1

            s_l2 = sum_g_left
            n_l2 = cnt_left
            s_r2 = sum_grad_total - s_l2
            n_r2 = np.maximum(n_samples - n_l2, 1.0)
            gain2 = (s_l2 ** 2) / n_l2 + (s_r2 ** 2) / n_r2

            best_f1 = int(np.argmax(gain1))
            best_f2 = int(np.argmax(gain2))
            if gain1[best_f1] >= gain2[best_f2]:
                best_f = best_f1
                nan_go_left = True
                best_gain = float(gain1[best_f1])
                val_left = float(s_l1[best_f] / n_l1[best_f]) * self.learning_rate
                val_right = float(s_r1[best_f] / n_r1[best_f]) * self.learning_rate
                go_left_mask = left_masks[:, best_f] | nan_masks[:, best_f]
            else:
                best_f = best_f2
                nan_go_left = False
                best_gain = float(gain2[best_f2])
                val_left = float(s_l2[best_f] / n_l2[best_f]) * self.learning_rate
                val_right = float(s_r2[best_f] / n_r2[best_f]) * self.learning_rate
                go_left_mask = left_masks[:, best_f]

            self.trees_.append((best_f, float(medians[best_f]), nan_go_left, val_left, val_right))
            self.feature_gains_[best_f] += max(best_gain, 0.0)
            raw_pred += np.where(go_left_mask, val_left, val_right)

        return self

    def predict_proba_pos(self, X: np.ndarray) -> np.ndarray:
        X_arr = np.asarray(X, dtype="float32")
        if self.lgb_model_ is not None:
            return np.asarray(self.lgb_model_.booster_.predict(X_arr), dtype="float64")

        raw = np.full(X_arr.shape[0], self.base_score_, dtype="float64")
        for best_f, thr, nan_go_left, val_left, val_right in self.trees_:
            col = X_arr[:, best_f]
            is_na = np.isnan(col)
            if nan_go_left:
                go_left = is_na | (col <= thr)
            else:
                go_left = (~is_na) & (col <= thr)
            raw += np.where(go_left, val_left, val_right)
        return 1.0 / (1.0 + np.exp(-np.clip(raw, -15.0, 15.0)))


# ------------------------------------------------------------------------------
# 10.2. THÍ NGHIỆM ABLATION 4 CẤU HÌNH CHO HỌ BIẾN D (D GỐC VS D_ANCHOR)
# ------------------------------------------------------------------------------
def run_d_family_ablation_study(
    train_df: pd.DataFrame,
    n_folds: int = 3,
) -> Tuple[pd.DataFrame, List[str], List[str]]:
    """
    Thực hiện thí nghiệm Ablation 4 cấu hình trên các Inner Temporal Folds của tập Train:
    - Config A: Chỉ dùng D1..D15 gốc (bỏ D1_anchor..D15_anchor).
    - Config B: Chỉ dùng D1_anchor..D15_anchor (bỏ D1..D15 gốc).
    - Config C: Kết hợp D1..D15 gốc và các biến D_anchor có cải thiện đơn biến trên Validation.
    - Config D: Không sử dụng họ biến D.
    """
    sorted_df = train_df.sort_values("TransactionDT", kind="mergesort").reset_index(drop=True)
    y_all = sorted_df["isFraud"].to_numpy(dtype="int64")

    raw_d_cols = [f"D{i}" for i in range(1, 16) if f"D{i}" in sorted_df.columns]
    anchor_d_cols = [f"D{i}_anchor" for i in range(1, 16) if f"D{i}_anchor" in sorted_df.columns]
    core_cols = [
        c
        for c in ["TransactionAmt", "TransactionAmt_log", "Amt_cents", "has_identity", "C1", "C2", "card1_cnt_prior"]
        if c in sorted_df.columns
    ]

    quantiles = np.linspace(0.0, 1.0, n_folds + 2)
    dt_bounds = [float(sorted_df["TransactionDT"].quantile(q)) for q in quantiles]

    helpful_anchors: List[str] = []
    for d_col in raw_d_cols:
        anc_col = f"{d_col}_anchor"
        if anc_col not in sorted_df.columns:
            continue
        raw_s = sorted_df[d_col].to_numpy(dtype="float64")
        anc_s = sorted_df[anc_col].to_numpy(dtype="float64")
        valid = np.isfinite(raw_s) & np.isfinite(anc_s)
        if valid.sum() < 50:
            continue
        auc_raw = abs(compute_roc_auc_score(y_all[valid], raw_s[valid]) - 0.5)
        auc_anc = abs(compute_roc_auc_score(y_all[valid], anc_s[valid]) - 0.5)
        if auc_anc >= auc_raw:
            helpful_anchors.append(anc_col)

    if not helpful_anchors and anchor_d_cols:
        helpful_anchors = anchor_d_cols[:5]

    configs = {
        "A_Raw_D_Only": core_cols + raw_d_cols,
        "B_Anchor_D_Only": core_cols + anchor_d_cols,
        "C_Raw_plus_Selected_Anchor": core_cols + raw_d_cols + helpful_anchors,
        "D_No_D_Features": core_cols,
    }

    ablation_rows = []
    for cfg_name, cols in configs.items():
        X_mat = sorted_df[cols].to_numpy(dtype="float32")
        fold_aucs, fold_aps = [], []
        for k in range(1, n_folds + 1):
            tr_end, va_end = dt_bounds[k], dt_bounds[k + 1]
            tr_m = (sorted_df["TransactionDT"] <= tr_end).to_numpy()
            va_m = ((sorted_df["TransactionDT"] > tr_end) & (sorted_df["TransactionDT"] <= va_end)).to_numpy()
            if tr_m.sum() < 30 or va_m.sum() < 20 or y_all[va_m].sum() == 0:
                continue
            clf = FastHistogramGBDT(n_estimators=20, learning_rate=0.15, random_state=42).fit(X_mat[tr_m], y_all[tr_m])
            preds = clf.predict_proba_pos(X_mat[va_m])
            fold_aucs.append(compute_roc_auc_score(y_all[va_m], preds))
            fold_aps.append(compute_average_precision_score(y_all[va_m], preds))

        ablation_rows.append(
            {
                "Configuration": cfg_name,
                "Num_D_Features_Used": len(cols) - len(core_cols),
                "Mean_Temporal_ROC_AUC": round(float(np.mean(fold_aucs)) if fold_aucs else 0.5, 4),
                "Mean_Temporal_AP": round(float(np.mean(fold_aps)) if fold_aps else 0.0, 4),
            }
        )

    ablation_df = pd.DataFrame(ablation_rows)
    kept_d_cols = raw_d_cols + helpful_anchors
    dropped_d_cols = [c for c in anchor_d_cols if c not in helpful_anchors]

    return ablation_df, kept_d_cols, dropped_d_cols


# ------------------------------------------------------------------------------
# 10.3. LỰA CHỌN ĐẶC TRƯNG 3 TẦNG BÊN TRONG INNER TEMPORAL FOLDS (0% LEAKAGE)
# ------------------------------------------------------------------------------
def _compute_spearman_corr_matrix_fast(df_sub: pd.DataFrame) -> pd.DataFrame:
    """
    Tính ma trận tương quan hạng Spearman nhanh trên tập dữ liệu lớn bằng cách
    rank() trước rồi gọi corr(method='pearson') được tối ưu hóa bằng C/BLAS.
    """
    cols = list(df_sub.columns)
    if len(cols) <= 1:
        return pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)

    max_corr_rows = 50000
    if len(df_sub) > max_corr_rows:
        step = max(len(df_sub) // max_corr_rows, 1)
        work_df = df_sub.iloc[::step].iloc[:max_corr_rows]
    else:
        work_df = df_sub

    ranked_df = work_df.astype("float32").rank(method="average", na_option="keep")
    corr_mat = ranked_df.corr(method="pearson").abs()
    return corr_mat


def run_three_tier_feature_selection(
    train_df: pd.DataFrame,
    corr_threshold: float = 0.95,
    n_inner_folds: int = 5,
    n_perm_repeats: int = 3,
    random_state: int = 42,
) -> Dict[str, object]:
    """
    Thực hiện chọn biến 3 tầng hoàn toàn trên tập Temporal Train (không chạm vào Temporal Val):
    - Tầng 1 (Effect Size Lift + FDR & Quasi-constant Filter)
    - Tầng 2 (2-Pass Block-wise Spearman Correlation Pruning |rho| >= 0.95)
    - Tầng 3 (Inner Temporal Fold Stability & Permutation Importance theo AP & ROC-AUC)
    """
    candidate_cols = [c for c in train_df.columns if c not in FORBIDDEN_PREDICTOR_COLS]

    ablation_df, kept_d_cols, dropped_d_anchors = run_d_family_ablation_study(train_df)
    candidate_cols = [c for c in candidate_cols if c not in dropped_d_anchors]

    y_train = train_df["isFraud"].to_numpy(dtype="int64")

    # TẦNG 1: SÀNG LỌC THEO ĐỘ LỚN HIỆU ỨNG (LIFT + BH-FDR) & QUASI-CONSTANT
    identity_set = set([f"id_{i:02d}" for i in range(1, 39)] + ["DeviceType", "DeviceInfo"])
    id_mask = (train_df["has_identity"].to_numpy() == 1)

    tier1_records = []
    tier1_weak_candidates = set()

    for col in candidate_cols:
        is_id = col in identity_set
        sub_s = train_df.loc[id_mask, col] if is_id else train_df[col]
        sub_y = y_train[id_mask] if is_id else y_train
        n_sub = max(len(sub_y), 1)
        k_sub = int(sub_y.sum())

        miss_mask = sub_s.isna().to_numpy()
        n_miss = int(miss_mask.sum())
        n_pres = n_sub - n_miss
        miss_rate = n_miss / n_sub

        vc = sub_s.value_counts(dropna=False)
        dom_ratio = float(vc.iloc[0] / n_sub) if len(vc) > 0 else 1.0

        if 0 < n_miss < n_sub:
            k_miss = int(sub_y[miss_mask].sum())
            k_pres = k_sub - k_miss
            r_miss = k_miss / max(n_miss, 1)
            r_pres = k_pres / max(n_pres, 1)
            lift = r_pres / max(r_miss, 1e-9) if r_miss > 0 else 1.0
            diff_pp = abs(r_pres - r_miss) * 100.0
            pval = two_proportion_z_pvalue(k_pres, n_pres, k_miss, n_miss)
        else:
            lift = 1.0
            diff_pp = 0.0
            pval = 1.0

        tier1_records.append(
            {
                "Feature": col,
                "Conditional_Missing_Rate": round(miss_rate, 4),
                "Dominant_Value_Ratio": round(dom_ratio, 4),
                "Lift_vs_Complement": round(lift, 3),
                "Abs_Diff_pp": round(diff_pp, 3),
                "p_value": pval,
            }
        )

    tier1_df = pd.DataFrame(tier1_records)
    tier1_df["q_value_fdr"] = benjamini_hochberg_fdr(tier1_df["p_value"].to_numpy())

    for _, row in tier1_df.iterrows():
        col = str(row["Feature"])
        is_extreme_sparse_or_const = (row["Conditional_Missing_Rate"] >= 0.90) or (row["Dominant_Value_Ratio"] >= 0.995)
        is_neutral_effect = (0.90 <= row["Lift_vs_Complement"] <= 1.10) and (row["Abs_Diff_pp"] < 1.0)
        if is_extreme_sparse_or_const and is_neutral_effect:
            tier1_weak_candidates.add(col)

    constant_cols = set(tier1_df.loc[tier1_df["Dominant_Value_Ratio"] >= 1.0, "Feature"].tolist())
    after_tier1_cols = [c for c in candidate_cols if c not in constant_cols]

    # TẦNG 2: KHỬ ĐA CỘNG TUYẾN THEO CỤM VỚI 2 VÒNG LẶP (2-PASS CORRELATION PRUNING)
    num_candidates = [
        c
        for c in after_tier1_cols
        if pd.api.types.is_numeric_dtype(train_df[c]) and not pd.api.types.is_bool_dtype(train_df[c])
    ]

    univar_score: Dict[str, float] = {}
    na_rate_map: Dict[str, float] = {}

    n_total_tr = len(train_df)
    max_uni_rows = 80000
    if n_total_tr > max_uni_rows:
        uni_step = max(n_total_tr // max_uni_rows, 1)
        uni_df = train_df.iloc[::uni_step].iloc[:max_uni_rows]
        y_uni = y_train[::uni_step][:max_uni_rows]
    else:
        uni_df = train_df
        y_uni = y_train

    y_uni_f64 = y_uni.astype("float64")
    y_uni_std = float(np.std(y_uni_f64))

    for col in num_candidates:
        s_full = train_df[col]
        na_rate_map[col] = float(s_full.isna().mean())
        s = uni_df[col]
        valid = s.notna().to_numpy()
        if valid.sum() >= 20 and y_uni_std > 0:
            s_val = s.to_numpy(dtype="float32")[valid]
            y_val_sub = y_uni_f64[valid]
            if np.min(s_val) != np.max(s_val) and np.min(y_val_sub) != np.max(y_val_sub):
                ranks = pd.Series(s_val).rank(method="average").to_numpy(dtype="float64")
                r_mean = float(np.mean(ranks))
                y_mean = float(np.mean(y_val_sub))
                cov = float(np.mean((ranks - r_mean) * (y_val_sub - y_mean)))
                denom = float(np.std(ranks) * np.std(y_val_sub))
                rho = abs(cov / denom) if denom > 1e-12 else 0.0
                univar_score[col] = rho if np.isfinite(rho) else 0.0
            else:
                univar_score[col] = 0.0
        else:
            univar_score[col] = 0.0

    blocks: Dict[int, List[str]] = {}
    for col in num_candidates:
        if col.startswith("C") and col[1:].isdigit():
            b_key = 1001
        elif col.startswith("D"):
            b_key = 1002
        elif col.startswith("V") and col[1:].isdigit():
            mask_arr = train_df[col].isna().to_numpy(dtype=np.uint8)
            b_key = int(pd.util.hash_array(mask_arr).sum())
        else:
            b_key = 1003
        blocks.setdefault(b_key, []).append(col)

    dropped_by_corr: Dict[str, str] = {}

    def _prune_block_once(cols_in_block: List[str]) -> List[str]:
        if len(cols_in_block) <= 1:
            return cols_in_block
        ordered = sorted(
            cols_in_block,
            key=lambda c: (-round(univar_score.get(c, 0.0), 4), na_rate_map.get(c, 1.0), c),
        )
        corr_mat = _compute_spearman_corr_matrix_fast(train_df[ordered])
        kept: List[str] = []
        for c in ordered:
            is_redundant = False
            for k_col in kept:
                val = corr_mat.loc[c, k_col]
                if np.isfinite(val) and val >= corr_threshold:
                    dropped_by_corr[c] = f"Collinear with {k_col} (|rho|={val:.3f} >= {corr_threshold})"
                    is_redundant = True
                    break
            if not is_redundant:
                kept.append(c)
        return kept

    kept_num_after_tier2: List[str] = []
    for _, blk_cols in blocks.items():
        pass1_cols = _prune_block_once(blk_cols)
        pass2_cols = _prune_block_once(pass1_cols)
        if len(pass2_cols) > 1:
            c_mat = _compute_spearman_corr_matrix_fast(train_df[pass2_cols]).to_numpy(copy=True)
            np.fill_diagonal(c_mat, 0.0)
            max_rem_corr = float(np.nanmax(c_mat)) if np.isfinite(c_mat).any() else 0.0
            assert max_rem_corr < corr_threshold, f"Còn cặp tương quan {max_rem_corr:.3f} >= {corr_threshold}!"
        kept_num_after_tier2.extend(pass2_cols)

    kept_num_set = set(kept_num_after_tier2)
    after_tier2_cols = [c for c in after_tier1_cols if (c not in num_candidates) or (c in kept_num_set)]

    # TẦNG 3: INNER TEMPORAL FOLDS STABILITY & PERMUTATION IMPORTANCE (AP & ROC-AUC)
    sorted_tr = train_df.sort_values("TransactionDT", kind="mergesort").reset_index(drop=True)
    y_sorted = sorted_tr["isFraud"].to_numpy(dtype="int64")

    n_feats_t3 = len(after_tier2_cols)
    X_eval_mat = np.empty((len(sorted_tr), n_feats_t3), dtype="float32")
    for j_idx, col in enumerate(after_tier2_cols):
        if pd.api.types.is_numeric_dtype(sorted_tr[col]) and not pd.api.types.is_bool_dtype(sorted_tr[col]):
            X_eval_mat[:, j_idx] = sorted_tr[col].to_numpy(dtype="float32")
        else:
            X_eval_mat[:, j_idx] = pd.Categorical(sorted_tr[col]).codes.astype("float32")

    quantiles = np.linspace(0.0, 1.0, n_inner_folds + 2)
    dt_bounds = [float(sorted_tr["TransactionDT"].quantile(q)) for q in quantiles]

    fold_gain_matrix = np.zeros((n_inner_folds, n_feats_t3), dtype="float64")
    fold_perm_ap_matrix = np.zeros((n_inner_folds, n_feats_t3), dtype="float64")
    fold_perm_auc_matrix = np.zeros((n_inner_folds, n_feats_t3), dtype="float64")

    rng = np.random.default_rng(random_state)

    for k in range(1, n_inner_folds + 1):
        tr_end, va_end = dt_bounds[k], dt_bounds[k + 1]
        f_tr = (sorted_tr["TransactionDT"] <= tr_end).to_numpy()
        f_va = ((sorted_tr["TransactionDT"] > tr_end) & (sorted_tr["TransactionDT"] <= va_end)).to_numpy()
        if f_tr.sum() < 30 or f_va.sum() < 20 or y_sorted[f_va].sum() == 0:
            continue

        model = FastHistogramGBDT(n_estimators=25, learning_rate=0.15, random_state=random_state + k)
        model.fit(X_eval_mat[f_tr], y_sorted[f_tr])
        assert model.feature_gains_ is not None
        fold_gain_matrix[k - 1, :] = model.feature_gains_

        X_va_full = X_eval_mat[f_va]
        y_va_full = y_sorted[f_va]

        max_perm_rows = 25000
        if len(y_va_full) > max_perm_rows:
            p_step = max(len(y_va_full) // max_perm_rows, 1)
            X_va = X_va_full[::p_step][:max_perm_rows].copy()
            y_va = y_va_full[::p_step][:max_perm_rows]
        else:
            X_va = X_va_full.copy()
            y_va = y_va_full

        base_preds = model.predict_proba_pos(X_va)
        base_ap = compute_average_precision_score(y_va, base_preds)
        base_auc = compute_roc_auc_score(y_va, base_preds)

        active_feat_indices = np.flatnonzero(
            (model.feature_gains_ > 0) | (np.arange(n_feats_t3) < min(60, n_feats_t3))
        )
        for j_idx in active_feat_indices:
            orig_col = X_va[:, j_idx].copy()
            ap_drops, auc_drops = [], []
            for _ in range(n_perm_repeats):
                X_va[:, j_idx] = rng.permutation(orig_col)
                p_perm = model.predict_proba_pos(X_va)
                ap_drops.append(base_ap - compute_average_precision_score(y_va, p_perm))
                auc_drops.append(base_auc - compute_roc_auc_score(y_va, p_perm))
            X_va[:, j_idx] = orig_col
            fold_perm_ap_matrix[k - 1, j_idx] = float(np.mean(ap_drops))
            fold_perm_auc_matrix[k - 1, j_idx] = float(np.mean(auc_drops))

    positive_folds_count = np.sum(
        (fold_gain_matrix > 0) | (fold_perm_ap_matrix > 1e-5) | (fold_perm_auc_matrix > 1e-5),
        axis=0,
    )
    mean_gain = np.mean(fold_gain_matrix, axis=0)
    mean_perm_ap = np.mean(fold_perm_ap_matrix, axis=0)
    mean_perm_auc = np.mean(fold_perm_auc_matrix, axis=0)

    protected_domain_cols = {
        "TransactionAmt",
        "TransactionAmt_log",
        "Amt_cents",
        "is_round_dollar",
        "has_identity",
        "missing_rate_C",
        "missing_rate_D",
        "missing_rate_M",
        "missing_rate_V",
        "missing_rate_id",
        "hour_sin",
        "hour_cos",
        "day_sin",
        "day_cos",
        "email_domain_match",
        "device_brand",
        "os_family",
        "browser_family",
        "card1_cnt_prior",
        "card1_cnt_prior_log",
        "card1_mean_amt_prior",
        "Amt_to_card1_mean_prior",
        "has_card1_history",
        "card1_is_missing",
        "ProductCD",
        "card4",
        "card6",
        "DeviceType",
    }

    selection_rows = []
    final_selected_features: List[str] = []

    t2_idx_map = {col: idx for idx, col in enumerate(after_tier2_cols)}
    tier1_map = tier1_df.set_index("Feature").to_dict(orient="index")

    for col in candidate_cols:
        t1_info = tier1_map.get(col, {})
        if col in constant_cols:
            status = "Dropped_Tier1_Constant"
            group_label = "Low_Value_Dropped"
            reason = "Constant / 100% single value on Temporal Train"
            m_gain, m_ap, m_auc, pos_f = 0.0, 0.0, 0.0, 0
        elif col in dropped_by_corr:
            status = "Dropped_Tier2_Collinear"
            group_label = "Low_Value_Dropped"
            reason = dropped_by_corr[col]
            m_gain, m_ap, m_auc, pos_f = 0.0, 0.0, 0.0, 0
        else:
            idx = t2_idx_map[col]
            m_gain = float(mean_gain[idx])
            m_ap = float(mean_perm_ap[idx])
            m_auc = float(mean_perm_auc[idx])
            pos_f = int(positive_folds_count[idx])
            uni_s = univar_score.get(col, 0.0)

            if pos_f >= 2 or col in protected_domain_cols:
                status = "Selected_Stable"
                group_label = "Stable_Feature"
                reason = f"Active across {pos_f}/{n_inner_folds} temporal folds or core domain feature"
                final_selected_features.append(col)
            elif (pos_f == 1 or uni_s >= 0.03) and (col not in tier1_weak_candidates):
                status = "Selected_Regime_Specific"
                group_label = "Regime_Specific_Feature"
                reason = f"Regime-specific signal (active folds={pos_f}, univar_corr={uni_s:.3f})"
                final_selected_features.append(col)
            else:
                status = "Dropped_Tier3_Low_Importance"
                group_label = "Low_Value_Dropped"
                reason = "Zero/negative temporal permutation importance and weak univariate lift"

        selection_rows.append(
            {
                "Feature": col,
                "Status": status,
                "Stability_Group": group_label,
                "Active_Temporal_Folds": pos_f,
                "Mean_GBDT_Gain": round(m_gain, 4),
                "Mean_Perm_AP_Drop": round(m_ap, 5),
                "Mean_Perm_AUC_Drop": round(m_auc, 5),
                "Conditional_Missing_Rate": t1_info.get("Conditional_Missing_Rate", 0.0),
                "Lift_vs_Complement": t1_info.get("Lift_vs_Complement", 1.0),
                "Decision_Reason": reason,
            }
        )

    assert FORBIDDEN_PREDICTOR_COLS.isdisjoint(set(final_selected_features)), (
        "LỖI NGHIÊM TRỌNG: Có cột cấm (TransactionID/isFraud/TransactionDT/elapsed_day) lọt vào Final Features!"
    )

    selection_report_df = pd.DataFrame(selection_rows)
    return {
        "final_selected_features": final_selected_features,
        "selection_report_df": selection_report_df,
        "ablation_df": ablation_df,
        "kept_d_cols": kept_d_cols,
        "dropped_d_anchors": dropped_d_anchors,
    }


# ------------------------------------------------------------------------------
# 10.4. BA CẤU HÌNH PREPROCESSOR CHUYÊN BIỆT CHO NGƯỜI 3 (LINEAR / SKLEARN_TREE / GBDT)
# ------------------------------------------------------------------------------
class ProductionTabularPreprocessor:
    """
    Bộ tiền xử lý bảng chuẩn sản xuất cho Người 3 với 3 chế độ (model_family):
    1. 'linear'       : Điền Median + cờ _is_missing + log1p(họ C) + RobustScaler + One-Hot.
    2. 'sklearn_tree' : Điền Median + cờ _is_missing + One-Hot, xuất ma trận dense float32.
    3. 'gbdt'         : Giữ nguyên NaN tự nhiên (Native Missing Support) + One-Hot, xuất float32.
    """

    def __init__(
        self,
        selected_features: List[str],
        model_family: str = "gbdt",
        max_missing_indicators: int = 25,
        max_onehot_cardinality: int = 20,
    ) -> None:
        assert model_family in ("linear", "sklearn_tree", "gbdt"), (
            "model_family phải thuộc ('linear', 'sklearn_tree', 'gbdt')"
        )
        self.selected_features = [c for c in selected_features if c not in FORBIDDEN_PREDICTOR_COLS]
        self.model_family = model_family
        self.max_missing_indicators = max_missing_indicators
        self.max_onehot_cardinality = max_onehot_cardinality

        self.num_cols_: List[str] = []
        self.cat_ohe_cols_: List[str] = []
        self.c_log_cols_: List[str] = []
        self.missing_indicator_cols_: List[str] = []
        self.medians_: Dict[str, float] = {}
        self.iqr_scales_: Dict[str, Tuple[float, float]] = {}
        self.ohe_categories_: Dict[str, List[str]] = {}
        self.feature_names_out_: List[str] = []

    def fit(self, train_df: pd.DataFrame) -> "ProductionTabularPreprocessor":
        active_cols = [c for c in self.selected_features if c in train_df.columns]
        if self.model_family == "linear":
            linear_drop = {"n_missing_id", "missing_rate_id", "elapsed_hour", "day_cycle"}
            active_cols = [c for c in active_cols if c not in linear_drop]

        self.num_cols_.clear()
        self.cat_ohe_cols_.clear()

        for col in active_cols:
            s = train_df[col]
            if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s) and col not in (
                "ProductCD",
                "card4",
                "card6",
                "DeviceType",
                "device_brand",
                "os_family",
                "browser_family",
                "P_email_provider",
                "R_email_provider",
                "P_email_tld",
                "R_email_tld",
            ):
                self.num_cols_.append(col)
            else:
                if int(s.nunique(dropna=True)) <= self.max_onehot_cardinality:
                    self.cat_ohe_cols_.append(col)

        self.c_log_cols_ = [
            c for c in self.num_cols_ if c.startswith("C") and c[1:].isdigit()
        ] if self.model_family == "linear" else []

        if self.model_family in ("linear", "sklearn_tree"):
            miss_rates = [(c, float(train_df[c].isna().mean())) for c in self.num_cols_]
            miss_candidates = [c for c, r in sorted(miss_rates, key=lambda x: -x[1]) if 0.0 < r < 0.98]
            self.missing_indicator_cols_ = miss_candidates[: self.max_missing_indicators]
        else:
            self.missing_indicator_cols_ = []

        self.medians_.clear()
        self.iqr_scales_.clear()
        for col in self.num_cols_:
            vals = train_df[col].dropna().to_numpy(dtype="float64")
            if col in self.c_log_cols_:
                vals = np.log1p(np.maximum(vals, 0.0))
            if len(vals) > 0:
                q25, q50, q75 = np.percentile(vals, [25, 50, 75])
                iqr = float(q75 - q25)
                scale = iqr if iqr > 1e-6 else float(np.std(vals))
                scale = scale if scale > 1e-6 else 1.0
                self.medians_[col] = float(q50)
                self.iqr_scales_[col] = (float(q50), scale)
            else:
                self.medians_[col] = 0.0
                self.iqr_scales_[col] = (0.0, 1.0)

        self.ohe_categories_.clear()
        for col in self.cat_ohe_cols_:
            s_str = train_df[col].astype("object").fillna("__MISSING__").astype(str)
            levels = sorted(s_str.unique().tolist())
            self.ohe_categories_[col] = levels

        names: List[str] = list(self.num_cols_)
        for col in self.missing_indicator_cols_:
            names.append(f"{col}_is_missing")
        for col in self.cat_ohe_cols_:
            for lvl in self.ohe_categories_[col]:
                names.append(f"ohe__{col}__{lvl}")
        self.feature_names_out_ = names
        return self

    def get_feature_names_out(self) -> List[str]:
        return list(self.feature_names_out_)

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        n_rows = len(df)
        n_out = len(self.feature_names_out_)
        X_out = np.empty((n_rows, n_out), dtype="float32")
        col_cursor = 0

        for col in self.num_cols_:
            raw = df[col].to_numpy(dtype="float64") if col in df.columns else np.full(n_rows, np.nan)
            if col in self.c_log_cols_:
                raw = np.where(np.isfinite(raw), np.log1p(np.maximum(raw, 0.0)), np.nan)

            if self.model_family == "gbdt":
                X_out[:, col_cursor] = raw.astype("float32")
            elif self.model_family == "sklearn_tree":
                med = self.medians_[col]
                X_out[:, col_cursor] = np.where(np.isfinite(raw), raw, med).astype("float32")
            else:
                med, scale = self.iqr_scales_[col]
                imputed = np.where(np.isfinite(raw), raw, med)
                X_out[:, col_cursor] = ((imputed - med) / scale).astype("float32")
            col_cursor += 1

        for col in self.missing_indicator_cols_:
            is_na = df[col].isna().to_numpy() if col in df.columns else np.ones(n_rows, dtype=bool)
            X_out[:, col_cursor] = is_na.astype("float32")
            col_cursor += 1

        for col in self.cat_ohe_cols_:
            s_str = (
                df[col].astype("object").fillna("__MISSING__").astype(str).to_numpy()
                if col in df.columns
                else np.full(n_rows, "__MISSING__", dtype=object)
            )
            for lvl in self.ohe_categories_[col]:
                X_out[:, col_cursor] = (s_str == lvl).astype("float32")
                col_cursor += 1

        assert col_cursor == n_out, "Lỗi lệch số cột khi transform!"
        return X_out


# ------------------------------------------------------------------------------
# 10.5. BÀI KIỂM THỬ TEST D (LABEL ISOLATION) & TEST E (0 / NAN / UNSEEN & RELOAD)
# ------------------------------------------------------------------------------
def run_test_d_and_test_e(
    fe_df: pd.DataFrame,
    split_df: pd.DataFrame,
    selected_features: List[str],
    preprocessors: Dict[str, ProductionTabularPreprocessor],
    output_dir: Path,
) -> Dict[str, bool]:
    """
    Thực thi 2 bài kiểm thử nghiệm thu cuối cùng trước khi bàn giao cho Người 3:
    - Test D (Validation Label Isolation)
    - Test E (Matrix Semantics 0 / NaN / Unseen Category & Artifact Reload)
    """
    val_mask = (split_df["temporal_role"].to_numpy() == "val")
    train_mask = (split_df["temporal_role"].to_numpy() == "train")

    assert not np.any(val_mask & train_mask), "Test D thất bại: Tập Train và Val bị giao nhau!"
    if len(fe_df) <= 10000:
        perturbed_fe_df = fe_df.copy()
        perturbed_fe_df.loc[val_mask, "isFraud"] = 1 - perturbed_fe_df.loc[val_mask, "isFraud"]
        sel_perturbed = run_three_tier_feature_selection(
            perturbed_fe_df.loc[train_mask],
            corr_threshold=0.95,
            n_inner_folds=5,
            n_perm_repeats=3,
            random_state=42,
        )
        assert sel_perturbed["final_selected_features"] == selected_features, (
            "Test D thất bại: Danh sách biến được chọn bị thay đổi khi hoán vị nhãn Validation!"
        )
    else:
        tr_idx = np.flatnonzero(train_mask)[:2000]
        va_idx = np.flatnonzero(val_mask)[:500]
        mini_idx = np.concatenate([tr_idx, va_idx])
        mini_fe_1 = fe_df.iloc[mini_idx].copy().reset_index(drop=True)
        mini_fe_2 = mini_fe_1.copy()
        mini_val_mask = np.zeros(len(mini_fe_1), dtype=bool)
        mini_val_mask[len(tr_idx):] = True
        mini_train_mask = ~mini_val_mask
        mini_fe_2.loc[mini_val_mask, "isFraud"] = 1 - mini_fe_2.loc[mini_val_mask, "isFraud"]

        assert mini_fe_1.loc[mini_train_mask, "isFraud"].equals(mini_fe_2.loc[mini_train_mask, "isFraud"]), (
            "Test D thất bại: Hoán vị nhãn Validation làm thay đổi nhãn Train!"
        )
        for fam, prep_obj in preprocessors.items():
            X_va_1 = prep_obj.transform(mini_fe_1.loc[mini_val_mask])
            X_va_2 = prep_obj.transform(mini_fe_2.loc[mini_val_mask])
            assert np.allclose(X_va_1, X_va_2, rtol=0.0, atol=0.0, equal_nan=True), (
                f"Test D thất bại: Ma trận Validation nhánh {fam} bị phụ thuộc vào cột mục tiêu isFraud!"
            )

    probe_df = fe_df.loc[val_mask].head(10).copy().reset_index(drop=True)
    num_probe_col = preprocessors["gbdt"].num_cols_[0]
    cat_probe_col = preprocessors["gbdt"].cat_ohe_cols_[0]

    probe_df.loc[0, num_probe_col] = 0.0
    probe_df.loc[1, num_probe_col] = np.nan
    probe_df[cat_probe_col] = probe_df[cat_probe_col].astype("object")
    probe_df.loc[2, cat_probe_col] = "__UNSEEN_CATEGORY_999__"

    X_gbdt_probe = preprocessors["gbdt"].transform(probe_df)
    col_pos_gbdt = preprocessors["gbdt"].get_feature_names_out().index(num_probe_col)
    assert X_gbdt_probe[0, col_pos_gbdt] == 0.0, "Test E (T1) thất bại: Giá trị 0.0 bị biến đổi sai ở nhánh GBDT!"
    assert np.isnan(X_gbdt_probe[1, col_pos_gbdt]), "Test E (T2) thất bại: Giá trị NaN bị mất ở nhánh GBDT!"

    ohe_indices = [
        i
        for i, name in enumerate(preprocessors["gbdt"].get_feature_names_out())
        if name.startswith(f"ohe__{cat_probe_col}__")
    ]
    assert np.all(X_gbdt_probe[2, ohe_indices] == 0.0), (
        "Test E (T4) thất bại: Unseen category không được ánh xạ về vector 0!"
    )

    for fam in ("linear", "sklearn_tree"):
        X_fam = preprocessors[fam].transform(probe_df)
        assert not np.isnan(X_fam).any(), f"Test E thất bại: Nhánh {fam} vẫn còn chứa NaN!"
        assert not np.isinf(X_fam).any(), f"Test E thất bại: Nhánh {fam} chứa giá trị Inf!"

    for fam, prep_obj in preprocessors.items():
        file_path = output_dir / f"preprocessor_{fam}.joblib"
        with open(file_path, "wb") as f_out:
            pickle.dump(prep_obj, f_out)
        with open(file_path, "rb") as f_in:
            reloaded_prep = pickle.load(f_in)
        X_before = prep_obj.transform(probe_df)
        X_after = reloaded_prep.transform(probe_df)
        assert np.allclose(X_before, X_after, rtol=0.0, atol=0.0, equal_nan=True), (
            f"Test E (Reload) thất bại: Kết quả sau khi nạp lại {file_path.name} không khớp!"
        )

    return {
        "Test_D_Validation_Label_Isolation": True,
        "Test_E_Zero_NaN_Unseen_and_Artifact_Reload": True,
    }


# ------------------------------------------------------------------------------
# 10.6. TRỰC QUAN HÓA KẾT QUẢ CHỌN BIẾN & ABLATION (FIGURE 7)
# ------------------------------------------------------------------------------
def generate_step3_figure(
    selection_report_df: pd.DataFrame,
    ablation_df: pd.DataFrame,
    output_dir: Path,
) -> str:
    """Vẽ Figure 7: Biểu đồ Top 15 biến ổn định nhất & Kết quả Ablation họ D."""
    plt.rcParams["font.sans-serif"] = "DejaVu Sans"
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), dpi=150)

    ax = axes[0]
    selected_sub = (
        selection_report_df[selection_report_df["Stability_Group"] != "Low_Value_Dropped"]
        .sort_values(["Active_Temporal_Folds", "Mean_GBDT_Gain"], ascending=[False, False])
        .head(15)
        .iloc[::-1]
    )
    colors = np.where(selected_sub["Stability_Group"] == "Stable_Feature", "#1f77b4", "#ff7f0e")
    ax.barh(selected_sub["Feature"], selected_sub["Mean_GBDT_Gain"], color=colors)
    ax.set_title("A. Top 15 Đặc trưng dẫn đầu qua 5 Inner Temporal Folds", fontsize=11, fontweight="bold")
    ax.set_xlabel("Mean Split Gain trên các Fold Thời gian (Xanh: Stable | Cam: Regime-Specific)")
    ax.grid(axis="x", linestyle="--", alpha=0.4)

    ax = axes[1]
    x_pos = np.arange(len(ablation_df))
    width = 0.36
    b1 = ax.bar(x_pos - width / 2, ablation_df["Mean_Temporal_ROC_AUC"], width=width, color="#2b8cbe", label="Temporal ROC-AUC")
    b2 = ax.bar(x_pos + width / 2, ablation_df["Mean_Temporal_AP"], width=width, color="#e34a33", label="Temporal Average Precision (AP)")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(ablation_df["Configuration"], rotation=15, ha="right")
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2.0, h + 0.01, f"{h:.3f}", ha="center", va="bottom", fontsize=8.5)
    ax.set_ylim(0, 1.12)
    ax.set_title("B. Kết quả Ablation 4 Cấu hình Họ biến D trên Inner Temporal Folds", fontsize=11, fontweight="bold")
    ax.set_ylabel("Điểm đánh giá (Score)")
    ax.legend(frameon=True)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    fig.tight_layout()
    fig_path = output_dir / "fig7_feature_selection_and_stability.png"
    fig.savefig(fig_path)
    plt.close(fig)
    return str(fig_path)


# ------------------------------------------------------------------------------
# 10.7. HÀM CHẠY TRỌN GÓI BƯỚC 3 & BƯỚC 4 (XUẤT ĐỦ HỢP ĐỒNG BÀN GIAO CHO NGƯỜI 3)
# ------------------------------------------------------------------------------
def run_step3_and_step4_pipeline(
    fe_df: pd.DataFrame,
    split_df: pd.DataFrame,
    output_dir: str = str(STEP34_OUTPUT_DIR),
) -> Dict[str, object]:
    """
    Thực thi trọn gói Bước 3 (Feature Selection 3 tầng + Ablation họ D)
    và Bước 4 (Đóng gói 3 Preprocessor + Kiểm thử Test D, Test E + Xuất Hợp đồng bàn giao)
    tại OUTPUT_DIR / 'artifacts_step3_4_handover'.
    """
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    train_mask = (split_df["temporal_role"].to_numpy() == "train")
    val_mask = (split_df["temporal_role"].to_numpy() == "val")
    train_fe_df = fe_df.loc[train_mask]
    val_fe_df = fe_df.loc[val_mask]

    # 1. Chạy Bước 3: Lựa chọn đặc trưng 3 tầng chỉ trên tập Temporal Train
    fs_res = run_three_tier_feature_selection(
        train_fe_df,
        corr_threshold=0.95,
        n_inner_folds=5,
        n_perm_repeats=3,
        random_state=42,
    )
    selected_features: List[str] = fs_res["final_selected_features"]
    selection_report_df: pd.DataFrame = fs_res["selection_report_df"]
    ablation_df: pd.DataFrame = fs_res["ablation_df"]

    # 2. Chạy Bước 4: Khởi tạo và Fit 3 cấu hình Preprocessor chỉ trên tập Temporal Train
    preprocessors: Dict[str, ProductionTabularPreprocessor] = {}
    matrix_shapes: Dict[str, Dict[str, Tuple[int, int]]] = {}
    matrix_checksums: Dict[str, str] = {}

    for fam in ("linear", "sklearn_tree", "gbdt"):
        prep = ProductionTabularPreprocessor(selected_features=selected_features, model_family=fam)
        prep.fit(train_fe_df)
        preprocessors[fam] = prep

        X_tr_mat = prep.transform(train_fe_df)
        X_va_mat = prep.transform(val_fe_df)
        matrix_shapes[fam] = {"train_shape": X_tr_mat.shape, "val_shape": X_va_mat.shape}
        matrix_checksums[fam] = hashlib.sha256(np.nan_to_num(X_va_mat, nan=-999999.0).tobytes()).hexdigest()[:16]

    # 3. Thực thi bộ kiểm thử nghiệm thu cuối cùng (Test D & Test E)
    test_de_results = run_test_d_and_test_e(
        fe_df=fe_df,
        split_df=split_df,
        selected_features=selected_features,
        preprocessors=preprocessors,
        output_dir=out_path,
    )

    # 4. Vẽ biểu đồ tổng hợp Bước 3 (Figure 7) và xuất toàn bộ tệp bàn giao cho Người 3
    fig7_path = generate_step3_figure(selection_report_df, ablation_df, output_dir=out_path)

    selection_report_df.to_csv(out_path / "selection_and_feature_dictionary.csv", index=False)
    ablation_df.to_csv(out_path / "fs_d_ablation_results.csv", index=False)
    try:
        split_df.to_parquet(out_path / "split_indices.parquet", index=False)
    except Exception:
        split_df.to_csv(out_path / "split_indices.csv", index=False)

    final_features_payload = {
        "primary_split_protocol": "timestamp_safe_temporal_80_20",
        "secondary_benchmark_protocol": "stratified_80_20_seed_42",
        "primary_evaluation_metrics": ["ROC-AUC", "Average Precision (AP)"],
        "forbidden_excluded_columns": sorted(list(FORBIDDEN_PREDICTOR_COLS)),
        "n_candidate_features_before_selection": len(selection_report_df),
        "n_final_selected_raw_features": len(selected_features),
        "stability_breakdown": selection_report_df["Stability_Group"].value_counts().to_dict(),
        "selected_raw_features": selected_features,
        "transformed_feature_names_by_branch": {
            fam: prep.get_feature_names_out() for fam, prep in preprocessors.items()
        },
        "transformed_matrix_shapes": matrix_shapes,
        "validation_matrix_sha256_prefix": matrix_checksums,
        "unit_tests_passed": test_de_results,
        "ram_usage_stats_mb": {
            "fe_df_memory_mb": round(get_df_memory_mb(fe_df), 2),
            **get_process_rss_mb(),
        },
    }
    with open(out_path / "final_features.json", "w", encoding="utf-8") as f:
        json.dump(final_features_payload, f, indent=2, ensure_ascii=False)

    requirements_text = (
        "numpy>=1.24.0\n"
        "pandas>=2.0.0\n"
        "scikit-learn>=1.3.0\n"
        "lightgbm>=4.0.0\n"
        "xgboost>=2.0.0\n"
        "matplotlib>=3.7.0\n"
        "pyarrow>=12.0.0\n"
    )
    (out_path / "requirements.txt").write_text(requirements_text, encoding="utf-8")

    print("=" * 78)
    print("HOÀN TẤT BƯỚC 3 & BƯỚC 4: FEATURE SELECTION, UNIT TESTS (D, E) & BÀN GIAO")
    print("=" * 78)
    print(f"- Tổng số biến ứng viên đầu vào Bước 3 : {len(selection_report_df)} biến")
    print(f"- Số biến được giữ lại (Final Features): {len(selected_features)} biến")
    print(f"- Phân nhóm độ ổn định                 : {final_features_payload['stability_breakdown']}")
    print("\n[Bảng 3.1] Kết quả Thí nghiệm Ablation 4 cấu hình cho họ biến D (D1-D15 vs D_anchor):")
    print(ablation_df.to_string(index=False))
    print("\n[Bảng 3.2] Kích thước ma trận đầu ra của 3 Cấu hình Preprocessor bàn giao cho Người 3:")
    for fam, shp in matrix_shapes.items():
        print(
            f"  + preprocessor_{fam:<12}: Train = {shp['train_shape']} | "
            f"Val = {shp['val_shape']} | SHA256 = {matrix_checksums[fam]}"
        )
    print("\n- Kết quả Kiểm thử Tự động Bước 3 & 4:")
    for t_name, passed in test_de_results.items():
        print(f"  + {t_name}: {'PASSED' if passed else 'FAILED'}")
    print(
        f"- Bộ nhớ Tiến trình cuối Bước 4: Current RSS = {final_features_payload['ram_usage_stats_mb']['current_rss_mb']:.2f} MB | "
        f"Peak RSS = {final_features_payload['ram_usage_stats_mb']['peak_rss_mb']:.2f} MB"
    )
    print(f"\nĐã đóng gói đầy đủ bộ 6 nhóm file bàn giao cho Người 3 tại: {out_path}")

    return {
        "selected_features": selected_features,
        "selection_report_df": selection_report_df,
        "ablation_df": ablation_df,
        "preprocessors": preprocessors,
        "final_features_payload": final_features_payload,
        "fig7_path": fig7_path,
    }

print("Đã khởi tạo xong toàn bộ hàm và class của Bước 3 & 4 (step3_4_selection_and_handover).")

### CELL 11: Thực thi Bước 3 & 4 và Tổng kết Toàn bộ Pipeline (Step 0 -> Step 4)


In [ ]:
# ==============================================================================
# CELL 11: THỰC THI BƯỚC 3 & 4 VÀ TỔNG KẾT BÀN GIAO TOÀN BỘ PIPELINE (STEP 0 -> STEP 4)
# ==============================================================================
# Giải thích:
# - Chạy hàm run_step3_and_step4_pipeline() trên fe_df của Bước 2 và split_df của Bước 0.
# - Xuất toàn bộ tệp bàn giao (final_features.json, 3 file preprocessor_*.joblib,
#   selection_and_feature_dictionary.csv, fs_d_ablation_results.csv, fig7) vào:
#   `/content/drive/MyDrive/IEEE_CIS_Project/artifacts/artifacts_step3_4_handover`
# - In báo cáo tổng kết cuối cùng theo đúng định dạng của Notebook đích.

print("\n========== RUNNING STEP 3 & STEP 4 ==========")
print("Lưu ý: Bước này tính toán trên nhiều cột nên có thể mất 5-15 phút, vui lòng đợi...")

step34_res = run_step3_and_step4_pipeline(
    fe_df=step2_res["fe_df"],
    split_df=step0_res["split_df"],
    output_dir=str(OUTPUT_DIR / "artifacts_step3_4_handover"),
)
print("STEP 3 & STEP 4 COMPLETED!")

# ==============================================================================
# TỔNG KẾT TOÀN BỘ PIPELINE (THEO ĐÚNG FORMAT ĐÍCH)
# ==============================================================================
print("\n" + "=" * 75)
print("      IEEE-CIS FRAUD DETECTION PIPELINE")
print("              STEP 0 -> STEP 4")
print("=" * 75)

print("\n[STEP 0] Setup & Temporal Split: COMPLETED")
print("[STEP 1] Advanced EDA: COMPLETED")
print("[STEP 2] Feature Engineering: COMPLETED")
print("[STEP 3] Feature Selection: COMPLETED")
print("[STEP 4] Preprocessor & Handover: COMPLETED")

print("\n========== FINAL RESULTS ==========")
print("Number of selected features:", len(step34_res["selected_features"]))

print("\nAblation Results:")
print(step34_res["ablation_df"].to_string(index=False))

print(
    "\nFeature Selection Report:",
    len(step34_res["selection_report_df"]),
    "rows",
)
print("\nFigure 7:", step34_res["fig7_path"])
print("\nHandover directory:", OUTPUT_DIR / "artifacts_step3_4_handover")

print("\n" + "=" * 75)
print("      PIPELINE STEP 0 -> STEP 4 FINISHED")
print("=" * 75)

gc.collect()

# Cell 12: Inspect DataFrame memory usage

**Explanation:** Identifies the largest Pandas DataFrames currently stored in memory. This helps detect potential sources of high RAM consumption before proceeding with model training.

**Code overview:** The code checks all global variables, calculates the memory usage and shape of each DataFrame, sorts them from largest to smallest, and displays the 15 largest DataFrames.

In [ ]:

import pandas as pd

dataframes = []

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        memory_mb = obj.memory_usage(deep=True).sum() / (1024 ** 2)
        dataframes.append((name, memory_mb, obj.shape))

dataframes.sort(key=lambda x: x[1], reverse=True)

print("Largest DataFrames:")
for name, memory_mb, shape in dataframes[:15]:
    print(f"{name}: {memory_mb:.2f} MB | Shape: {shape}")


#  MACHINE LEARNING MODEL TRAINING AND EVALUATION

## Cell 1: Prepare project code in the Colab runtime
**Explanation:** Defines the project directories and prepares a runtime copy of the Python source files. This separates the persistent project files in Drive from files used during execution.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:

import gc
import ctypes

# Run garbage collection
gc.collect()

# Try to return unused memory to the operating system
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

# Display current RAM usage
import psutil
import os

process = psutil.Process(os.getpid())
ram_used = process.memory_info().rss / (1024 ** 3)

print(f"Current RAM usage: {ram_used:.2f} GB")


## Cell 2: Prepare project code in the Colab runtime

**Explanation:** Defines the project directories and prepares a runtime copy of the Python source files. This separates the persistent project files in Drive from files used during execution.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Prepare project code in the Colab runtime.
import os
import shutil
import ast

project_path = "/content/drive/MyDrive/IEEE_CIS_Project"
drive_code = os.path.join(project_path, "code")
runtime_code = "/content/ieee_cis_code"

os.makedirs(runtime_code, exist_ok=True)

files = [
    "step0_setup_and_split.py",
    "step1_advanced_eda.py",
    "step2_feature_engineering.py",
    "step3_4_selection_and_handover.py"
]

for filename in files:
    source = os.path.join(drive_code, "Copy of " + filename)
    destination = os.path.join(runtime_code, filename)

    shutil.copy2(source, destination)

print("Đã sao chép code vào Colab:")
for filename in os.listdir(runtime_code):
    print("-", filename)

# Kiểm tra các hàm và class trong Step 2
step2_file = os.path.join(runtime_code, "step2_feature_engineering.py")

with open(step2_file, "r", encoding="utf-8") as f:
    tree = ast.parse(f.read())

print("\n===== STEP 2: FUNCTIONS AND CLASSES =====")
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        print("Function:", node.name)
    elif isinstance(node, ast.ClassDef):
        print("Class:", node.name)

print("\n===== STEP 2: IMPORTS =====")
for node in tree.body:
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        print(ast.unparse(node))

## Cell 3: Inspect feature-engineering source code

**Explanation:** Parses the Step 2 Python module with the AST library and inspects its top-level definitions. This is a code-inspection step; it does not run feature engineering.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect feature-engineering source code.
import ast

step2_file = "/content/ieee_cis_code/step2_feature_engineering.py"

with open(step2_file, "r", encoding="utf-8") as f:
    source = f.read()

tree = ast.parse(source)

for node in tree.body:
    if isinstance(node, ast.FunctionDef) and node.name == "run_step2_feature_engineering":
        print("===== FUNCTION SIGNATURE =====")
        print(ast.unparse(node.args))

        print("\n===== FUNCTION BODY =====")
        print(ast.get_source_segment(source, node))

## Cell 4: Inspect data setup and split source code

**Explanation:** Parses the Step 0 module and inspects its top-level definitions to review the data preparation and splitting implementation.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect data setup and split source code.
import ast

step0_file = "/content/ieee_cis_code/step0_setup_and_split.py"

with open(step0_file, "r", encoding="utf-8") as f:
    source0 = f.read()

tree0 = ast.parse(source0)

for node in tree0.body:
    if isinstance(node, ast.FunctionDef):
        if node.name in [
            "run_step0_pipeline",
            "get_ieee_cis_canonical_schema"
        ]:
            print("\n" + "=" * 70)
            print("FUNCTION:", node.name)
            print("SIGNATURE:")
            print(ast.unparse(node.args))
            print("BODY:")
            print(ast.get_source_segment(source0, node))

## Cell 5: Load saved split indices

**Explanation:** Reads the saved split-index artifact and displays its contents or summary so the notebook can verify which records belong to each data split.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Load saved split indices.
import pandas as pd
import os

project_path = "/content/drive/MyDrive/IEEE_CIS_Project"

split_path = os.path.join(
    project_path,
    "artifacts",
    "artifacts_step0",
    "split_indices.parquet"
)

split_df = pd.read_parquet(split_path)

print("===== SPLIT DATA =====")
print("Shape:", split_df.shape)
print("\nColumns:")
print(split_df.columns.tolist())

print("\n===== TEMPORAL ROLE =====")
print(split_df["temporal_role"].value_counts(dropna=False))

print("\n===== FIRST 5 ROWS =====")
display(split_df.head())

## Cell 6: Inspect key operations in the Step 0 source

**Explanation:** Searches the Step 0 source file for important operations and prints matching lines for implementation review.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect key operations in the Step 0 source.
step0_file = "/content/ieee_cis_code/step0_setup_and_split.py"

with open(step0_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

keywords = ["merge(", "has_identity", "Amt_cents", "TransactionID"]

for i, line in enumerate(lines):
    if any(k in line for k in keywords):
        start = max(0, i - 3)
        end = min(len(lines), i + 5)

        print(f"\n===== Lines {start+1}-{end} =====")
        for j in range(start, end):
            print(f"{j+1}: {lines[j].rstrip()}")

## Cell 7: Inspect selected Step 0 functions

**Explanation:** Uses Python's AST to locate and display selected function definitions from the Step 0 module without executing the module again.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect selected Step 0 functions.
import ast

step0_file = "/content/ieee_cis_code/step0_setup_and_split.py"

with open(step0_file, "r", encoding="utf-8") as f:
    source = f.read()

tree = ast.parse(source)

target_functions = [
    "extract_pre_downcast_features",
    "run_step0_pipeline"
]

for node in tree.body:
    if isinstance(node, ast.FunctionDef) and node.name in target_functions:
        print("\n" + "=" * 70)
        print("FUNCTION:", node.name)
        print(ast.get_source_segment(source, node))

## Cell 8: Check runtime and library environment

**Explanation:** Displays the Python and key package versions, together with system information, to document the environment used for model development.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Check runtime and library environment.
import sys
import psutil
import pandas as pd
import numpy as np
import sklearn
import joblib

print("===== PYTHON & LIBRARIES =====")
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

print("\n===== RAM =====")
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")
print(f"Used RAM: {ram.percent}%")

## Cell 9: Review imports used by project modules

**Explanation:** Parses the project source files and collects their import statements. This helps identify the external libraries required by the pipeline.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Review imports used by project modules.
import ast
import os

code_path = "/content/ieee_cis_code"

files = [
    "step0_setup_and_split.py",
    "step2_feature_engineering.py"
]

imports = set()

for filename in files:
    path = os.path.join(code_path, filename)

    with open(path, "r", encoding="utf-8") as f:
        tree = ast.parse(f.read())

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for item in node.names:
                imports.add(item.name.split(".")[0])

        elif isinstance(node, ast.ImportFrom) and node.module:
            imports.add(node.module.split(".")[0])

print("===== REQUIRED LIBRARIES =====")
for lib in sorted(imports):
    print("-", lib)

## Cell 10: Load and merge transaction datasets

**Explanation:** Loads the transaction and identity CSV files and combines the relevant records for subsequent preprocessing. The resulting data frame is the starting point for the modeling pipeline.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Load and merge transaction datasets.
import os
import gc
import pandas as pd

project_path = "/content/drive/MyDrive/IEEE_CIS_Project"
data_path = os.path.join(project_path, "data")

print("Đang đọc train_transaction.csv...")
train_transaction = pd.read_csv(
    os.path.join(data_path, "train_transaction.csv")
)

print("Đang đọc train_identity.csv...")
train_identity = pd.read_csv(
    os.path.join(data_path, "train_identity.csv")
)

print("\n===== ORIGINAL SHAPES =====")
print("Transaction:", train_transaction.shape)
print("Identity:", train_identity.shape)

print("\nĐang ghép dữ liệu...")
train_df = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print("\n===== MERGED DATA =====")
print("Shape:", train_df.shape)
print("Unique TransactionID:", train_df["TransactionID"].nunique())
print("Identity coverage:",
      round(train_df["DeviceType"].notna().mean() * 100, 2), "%")

del train_transaction, train_identity
gc.collect()

print("\nĐã giải phóng hai bảng trung gian.")

## Cell 11: Create Step 0 features

**Explanation:** Imports and applies the feature-extraction function from Step 0 to the training data, producing the features required by the later pipeline stages.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Create Step 0 features.
import sys

sys.path.insert(0, "/content/ieee_cis_code")

from step0_setup_and_split import extract_pre_downcast_features

train_df = extract_pre_downcast_features(train_df)

print("===== STEP 0 FEATURES =====")
print("Shape:", train_df.shape)
print("\nIdentity coverage:")
print(train_df["has_identity"].value_counts(normalize=True) * 100)

print("\nAmt_cents:")
print(train_df["Amt_cents"].describe())

print("\nMissing required columns:")
print(train_df[["has_identity", "Amt_cents"]].isna().sum())

## Cell 12: Verify split consistency

**Explanation:** Checks that the loaded training data and saved split information have consistent row counts and transaction identifiers before continuing.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Verify split consistency.
import numpy as np

print("===== SPLIT CONSISTENCY CHECK =====")

print("Rows in train_df:", len(train_df))
print("Rows in split_df:", len(split_df))

print("TransactionID identical:",
      np.array_equal(
          train_df["TransactionID"].to_numpy(),
          split_df["TransactionID"].to_numpy()
      ))

print("TransactionDT identical:",
      np.array_equal(
          train_df["TransactionDT"].to_numpy(),
          split_df["TransactionDT"].to_numpy()
      ))

print("isFraud identical:",
      np.array_equal(
          train_df["isFraud"].to_numpy(),
          split_df["isFraud"].to_numpy()
      ))

print("\nTemporal split:")
print(split_df["temporal_role"].value_counts())

print("\nStratified split:")
print(split_df["stratified_role"].value_counts())

## Cell 13: Reduce dataframe memory usage

**Explanation:** Applies the project's canonical numerical and categorical schema and safe downcasting procedure to reduce memory consumption while preserving the data structure.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Reduce dataframe memory usage.
from step0_setup_and_split import (
    get_ieee_cis_canonical_schema,
    downcast_dataframe_safely
)

num_cols, cat_cols = get_ieee_cis_canonical_schema()

train_df, mem_stats = downcast_dataframe_safely(
    train_df,
    num_cols=num_cols,
    cat_cols=cat_cols
)

print("===== DOWNCAST RESULT =====")
print("Shape:", train_df.shape)
print("Memory before:", mem_stats["df_memory_before_mb"], "MB")
print("Memory after:", mem_stats["df_memory_after_mb"], "MB")
print("Memory saved:", mem_stats["memory_saved_pct"], "%")

## Cell 14: Inspect Step 3–4 handover artifacts

**Explanation:** Lists the files produced by feature selection and handover, including their names and sizes, to confirm that the required artifacts are available.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect Step 3–4 handover artifacts.
import os

handover_path = os.path.join(
    project_path,
    "artifacts",
    "artifacts_step3_4_handover"
)

print("===== HANDOVER FILES =====")

for filename in os.listdir(handover_path):
    filepath = os.path.join(handover_path, filename)
    print(filename, "-", round(os.path.getsize(filepath) / (1024**2), 2), "MB")

## Cell 15: Load model-specific preprocessors

**Explanation:** Imports the preprocessing implementation and loads the saved preprocessing objects for the Linear, tree-based, and GBDT model families.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Load model-specific preprocessors.
import sys
import joblib
import os

sys.path.insert(0, "/content/ieee_cis_code")

# Import module chứa class ProductionTabularPreprocessor
import step3_4_selection_and_handover

# Load 3 preprocessor
preprocessor_linear = joblib.load(
    os.path.join(handover_path, "preprocessor_linear.joblib")
)

preprocessor_tree = joblib.load(
    os.path.join(handover_path, "preprocessor_sklearn_tree.joblib")
)

preprocessor_gbdt = joblib.load(
    os.path.join(handover_path, "preprocessor_gbdt.joblib")
)

print("===== PREPROCESSORS LOADED =====")
print("Linear:", type(preprocessor_linear))
print("Tree:", type(preprocessor_tree))
print("GBDT:", type(preprocessor_gbdt))

## Cell 16: Inspect preprocessor capabilities

**Explanation:** Checks the available methods and attributes of each loaded preprocessor to understand how it transforms data and exposes selected features.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect preprocessor capabilities.
preprocessors = {
    "Linear": preprocessor_linear,
    "Sklearn Tree": preprocessor_tree,
    "GBDT": preprocessor_gbdt
}

for name, prep in preprocessors.items():
    print(f"\n===== {name} =====")
    print("Methods:", [
        method for method in dir(prep)
        if not method.startswith("_") and callable(getattr(prep, method))
    ])
    print("Attributes:", [
        attr for attr in vars(prep).keys()
    ])
    print("Input feature count:", len(prep.feature_names_in_)
          if hasattr(prep, "feature_names_in_") else "Not available")

# Cell 17 & 18: Restore the feature-engineered DataFrame

**Explanation:** Restores the feature-engineered DataFrame from Step 2 using the existing step2_res["fe_df"] variable. This avoids rerunning Feature Engineering and creating unnecessary memory usage.

**Code overview:** The code assigns the existing DataFrame to fe_df and displays its shape to verify that the data has been successfully restored.văn bản in đậm

In [ ]:
# Restore feature-engineered DataFrame from Step 2
fe_df = step2_res["fe_df"]

print("Feature-engineered DataFrame restored!")
print("Shape:", fe_df.shape)

## Cell 19: Validate selected features

**Explanation:** Reviews the selected feature lists for each model family and checks whether the selected columns are available in the feature-engineered data.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Validate selected features.
preprocessors = {
    "Linear": preprocessor_linear,
    "Sklearn Tree": preprocessor_tree,
    "GBDT": preprocessor_gbdt
}

for name, prep in preprocessors.items():
    selected = prep.selected_features
    missing = [col for col in selected if col not in fe_df.columns]

    print(f"\n===== {name} =====")
    print("Model family:", prep.model_family)
    print("Selected features:", len(selected))
    print("Numerical columns:", len(prep.num_cols_))
    print("Categorical OHE columns:", len(prep.cat_ohe_cols_))
    print("Missing indicator columns:", len(prep.missing_indicator_cols_))
    print("Output features:", len(prep.feature_names_out_))
    print("Missing input columns:", len(missing))
    if missing:
        print("Missing examples:", missing[:10])

## Cell 20: Test preprocessing transformations

**Explanation:** Transforms a sample of the feature-engineered data with each preprocessor and reports the resulting matrix information to validate the transformation stage.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Test preprocessing transformations.
preprocessors = {
    "Linear": preprocessor_linear,
    "Sklearn Tree": preprocessor_tree,
    "GBDT": preprocessor_gbdt
}

for name, prep in preprocessors.items():
    sample_X = prep.transform(fe_df.iloc[:1000])

    print(f"\n===== {name} =====")
    print("Output type:", type(sample_X))
    print("Output shape:", sample_X.shape)
    print("Expected columns:", len(prep.feature_names_out_))

## Cell 21: Prepare model input matrices

**Explanation:** Creates the model-input output directory and prepares the transformed training and validation matrices and target labels for model training.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Prepare model input matrices.
import numpy as np
import os
import gc

# Tạo thư mục lưu dữ liệu đầu vào cho mô hình
model_input_path = os.path.join(
    project_path,
    "artifacts",
    "artifacts_step5_model_inputs"
)
os.makedirs(model_input_path, exist_ok=True)

# Lấy vị trí các dòng Train và Validation
train_pos = np.flatnonzero(
    split_df["temporal_role"].to_numpy() == "train"
)
val_pos = np.flatnonzero(
    split_df["temporal_role"].to_numpy() == "val"
)

# Lưu nhãn tương ứng
y_train = train_df["isFraud"].to_numpy()[train_pos]
y_val = train_df["isFraud"].to_numpy()[val_pos]

np.save(os.path.join(model_input_path, "y_train.npy"), y_train)
np.save(os.path.join(model_input_path, "y_val.npy"), y_val)

# Các preprocessor
preprocessors = {
    "linear": preprocessor_linear,
    "tree": preprocessor_tree,
    "gbdt": preprocessor_gbdt
}

batch_size = 20000

for name, prep in preprocessors.items():

    n_features = len(prep.feature_names_out_)

    X_train = np.lib.format.open_memmap(
        os.path.join(model_input_path, f"X_{name}_train.npy"),
        mode="w+",
        dtype=np.float32,
        shape=(len(train_pos), n_features)
    )

    X_val = np.lib.format.open_memmap(
        os.path.join(model_input_path, f"X_{name}_val.npy"),
        mode="w+",
        dtype=np.float32,
        shape=(len(val_pos), n_features)
    )

    for positions, output in [
        (train_pos, X_train),
        (val_pos, X_val)
    ]:
        for start in range(0, len(positions), batch_size):
            batch_pos = positions[start:start + batch_size]

            X_batch = prep.transform(fe_df.iloc[batch_pos])

            output[start:start + len(batch_pos)] = X_batch.astype(
                np.float32
            )

    X_train.flush()
    X_val.flush()

    print(f"\n===== {name.upper()} =====")
    print("Train shape:", X_train.shape)
    print("Validation shape:", X_val.shape)

    del X_train, X_val
    gc.collect()

print("\nALL TRANSFORMATIONS COMPLETED!")

## Cell 22: Check target class distribution

**Explanation:** Counts the legitimate and fraudulent observations in the training and validation targets. This documents the class imbalance before model fitting.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Check target class distribution.
import numpy as np
import os

print("===== LABEL DISTRIBUTION =====")

for name, y in [("Train", y_train), ("Validation", y_val)]:
    unique, counts = np.unique(y, return_counts=True)

    print(f"\n{name}:")
    for label, count in zip(unique, counts):
        print(f"Class {label}: {count:,} ({count / len(y) * 100:.3f}%)")

print("\n===== MATRIX VALIDATION =====")

for name, n_features in [
    ("linear", 415),
    ("tree", 418),
    ("gbdt", 393)
]:
    print(f"\n{name.upper()}")

    for split, n_rows in [
        ("train", len(y_train)),
        ("val", len(y_val))
    ]:
        path = os.path.join(
            model_input_path,
            f"X_{name}_{split}.npy"
        )

        X = np.load(path, mmap_mode="r")

        print(f"{split}: shape={X.shape}")

        assert X.shape == (n_rows, n_features)

        # Kiểm tra NaN và Inf theo từng batch
        all_finite = True

        for start in range(0, n_rows, 50000):
            batch = X[start:start + 50000]

            if not np.isfinite(batch).all():
                all_finite = False
                break

        print("All values finite:", all_finite)

print("\nALL CHECKS COMPLETED!")

## Cell 23: Validate GBDT input arrays

**Explanation:** Loads the saved GBDT matrices in memory-mapped mode and checks for invalid numerical values such as NaN and infinity.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Validate GBDT input arrays.
import numpy as np
import os

for split in ["train", "val"]:

    path = os.path.join(
        model_input_path,
        f"X_gbdt_{split}.npy"
    )

    X = np.load(path, mmap_mode="r")
    nan_count = 0
    inf_count = 0
    bad_columns = {}

    for start in range(0, X.shape[0], 20000):
        batch = X[start:start + 20000]

        nan_mask = np.isnan(batch)
        inf_mask = np.isinf(batch)

        nan_count += nan_mask.sum()
        inf_count += inf_mask.sum()

        column_counts = np.sum(nan_mask | inf_mask, axis=0)

        for j in np.flatnonzero(column_counts):
            bad_columns[j] = (
                bad_columns.get(j, 0) + int(column_counts[j])
            )

    print(f"\n===== GBDT {split.upper()} =====")
    print("NaN count:", nan_count)
    print("Inf count:", inf_count)

    print("Affected columns:")
    for j, count in sorted(
        bad_columns.items(),
        key=lambda x: x[1],
        reverse=True
    )[:20]:
        print(
            j,
            prep_name if False else preprocessor_gbdt.feature_names_out_[j],
            count
        )

## Cell 24: Inspect GBDT preprocessing configuration

**Explanation:** Displays key configuration details from the GBDT preprocessor, including its feature and missing-value handling settings.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect GBDT preprocessing configuration.
import inspect

print("===== GBDT PREPROCESSOR CONFIG =====")
print("Model family:", preprocessor_gbdt.model_family)
print("Missing indicator columns:", len(preprocessor_gbdt.missing_indicator_cols_))
print("Numerical columns:", len(preprocessor_gbdt.num_cols_))
print("Categorical columns:", len(preprocessor_gbdt.cat_ohe_cols_))

print("\n===== TRANSFORM IMPLEMENTATION =====")
print(inspect.getsource(type(preprocessor_gbdt).transform))

## Cell 25: Check machine-learning package availability

**Explanation:** Checks whether the LightGBM, XGBoost, and scikit-learn packages are installed in the current runtime.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Check machine-learning package availability.
import importlib.util

print("===== MODEL LIBRARIES =====")

for lib in ["lightgbm", "xgboost", "sklearn"]:
    print(lib, ":", importlib.util.find_spec(lib) is not None)

## Cell 26: Record machine-learning package versions

**Explanation:** Imports the modeling libraries and prints their versions so the software environment can be documented.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Record machine-learning package versions.
import lightgbm as lgb
import xgboost as xgb
import sklearn

print("LightGBM:", lgb.__version__)
print("XGBoost:", xgb.__version__)
print("Scikit-learn:", sklearn.__version__)

## Cell 27: Train and save the DummyClassifier baseline

**Explanation:** Fits a simple most-frequent-class baseline and evaluates its basic behavior. This baseline provides a reference point for the more advanced classifiers.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Train and save the DummyClassifier baseline.
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
import numpy as np
import pandas as pd

# Load labels if they are not already in memory
y_train = np.load(os.path.join(model_input_path, "y_train.npy"))
y_val = np.load(os.path.join(model_input_path, "y_val.npy"))

# Train baseline model
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(np.zeros((len(y_train), 1)), y_train)

# Predict
dummy_pred = dummy_model.predict(np.zeros((len(y_val), 1)))
dummy_proba = dummy_model.predict_proba(
    np.zeros((len(y_val), 1))
)[:, 1]

# Evaluate
dummy_results = {
    "Model": "Dummy Classifier",
    "Accuracy": accuracy_score(y_val, dummy_pred),
    "Precision": precision_score(y_val, dummy_pred, zero_division=0),
    "Recall": recall_score(y_val, dummy_pred, zero_division=0),
    "F1": f1_score(y_val, dummy_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, dummy_proba),
    "PR-AUC": average_precision_score(y_val, dummy_proba)
}

print("===== DUMMY CLASSIFIER RESULTS =====")
print(pd.DataFrame([dummy_results]).to_string(index=False))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, dummy_pred))

## Cell 28: Train Logistic Regression

**Explanation:** Fits the Logistic Regression classifier using the prepared linear-model matrices, measures training performance and calculates validation predictions and evaluation metrics.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Train Logistic Regression.
import numpy as np
import os
import joblib
import time

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Load preprocessed data
X_train = np.load(
    os.path.join(model_input_path, "X_linear_train.npy"),
    mmap_mode="r"
)
X_val = np.load(
    os.path.join(model_input_path, "X_linear_val.npy"),
    mmap_mode="r"
)

# Train model
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
    solver="lbfgs"
)

start_time = time.time()
lr_model.fit(X_train, y_train)
train_time = time.time() - start_time

# Predict
lr_pred = lr_model.predict(X_val)
lr_proba = lr_model.predict_proba(X_val)[:, 1]

# Evaluate
lr_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_val, lr_pred),
    "Precision": precision_score(y_val, lr_pred, zero_division=0),
    "Recall": recall_score(y_val, lr_pred, zero_division=0),
    "F1": f1_score(y_val, lr_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, lr_proba),
    "PR-AUC": average_precision_score(y_val, lr_proba),
    "Training Time (s)": train_time
}

print("===== LOGISTIC REGRESSION RESULTS =====")
print(lr_results)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, lr_pred))

# Save model
os.makedirs(os.path.join(project_path, "outputs"), exist_ok=True)
joblib.dump(
    lr_model,
    os.path.join(project_path, "outputs", "logistic_regression.joblib")
)

print("\nModel saved successfully.")

## Cell 29: Check Logistic Regression convergence

**Explanation:** Displays the number of iterations used by Logistic Regression and checks whether the optimization reached the configured iteration limit.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Check Logistic Regression convergence.
print("Iterations used:", lr_model.n_iter_)
print("Max iterations:", lr_model.max_iter)
print("Converged:", lr_model.n_iter_[0] < lr_model.max_iter)

## Cell 30: Train an alternative Logistic Regression configuration

**Explanation:** Fits a Logistic Regression model using the SAGA solver and records its training time and evaluation results for comparison.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Train an alternative Logistic Regression configuration.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *
import time
import joblib
import os

lr_saga = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
    solver="saga",
    tol=1e-3
)

start = time.time()
lr_saga.fit(X_train, y_train)
saga_time = time.time() - start

saga_pred = lr_saga.predict(X_val)
saga_proba = lr_saga.predict_proba(X_val)[:, 1]

print("===== LOGISTIC REGRESSION SAGA =====")
print("Iterations:", lr_saga.n_iter_)
print("Converged:", lr_saga.n_iter_[0] < lr_saga.max_iter)
print("Training time:", round(saga_time, 2), "seconds")

print("\nAccuracy:", accuracy_score(y_val, saga_pred))
print("Precision:", precision_score(y_val, saga_pred, zero_division=0))
print("Recall:", recall_score(y_val, saga_pred, zero_division=0))
print("F1:", f1_score(y_val, saga_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_val, saga_proba))
print("PR-AUC:", average_precision_score(y_val, saga_proba))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, saga_pred))

## Cell 31: Train the selected Logistic Regression configuration

**Explanation:** Fits the LBFGS Logistic Regression configuration and records its training time and model results.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Train the selected Logistic Regression configuration.
lr_lbfgs_v2 = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
    solver="lbfgs",
    tol=1e-3
)

start = time.time()
lr_lbfgs_v2.fit(X_train, y_train)
lbfgs_v2_time = time.time() - start

pred = lr_lbfgs_v2.predict(X_val)
proba = lr_lbfgs_v2.predict_proba(X_val)[:, 1]

print("===== LOGISTIC REGRESSION LBFGS V2 =====")
print("Iterations:", lr_lbfgs_v2.n_iter_)
print("Converged:", lr_lbfgs_v2.n_iter_[0] < lr_lbfgs_v2.max_iter)
print("Training time:", round(lbfgs_v2_time, 2), "seconds")

print("Accuracy:", accuracy_score(y_val, pred))
print("Precision:", precision_score(y_val, pred, zero_division=0))
print("Recall:", recall_score(y_val, pred, zero_division=0))
print("F1:", f1_score(y_val, pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_val, proba))
print("PR-AUC:", average_precision_score(y_val, proba))

print("Confusion Matrix:")
print(confusion_matrix(y_val, pred))

## Cell 32: Save the Logistic Regression model

**Explanation:** Serializes the selected Logistic Regression model to the project's outputs directory so it can be loaded later without retraining.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Save the Logistic Regression model.
import os
import joblib

save_path = os.path.join(
    project_path, "outputs", "logistic_regression.joblib"
)

joblib.dump(lr_lbfgs_v2, save_path)

print("Saved:", save_path)

## Cell 33: Release Logistic Regression memory

**Explanation:** Deletes large linear-model matrices and temporary model objects that are no longer needed, then runs garbage collection to free memory.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Release Logistic Regression memory.
import gc

# Xóa các ma trận Linear đang chiếm RAM
del X_train
del X_val

# Xóa các model Logistic Regression không còn cần trong RAM
del lr_lbfgs_v2
del lr_saga

gc.collect()

print("Garbage collection completed.")

## Cell 34: Monitor available system memory

**Explanation:** Uses the system memory utility to display current memory usage after the Logistic Regression stage.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Monitor available system memory.
!free -h

## Cell 35: Load Decision Tree input matrices

**Explanation:** Loads the prepared tree-based training and validation matrices and the corresponding target labels for Decision Tree training.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Load Decision Tree input matrices.
import os
import numpy as np

X_train = np.load(
    os.path.join(model_input_path, "X_tree_train.npy"),
    mmap_mode="r"
)

X_val = np.load(
    os.path.join(model_input_path, "X_tree_val.npy"),
    mmap_mode="r"
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

## Cell 36: Train and evaluate Decision Tree

**Explanation:** Fits a Decision Tree classifier and calculates predictions, probabilities, training time, and classification metrics on the validation data.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Train and evaluate Decision Tree.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)
import time

dt_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

start = time.time()
dt_model.fit(X_train, y_train)
dt_time = time.time() - start

pred = dt_model.predict(X_val)
proba = dt_model.predict_proba(X_val)[:, 1]

print("===== DECISION TREE =====")
print("Training time:", round(dt_time, 2), "seconds")

print("Accuracy:", accuracy_score(y_val, pred))
print("Precision:", precision_score(y_val, pred, zero_division=0))
print("Recall:", recall_score(y_val, pred, zero_division=0))
print("F1:", f1_score(y_val, pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_val, proba))
print("PR-AUC:", average_precision_score(y_val, proba))

print("Confusion Matrix:")
print(confusion_matrix(y_val, pred))

## Cell 37: Save the Decision Tree model

**Explanation:** Serializes the trained Decision Tree model to the outputs directory for later evaluation and reuse.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Save the Decision Tree model.
import os
import joblib

save_path = os.path.join(
    project_path, "outputs", "decision_tree.joblib"
)

joblib.dump(dt_model, save_path)

print("Saved:", save_path)

## Cell 38: Release Decision Tree memory

**Explanation:** Removes the tree-model matrices, estimator, and temporary predictions from memory, then runs garbage collection.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Release Decision Tree memory.
import gc

# Xóa dữ liệu Tree khỏi RAM
del X_train
del X_val

# Xóa mô hình Decision Tree khỏi RAM
del dt_model

# Xóa kết quả dự đoán tạm thời
del pred
del proba

gc.collect()

print("RAM cleanup completed.")

## Cell 39: Monitor memory after Decision Tree

**Explanation:** Displays system memory usage after the Decision Tree stage to monitor the resources available for the next model.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Monitor memory after Decision Tree.
!free -h

## Cell 40: Reload tree-based matrices

**Explanation:** Loads the tree-based training and validation arrays again in memory-mapped mode so they can be used by the Random Forest stage.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Reload tree-based matrices.
import os
import numpy as np

X_train = np.load(
    os.path.join(model_input_path, "X_tree_train.npy"),
    mmap_mode="r"
)

X_val = np.load(
    os.path.join(model_input_path, "X_tree_val.npy"),
    mmap_mode="r"
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

## Cell 41: Train and evaluate Random Forest

**Explanation:** Fits a Random Forest classifier and evaluates its predictions and probability scores on the validation set.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Train and evaluate Random Forest.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)
import time

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=10,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=2
)

start = time.time()
rf_model.fit(X_train, y_train)
rf_time = time.time() - start

pred = rf_model.predict(X_val)
proba = rf_model.predict_proba(X_val)[:, 1]

print("===== RANDOM FOREST =====")
print("Training time:", round(rf_time, 2), "seconds")

print("Accuracy:", accuracy_score(y_val, pred))
print("Precision:", precision_score(y_val, pred, zero_division=0))
print("Recall:", recall_score(y_val, pred, zero_division=0))
print("F1:", f1_score(y_val, pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_val, proba))
print("PR-AUC:", average_precision_score(y_val, proba))

print("Confusion Matrix:")
print(confusion_matrix(y_val, pred))

## Cell 42: Check available CPU cores

**Explanation:** Displays the number of CPU cores available to the runtime, which is useful when reviewing parallel model-training settings.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Check available CPU cores.
import os

print("CPU cores:", os.cpu_count())

## Cell 43: Save the Random Forest model

**Explanation:** Serializes the trained Random Forest estimator to the outputs directory for subsequent evaluation and feature-importance analysis.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Save the Random Forest model.
import os
import joblib

save_path = os.path.join(
    project_path, "outputs", "random_forest.joblib"
)

joblib.dump(rf_model, save_path)

print("Saved:", save_path)

## Cell 44: Release Random Forest memory

**Explanation:** Deletes the Random Forest estimator, input matrices, and temporary prediction arrays, then runs garbage collection.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Release Random Forest memory.
import gc

del X_train
del X_val
del rf_model
del pred
del proba

gc.collect()

print("RAM cleanup completed.")

## Cell 45: Perform an additional memory cleanup

**Explanation:** Checks whether selected temporary Random Forest variables remain in the runtime and deletes them before collecting unused memory.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Perform an additional memory cleanup.
import gc

for name in ["X_train", "X_val", "rf_model", "pred", "proba"]:
    if name in globals():
        del globals()[name]

gc.collect()

print("RAM cleanup completed.")

## Cell 46: Monitor memory before GBDT training

**Explanation:** Displays system memory usage before loading and training the gradient-boosting models.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Monitor memory before GBDT training.
!free -h

## Cell 47: Load GBDT input matrices

**Explanation:** Loads the prepared GBDT training and validation arrays and their target labels for LightGBM and XGBoost.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Load GBDT input matrices.
import numpy as np
import os

X_train = np.load(
    os.path.join(model_input_path, "X_gbdt_train.npy"),
    mmap_mode="r"
)

X_val = np.load(
    os.path.join(model_input_path, "X_gbdt_val.npy"),
    mmap_mode="r"
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("NaN in train:", np.isnan(X_train).sum())

## Cell 48: Train and evaluate LightGBM

**Explanation:** Fits the LightGBM classifier and calculates validation predictions, probability-based scores, training time, and evaluation metrics.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Train and evaluate LightGBM.
import time
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)

lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    class_weight="balanced",
    random_state=42,
    n_jobs=2,
    verbosity=-1
)

start = time.time()
lgbm_model.fit(X_train, y_train)
training_time = time.time() - start

pred = lgbm_model.predict(X_val)
proba = lgbm_model.predict_proba(X_val)[:, 1]

print("===== LIGHTGBM =====")
print(f"Training time: {training_time:.2f} seconds")
print("Accuracy:", accuracy_score(y_val, pred))
print("Precision:", precision_score(y_val, pred, zero_division=0))
print("Recall:", recall_score(y_val, pred, zero_division=0))
print("F1:", f1_score(y_val, pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_val, proba))
print("PR-AUC:", average_precision_score(y_val, proba))
print("Confusion Matrix:")
print(confusion_matrix(y_val, pred))

## Cell 49: Save the LightGBM model

**Explanation:** Serializes the trained LightGBM model to the outputs directory so it can be reused in the evaluation stage.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Save the LightGBM model.
import os
import joblib

save_path = os.path.join(
    project_path, "outputs", "lightgbm.joblib"
)

joblib.dump(lgbm_model, save_path)

print("Saved:", save_path)

## Cell 50: Release LightGBM memory

**Explanation:** Deletes the LightGBM estimator, input matrices, and temporary predictions, then runs garbage collection.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Release LightGBM memory.
import gc

for name in ["X_train", "X_val", "lgbm_model", "pred", "proba"]:
    if name in globals():
        del globals()[name]

gc.collect()

print("RAM cleanup completed.")

## Cell 51: Monitor memory after LightGBM

**Explanation:** Displays current system memory usage before proceeding to XGBoost.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Monitor memory after LightGBM.
!free -h

## Cell 52: Reload GBDT matrices for XGBoost

**Explanation:** Loads the GBDT training and validation matrices in memory-mapped mode for the XGBoost training stage.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Reload GBDT matrices for XGBoost.
import numpy as np
import os

X_train = np.load(
    os.path.join(model_input_path, "X_gbdt_train.npy"),
    mmap_mode="r"
)

X_val = np.load(
    os.path.join(model_input_path, "X_gbdt_val.npy"),
    mmap_mode="r"
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

## Cell 53: Train and evaluate XGBoost

**Explanation:** Fits the XGBoost classifier, accounts for class imbalance through its configured weighting, and calculates validation metrics.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Train and evaluate XGBoost.
import time
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)

# Handle class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    n_jobs=2,
    random_state=42,
    eval_metric="logloss"
)

start = time.time()
xgb_model.fit(X_train, y_train)
training_time = time.time() - start

pred = xgb_model.predict(X_val)
proba = xgb_model.predict_proba(X_val)[:, 1]

print("===== XGBOOST =====")
print(f"Training time: {training_time:.2f} seconds")
print("Accuracy:", accuracy_score(y_val, pred))
print("Precision:", precision_score(y_val, pred, zero_division=0))
print("Recall:", recall_score(y_val, pred, zero_division=0))
print("F1:", f1_score(y_val, pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_val, proba))
print("PR-AUC:", average_precision_score(y_val, proba))
print("Confusion Matrix:")
print(confusion_matrix(y_val, pred))

## Cell 54: Save the XGBoost model

**Explanation:** Serializes the trained XGBoost model to the outputs directory for later comparison and analysis.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Save the XGBoost model.
import os
import joblib

save_path = os.path.join(
    project_path, "outputs", "xgboost.joblib"
)

joblib.dump(xgb_model, save_path)

print("Saved:", save_path)

## Cell 55: Release XGBoost memory

**Explanation:** Deletes the XGBoost estimator, input matrices, and temporary predictions, then runs garbage collection.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Release XGBoost memory.
import gc

for name in [
    "X_train", "X_val", "xgb_model",
    "pred", "proba"
]:
    if name in globals():
        del globals()[name]

gc.collect()

print("RAM cleanup completed.")

## Cell 56: Audit saved model files

**Explanation:** Lists the saved model files and reports their sizes to check that the trained estimators were written to disk.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Audit saved model files.
import os

output_path = os.path.join(project_path, "outputs")

print("===== SAVED MODEL FILES =====")

for file in sorted(os.listdir(output_path)):
    if file.endswith(".joblib"):
        size_mb = os.path.getsize(
            os.path.join(output_path, file)
        ) / (1024 ** 2)

        print(f"{file}: {size_mb:.2f} MB")

## Cell 57: Verify the saved Logistic Regression model

**Explanation:** Checks that the Logistic Regression model file exists, reports its file size, and attempts to load the saved estimator.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Verify the saved Logistic Regression model.
import os
import joblib

lr_path = os.path.join(
    project_path, "outputs", "logistic_regression.joblib"
)

print("File exists:", os.path.exists(lr_path))
print("Exact size:", os.path.getsize(lr_path), "bytes")

lr_model = joblib.load(lr_path)

print("Loaded successfully:", type(lr_model))
print("Iterations:", lr_model.n_iter_)
print("Converged:", lr_model.n_iter_[0] < lr_model.max_iter)

## Cell 58: Test DummyClassifier serialization

**Explanation:** Fits a temporary DummyClassifier on a tiny artificial dataset and saves it to check model serialization. The next cell replaces this temporary baseline with one fitted on the actual training labels.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Test DummyClassifier serialization.
import os
import joblib
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit([[0], [1]], [0, 0])

save_path = os.path.join(
    project_path, "outputs", "dummy_classifier.joblib"
)

joblib.dump(dummy_model, save_path)

print("Saved:", save_path)
print("Strategy:", dummy_model.strategy)

## Cell 59: Fit and save the actual DummyClassifier baseline

**Explanation:** Fits the most-frequent-class baseline using the real training labels and saves it. This is the baseline model used in the later model comparison.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Fit and save the actual DummyClassifier baseline.
import os
import joblib
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(
    y_train.to_numpy().reshape(-1, 1) if hasattr(y_train, "to_numpy") else y_train.reshape(-1, 1),
    y_train
)

save_path = os.path.join(
    project_path, "outputs", "dummy_classifier.joblib"
)

joblib.dump(dummy_model, save_path)

print("Saved:", save_path)
print("Classes:", dummy_model.classes_)
print("Class probabilities:", dummy_model.class_prior_)
print("Training samples:", len(y_train))

## Cell 60: Verify DummyClassifier behavior and saved file

**Explanation:** Checks the fitted class labels, the model's prediction, and whether the serialized baseline file exists and has a non-zero size.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Verify DummyClassifier behavior and saved file.
print("Classes:", dummy_model.classes_)
print("Most frequent class:", dummy_model.predict([[0]]))
print("Saved file exists:", os.path.exists(save_path))
print("File size:", os.path.getsize(save_path), "bytes")

## Cell 61: Check DummyClassifier class prior

**Explanation:** Compares the class prior stored by the baseline model with the fraud rate in the training labels.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Check DummyClassifier class prior.
print("Class prior:", dummy_model.class_prior_)
print("Train fraud rate:", y_train.mean())

## Cell 62: Validate evaluation inputs and saved models

**Explanation:** Checks the validation target and verifies that the required model files and validation matrices are available before running the combined evaluation.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Validate evaluation inputs and saved models.
import os

print("===== VALIDATION LABELS =====")
print("y_val shape:", y_val.shape)
print("Fraud rate:", y_val.mean())

print("\n===== MODEL FILES =====")
output_path = os.path.join(project_path, "outputs")

for name in [
    "dummy_classifier.joblib",
    "logistic_regression.joblib",
    "decision_tree.joblib",
    "random_forest.joblib",
    "lightgbm.joblib",
    "xgboost.joblib"
]:
    path = os.path.join(output_path, name)
    print(name, "->", os.path.exists(path))

print("\n===== VALIDATION MATRICES =====")
for name in [
    "X_linear_val.npy",
    "X_tree_val.npy",
    "X_gbdt_val.npy"
]:
    path = os.path.join(model_input_path, name)
    print(name, "->", os.path.exists(path))

## Cell 63: Compare all classification models

**Explanation:** Loads the saved baseline and trained classifiers, calculates the main validation metrics, and saves a consolidated model-comparison table.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Compare all classification models.
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

output_path = os.path.join(project_path, "outputs")

# Load validation matrices
X_linear_val = np.load(
    os.path.join(model_input_path, "X_linear_val.npy"),
    mmap_mode="r"
)

X_tree_val = np.load(
    os.path.join(model_input_path, "X_tree_val.npy"),
    mmap_mode="r"
)

X_gbdt_val = np.load(
    os.path.join(model_input_path, "X_gbdt_val.npy"),
    mmap_mode="r"
)

# Define model and corresponding validation data
models = {
    "DummyClassifier": (
        "dummy_classifier.joblib", None
    ),
    "Logistic Regression": (
        "logistic_regression.joblib", X_linear_val
    ),
    "Decision Tree": (
        "decision_tree.joblib", X_tree_val
    ),
    "Random Forest": (
        "random_forest.joblib", X_tree_val
    ),
    "LightGBM": (
        "lightgbm.joblib", X_gbdt_val
    ),
    "XGBoost": (
        "xgboost.joblib", X_gbdt_val
    )
}

results = []

for model_name, (filename, X_val_model) in models.items():

    model = joblib.load(
        os.path.join(output_path, filename)
    )

    if model_name == "DummyClassifier":
        pred = model.predict(np.zeros((len(y_val), 1)))
        proba = model.predict_proba(
            np.zeros((len(y_val), 1))
        )[:, 1]
    else:
        pred = model.predict(X_val_model)
        proba = model.predict_proba(X_val_model)[:, 1]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred, zero_division=0),
        "F1": f1_score(y_val, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_val, proba),
        "PR_AUC": average_precision_score(y_val, proba)
    })

    print("Completed:", model_name)

# Create comparison table
comparison_df = pd.DataFrame(results)

# Save CSV
csv_path = os.path.join(
    output_path, "model_comparison.csv"
)

comparison_df.to_csv(csv_path, index=False)

print("\n===== MODEL COMPARISON =====")
print(comparison_df.to_string(index=False))

print("\nSaved:", csv_path)

## Cell 64: Calculate confusion matrices

**Explanation:** Generates confusion matrices for the evaluated models and saves or displays the counts of correct and incorrect predictions for each class.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Calculate confusion matrices.
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix

output_path = os.path.join(project_path, "outputs")

models = {
    "DummyClassifier": ("dummy_classifier.joblib", None),
    "Logistic Regression": ("logistic_regression.joblib", X_linear_val),
    "Decision Tree": ("decision_tree.joblib", X_tree_val),
    "Random Forest": ("random_forest.joblib", X_tree_val),
    "LightGBM": ("lightgbm.joblib", X_gbdt_val),
    "XGBoost": ("xgboost.joblib", X_gbdt_val)
}

confusion_results = []

for model_name, (filename, X_val_model) in models.items():

    model = joblib.load(os.path.join(output_path, filename))

    if model_name == "DummyClassifier":
        X_input = np.zeros((len(y_val), 1))
    else:
        X_input = X_val_model

    pred = model.predict(X_input)

    tn, fp, fn, tp = confusion_matrix(
        y_val, pred, labels=[0, 1]
    ).ravel()

    confusion_results.append({
        "Model": model_name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

confusion_df = pd.DataFrame(confusion_results)

confusion_path = os.path.join(
    output_path, "confusion_matrices.csv"
)

confusion_df.to_csv(confusion_path, index=False)

print("===== CONFUSION MATRICES =====")
print(confusion_df.to_string(index=False))

print("\nSaved:", confusion_path)

## Cell 65: Visualize model performance comparison

**Explanation:** Loads the model-comparison results and creates visual comparisons of the evaluation metrics to make differences between classifiers easier to interpret.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Visualize model performance comparison.
import os
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import numpy as np

output_path = os.path.join(project_path, "outputs")

# Load comparison results
comparison_df = pd.read_csv(
    os.path.join(output_path, "model_comparison.csv")
)

confusion_df = pd.read_csv(
    os.path.join(output_path, "confusion_matrices.csv")
)

# ===== FIGURE 1: MODEL PERFORMANCE =====
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ["F1", "PR_AUC"]

for ax, metric in zip(axes, metrics):
    sorted_df = comparison_df.sort_values(metric)

    ax.barh(sorted_df["Model"], sorted_df[metric])
    ax.set_title(f"Model Comparison - {metric}")
    ax.set_xlabel(metric)
    ax.set_xlim(0, 1)
    ax.grid(axis="x", alpha=0.3)

plt.tight_layout()

performance_path = os.path.join(
    output_path, "model_performance_comparison.png"
)

plt.savefig(performance_path, dpi=300, bbox_inches="tight")
plt.show()

# ===== FIGURE 2: CONFUSION MATRICES =====
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (_, row) in zip(axes.flat, confusion_df.iterrows()):

    cm = np.array([
        [row["TN"], row["FP"]],
        [row["FN"], row["TP"]]
    ])

    sns.heatmap(
        cm,
        annot=True,
        fmt=",",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=["Non-Fraud", "Fraud"],
        yticklabels=["Non-Fraud", "Fraud"]
    )

    ax.set_title(row["Model"])
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("Actual Label")

plt.tight_layout()

confusion_path = os.path.join(
    output_path, "confusion_matrices.png"
)

plt.savefig(confusion_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", performance_path)
print("Saved:", confusion_path)

## Cell 66: Inspect saved preprocessing objects

**Explanation:** Loads the saved preprocessors and checks their available feature information to support the interpretation of model inputs.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect saved preprocessing objects.
import os
import joblib
import numpy as np

handover_path = os.path.join(
    project_path, "artifacts", "artifacts_step3_4_handover"
)

preprocessor_files = {
    "Random Forest": "preprocessor_sklearn_tree.joblib",
    "LightGBM": "preprocessor_gbdt.joblib",
    "XGBoost": "preprocessor_gbdt.joblib"
}

model_files = {
    "Random Forest": "random_forest.joblib",
    "LightGBM": "lightgbm.joblib",
    "XGBoost": "xgboost.joblib"
}

for name in model_files:

    model = joblib.load(
        os.path.join(output_path, model_files[name])
    )

    preprocessor = joblib.load(
        os.path.join(handover_path, preprocessor_files[name])
    )

    importance = model.feature_importances_

    print(f"\n===== {name} =====")
    print("Model feature count:", len(importance))
    print("Preprocessor type:", type(preprocessor).__name__)

    try:
        feature_names = preprocessor.get_feature_names_out()
        print("Feature names count:", len(feature_names))
        print("First 10 feature names:", feature_names[:10])
        print("Feature count matches:", len(feature_names) == len(importance))
    except Exception as e:
        print("Cannot extract feature names:", repr(e))

## Cell 67: Analyze feature importance

**Explanation:** Loads the tree-based models and their feature names, extracts feature-importance values, and produces ranked feature-importance outputs or plots.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Analyze feature importance.
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt

output_path = os.path.join(project_path, "outputs")

handover_path = os.path.join(
    project_path, "artifacts", "artifacts_step3_4_handover"
)

model_configs = {
    "Random Forest": {
        "model": "random_forest.joblib",
        "preprocessor": "preprocessor_sklearn_tree.joblib"
    },
    "LightGBM": {
        "model": "lightgbm.joblib",
        "preprocessor": "preprocessor_gbdt.joblib"
    },
    "XGBoost": {
        "model": "xgboost.joblib",
        "preprocessor": "preprocessor_gbdt.joblib"
    }
}

importance_tables = {}

for model_name, config in model_configs.items():

    model = joblib.load(
        os.path.join(output_path, config["model"])
    )

    preprocessor = joblib.load(
        os.path.join(handover_path, config["preprocessor"])
    )

    feature_names = preprocessor.get_feature_names_out()
    importances = model.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })

    importance_df = importance_df.sort_values(
        "Importance", ascending=False
    )

    importance_tables[model_name] = importance_df

    # Top 20
    top20 = importance_df.head(20).sort_values("Importance")

    plt.figure(figsize=(10, 7))
    plt.barh(top20["Feature"], top20["Importance"])
    plt.title(f"Top 20 Feature Importance - {model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()

    filename = (
        model_name.lower().replace(" ", "_")
        + "_feature_importance.png"
    )

    plt.savefig(
        os.path.join(output_path, filename),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(f"\n===== TOP 20 - {model_name} =====")
    print(importance_df.head(20).to_string(index=False))

# Save all feature importance values
for model_name, df in importance_tables.items():

    filename = (
        model_name.lower().replace(" ", "_")
        + "_feature_importance.csv"
    )

    df.to_csv(
        os.path.join(output_path, filename),
        index=False
    )

print("\nAll feature importance files saved.")

## Cell 68: Plot precision–recall curves

**Explanation:** Loads the saved models, calculates precision and recall across classification thresholds, and plots their precision–recall curves for the imbalanced fraud-detection task.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Plot precision–recall curves.
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

output_path = os.path.join(project_path, "outputs")

models = {
    "DummyClassifier": ("dummy_classifier.joblib", None),
    "Logistic Regression": ("logistic_regression.joblib", X_linear_val),
    "Decision Tree": ("decision_tree.joblib", X_tree_val),
    "Random Forest": ("random_forest.joblib", X_tree_val),
    "LightGBM": ("lightgbm.joblib", X_gbdt_val),
    "XGBoost": ("xgboost.joblib", X_gbdt_val)
}

plt.figure(figsize=(10, 7))

pr_results = []

for model_name, (filename, X_val_model) in models.items():

    model = joblib.load(
        os.path.join(output_path, filename)
    )

    if model_name == "DummyClassifier":
        X_input = np.zeros((len(y_val), 1))
    else:
        X_input = X_val_model

    y_scores = model.predict_proba(X_input)[:, 1]

    precision, recall, thresholds = precision_recall_curve(
        y_val, y_scores
    )

    ap = average_precision_score(y_val, y_scores)

    plt.plot(
        recall,
        precision,
        label=f"{model_name} (AP = {ap:.4f})"
    )

    pr_results.append({
        "Model": model_name,
        "Average_Precision": ap
    })

# Random baseline = fraud prevalence
fraud_rate = np.mean(y_val)

plt.axhline(
    y=fraud_rate,
    linestyle="--",
    label=f"Random baseline ({fraud_rate:.4f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves - Model Comparison")

plt.xlim(0, 1)
plt.ylim(0, 1)

plt.grid(alpha=0.3)
plt.legend(loc="best")
plt.tight_layout()

pr_path = os.path.join(
    output_path, "precision_recall_curves.png"
)

plt.savefig(pr_path, dpi=300, bbox_inches="tight")
plt.show()

# Save AP comparison
pr_df = pd.DataFrame(pr_results)

pr_df.to_csv(
    os.path.join(output_path, "pr_curve_metrics.csv"),
    index=False
)

print("\n===== PRECISION-RECALL RESULTS =====")
print(pr_df.to_string(index=False))

print("\nFraud prevalence baseline:", fraud_rate)
print("Saved:", pr_path)

## Cell 69: Analyze classification thresholds

**Explanation:** Evaluates each model at a range of decision thresholds and records how precision, recall, and F1-score change as the threshold changes.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Analyze classification thresholds.
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

output_path = os.path.join(project_path, "outputs")

models = {
    "Logistic Regression": ("logistic_regression.joblib", X_linear_val),
    "Decision Tree": ("decision_tree.joblib", X_tree_val),
    "Random Forest": ("random_forest.joblib", X_tree_val),
    "LightGBM": ("lightgbm.joblib", X_gbdt_val),
    "XGBoost": ("xgboost.joblib", X_gbdt_val)
}

thresholds = np.arange(0.1, 1.0, 0.1)

threshold_results = []

for model_name, (filename, X_val_model) in models.items():

    model = joblib.load(
        os.path.join(output_path, filename)
    )

    y_scores = model.predict_proba(X_val_model)[:, 1]

    for threshold in thresholds:

        y_pred = (y_scores >= threshold).astype(int)

        threshold_results.append({
            "Model": model_name,
            "Threshold": round(threshold, 1),
            "Precision": precision_score(
                y_val, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_val, y_pred, zero_division=0
            ),
            "F1": f1_score(
                y_val, y_pred, zero_division=0
            )
        })

threshold_df = pd.DataFrame(threshold_results)

threshold_path = os.path.join(
    output_path, "threshold_analysis.csv"
)

threshold_df.to_csv(threshold_path, index=False)

print("===== THRESHOLD ANALYSIS =====")

for model_name in models:

    print(f"\n--- {model_name} ---")

    print(
        threshold_df[
            threshold_df["Model"] == model_name
        ].round(4).to_string(index=False)
    )

print("\nSaved:", threshold_path)

## Cell 70: Visualize threshold effects

**Explanation:** Reads the threshold-analysis results and plots precision, recall, and F1-score against the classification threshold for each model.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Visualize threshold effects.
import os
import pandas as pd
import matplotlib.pyplot as plt

output_path = os.path.join(project_path, "outputs")

threshold_df = pd.read_csv(
    os.path.join(output_path, "threshold_analysis.csv")
)

metrics = ["Precision", "Recall", "F1"]

for metric in metrics:

    plt.figure(figsize=(10, 6))

    for model_name in threshold_df["Model"].unique():

        subset = threshold_df[
            threshold_df["Model"] == model_name
        ].sort_values("Threshold")

        plt.plot(
            subset["Threshold"],
            subset[metric],
            marker="o",
            linewidth=2,
            label=model_name
        )

    plt.xlabel("Classification Threshold")
    plt.ylabel(metric)
    plt.title(f"{metric} vs Classification Threshold")

    plt.xticks(
        threshold_df["Threshold"].unique()
    )

    plt.ylim(0, 1)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    filename = f"threshold_{metric.lower()}_curve.png"

    plt.savefig(
        os.path.join(output_path, filename),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("Saved:", filename)

print("\nAll threshold curves generated.")

## Cell 71: Audit generated evaluation outputs

**Explanation:** Checks whether the expected model files, tables, and figures exist, reports their sizes, and saves an output manifest.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Audit generated evaluation outputs.
import os
import pandas as pd

output_path = os.path.join(project_path, "outputs")

expected_files = [
    "model_comparison.csv",
    "confusion_matrices.csv",
    "model_performance_comparison.png",
    "confusion_matrices.png",
    "random_forest_feature_importance.png",
    "lightgbm_feature_importance.png",
    "xgboost_feature_importance.png",
    "random_forest_feature_importance.csv",
    "lightgbm_feature_importance.csv",
    "xgboost_feature_importance.csv",
    "precision_recall_curves.png",
    "pr_curve_metrics.csv",
    "threshold_analysis.csv",
    "threshold_precision_curve.png",
    "threshold_recall_curve.png",
    "threshold_f1_curve.png",
    "dummy_classifier.joblib",
    "logistic_regression.joblib",
    "decision_tree.joblib",
    "random_forest.joblib",
    "lightgbm.joblib",
    "xgboost.joblib"
]

manifest = []

for filename in expected_files:
    path = os.path.join(output_path, filename)

    manifest.append({
        "File": filename,
        "Exists": os.path.exists(path),
        "Size_MB": (
            round(os.path.getsize(path) / (1024**2), 3)
            if os.path.exists(path) else None
        )
    })

manifest_df = pd.DataFrame(manifest)

print("===== OUTPUT FILE AUDIT =====")
print(manifest_df.to_string(index=False))

print("\nTotal files:", len(manifest_df))
print("Existing:", manifest_df["Exists"].sum())
print("Missing:", (~manifest_df["Exists"]).sum())

manifest_path = os.path.join(
    output_path, "evaluation_output_manifest.csv"
)

manifest_df.to_csv(manifest_path, index=False)

print("\nSaved:", manifest_path)

## Cell 72: Visualize feature-selection summary

**Explanation:** Creates a bar chart showing the counts of stable, regime-specific, and removed features based on the feature-selection results recorded in the notebook.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Visualize feature-selection summary.
import matplotlib.pyplot as plt
import os

# Output directory
output_dir = "/content/drive/MyDrive/IEEE_CIS_Project/outputs"
os.makedirs(output_dir, exist_ok=True)

# Actual feature selection results
categories = [
    "Stable Features",
    "Regime-Specific Features",
    "Low-Value Features (Removed)"
]

counts = [216, 84, 183]

# Create chart
plt.figure(figsize=(10, 6))

bars = plt.bar(categories, counts)

plt.title(
    "Feature Selection Summary",
    fontsize=16,
    fontweight="bold"
)
plt.xlabel("Feature Category", fontsize=12)
plt.ylabel("Number of Features", fontsize=12)

plt.bar_label(bars, padding=4, fontsize=11)

plt.ylim(0, max(counts) + 40)
plt.xticks(rotation=10, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()

# Save figure
save_path = os.path.join(
    output_dir,
    "feature_selection_summary.png"
)

plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

## Cell 73: Visualize DummyClassifier baseline metrics

**Explanation:** Creates a bar chart of the DummyClassifier's baseline Accuracy, Precision, Recall, and F1-score for comparison with the trained models.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Visualize DummyClassifier baseline metrics.

import matplotlib.pyplot as plt
import os

# Baseline performance
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
values = [0.965591, 0.0, 0.0, 0.0]

plt.figure(figsize=(9, 5))

bars = plt.bar(metrics, values)

plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("DummyClassifier Baseline Performance")

for bar, value in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.025,
        f"{value:.4f}",
        ha="center",
        fontsize=10
    )

plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()

output_dir = "/content/drive/MyDrive/IEEE_CIS_Project/outputs"
os.makedirs(output_dir, exist_ok=True)

plt.savefig(
    os.path.join(output_dir, "dummy_classifier_performance.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## Cell 74: Generate Logistic Regression predictions

**Explanation:** Loads the saved Logistic Regression model and produces validation class predictions and fraud probabilities.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Generate Logistic Regression predictions.
import joblib

model_path = "/content/drive/MyDrive/IEEE_CIS_Project/outputs/logistic_regression.joblib"

# Load the trained Logistic Regression model
lr_model = joblib.load(model_path)

# Generate class predictions
lr_predictions = lr_model.predict(X_linear_val)

# Generate fraud probabilities
lr_probabilities = lr_model.predict_proba(X_linear_val)[:, 1]

print("Prediction completed.")
print("Number of predictions:", len(lr_predictions))
print("Number of probabilities:", len(lr_probabilities))

## Cell 75: Calculate Logistic Regression evaluation metrics

**Explanation:** Loads Logistic Regression, generates validation predictions and probabilities, and calculates Accuracy, Precision, Recall, F1-score, ROC-AUC, and PR-AUC.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Calculate Logistic Regression evaluation metrics.
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Load the final Logistic Regression model
model_path = "/content/drive/MyDrive/IEEE_CIS_Project/outputs/logistic_regression.joblib"
lr_model = joblib.load(model_path)

# Generate predictions and probabilities
lr_predictions = lr_model.predict(X_linear_val)
lr_probabilities = lr_model.predict_proba(X_linear_val)[:, 1]

# Calculate evaluation metrics
lr_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_val, lr_predictions),
    "Precision": precision_score(y_val, lr_predictions, zero_division=0),
    "Recall": recall_score(y_val, lr_predictions, zero_division=0),
    "F1-score": f1_score(y_val, lr_predictions, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, lr_probabilities),
    "PR-AUC": average_precision_score(y_val, lr_probabilities)
}

# Display results
pd.DataFrame([lr_results]).round(6)

## Cell 76: Calculate Decision Tree evaluation metrics

**Explanation:** Loads the saved Decision Tree model and calculates the main classification metrics on the validation set.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Calculate Decision Tree evaluation metrics.
import joblib
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Load the trained Decision Tree model
model_path = "/content/drive/MyDrive/IEEE_CIS_Project/outputs/decision_tree.joblib"
dt_model = joblib.load(model_path)

# Generate predictions and probabilities
dt_predictions = dt_model.predict(X_tree_val)
dt_probabilities = dt_model.predict_proba(X_tree_val)[:, 1]

# Calculate evaluation metrics
dt_results = {
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_val, dt_predictions),
    "Precision": precision_score(y_val, dt_predictions, zero_division=0),
    "Recall": recall_score(y_val, dt_predictions, zero_division=0),
    "F1-score": f1_score(y_val, dt_predictions, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, dt_probabilities),
    "PR-AUC": average_precision_score(y_val, dt_probabilities)
}

pd.DataFrame([dt_results]).round(6)

## Cell 77: Calculate Random Forest evaluation metrics

**Explanation:** Loads the saved Random Forest model and calculates the main classification metrics on the validation set.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Calculate Random Forest evaluation metrics.
import joblib
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

rf_model = joblib.load(
    "/content/drive/MyDrive/IEEE_CIS_Project/outputs/random_forest.joblib"
)

pred = rf_model.predict(X_tree_val)
proba = rf_model.predict_proba(X_tree_val)[:, 1]

rf_results = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_val, pred),
    "Precision": precision_score(y_val, pred, zero_division=0),
    "Recall": recall_score(y_val, pred, zero_division=0),
    "F1-score": f1_score(y_val, pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, proba),
    "PR-AUC": average_precision_score(y_val, proba)
}

pd.DataFrame([rf_results]).round(6)

## Cell 78: Calculate LightGBM evaluation metrics

**Explanation:** Loads the saved LightGBM model and calculates the main classification metrics on the validation set.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Calculate LightGBM evaluation metrics.
import joblib
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

# Load the trained LightGBM model
model_path = "/content/drive/MyDrive/IEEE_CIS_Project/outputs/lightgbm.joblib"
lgbm_model = joblib.load(model_path)

# Generate predictions and probabilities
lgbm_predictions = lgbm_model.predict(X_gbdt_val)
lgbm_probabilities = lgbm_model.predict_proba(X_gbdt_val)[:, 1]

# Calculate evaluation metrics
lgbm_results = {
    "Model": "LightGBM",
    "Accuracy": accuracy_score(y_val, lgbm_predictions),
    "Precision": precision_score(y_val, lgbm_predictions, zero_division=0),
    "Recall": recall_score(y_val, lgbm_predictions, zero_division=0),
    "F1-score": f1_score(y_val, lgbm_predictions, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, lgbm_probabilities),
    "PR-AUC": average_precision_score(y_val, lgbm_probabilities)
}

pd.DataFrame([lgbm_results]).round(6)

## Cell 79: Calculate XGBoost evaluation metrics

**Explanation:** Loads the saved XGBoost model and calculates the main classification metrics on the validation set.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Calculate XGBoost evaluation metrics.
import joblib
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

# Load the trained XGBoost model
model_path = "/content/drive/MyDrive/IEEE_CIS_Project/outputs/xgboost.joblib"
xgb_model = joblib.load(model_path)

# Generate predictions and probabilities
xgb_predictions = xgb_model.predict(X_gbdt_val)
xgb_probabilities = xgb_model.predict_proba(X_gbdt_val)[:, 1]

# Calculate evaluation metrics
xgb_results = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_val, xgb_predictions),
    "Precision": precision_score(y_val, xgb_predictions, zero_division=0),
    "Recall": recall_score(y_val, xgb_predictions, zero_division=0),
    "F1-score": f1_score(y_val, xgb_predictions, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, xgb_probabilities),
    "PR-AUC": average_precision_score(y_val, xgb_probabilities)
}

pd.DataFrame([xgb_results]).round(6)

## Cell 80: Compare ROC and precision–recall curves

**Explanation:** Loads the trained classifiers, calculates their validation probability scores, and plots ROC and precision–recall curves for model comparison.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Compare ROC and precision–recall curves.

import os
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

# Load trained models
output_dir = "/content/drive/MyDrive/IEEE_CIS_Project/outputs"

models = {
    "Logistic Regression": (
        joblib.load(os.path.join(output_dir, "logistic_regression.joblib")),
        X_linear_val
    ),
    "Decision Tree": (
        joblib.load(os.path.join(output_dir, "decision_tree.joblib")),
        X_tree_val
    ),
    "Random Forest": (
        joblib.load(os.path.join(output_dir, "random_forest.joblib")),
        X_tree_val
    ),
    "LightGBM": (
        joblib.load(os.path.join(output_dir, "lightgbm.joblib")),
        X_gbdt_val
    ),
    "XGBoost": (
        joblib.load(os.path.join(output_dir, "xgboost.joblib")),
        X_gbdt_val
    )
}

# Generate probability predictions
probabilities = {}

for name, (model, X_val_model) in models.items():
    probabilities[name] = model.predict_proba(X_val_model)[:, 1]

# ==========================================
# 1. ROC CURVE
# ==========================================

plt.figure(figsize=(9, 7))

for name, proba in probabilities.items():
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc_score = roc_auc_score(y_val, proba)

    plt.plot(
        fpr, tpr,
        label=f"{name} (AUC = {auc_score:.4f})"
    )

plt.plot([0, 1], [0, 1], "k--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves of Classification Models")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

roc_path = os.path.join(output_dir, "roc_curves.png")
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.show()

# ==========================================
# 2. PRECISION-RECALL CURVE
# ==========================================

plt.figure(figsize=(9, 7))

for name, proba in probabilities.items():
    precision, recall, _ = precision_recall_curve(y_val, proba)
    ap_score = average_precision_score(y_val, proba)

    plt.plot(
        recall, precision,
        label=f"{name} (AP = {ap_score:.4f})"
    )

# Baseline: positive-class prevalence
baseline = y_val.mean()
plt.axhline(
    y=baseline,
    color="black",
    linestyle="--",
    label=f"Baseline (Prevalence = {baseline:.4f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves of Classification Models")
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.tight_layout()

pr_path = os.path.join(output_dir, "precision_recall_curves.png")
plt.savefig(pr_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", roc_path)
print("Saved:", pr_path)


## Cell 81: Visualize detailed confusion matrices

**Explanation:** Calculates and plots a confusion matrix for each trained classifier so the types of classification errors can be compared.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Visualize detailed confusion matrices.

import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.metrics import confusion_matrix

output_dir = "/content/drive/MyDrive/IEEE_CIS_Project/outputs"

models = {
    "Logistic Regression": (
        joblib.load(os.path.join(output_dir, "logistic_regression.joblib")),
        X_linear_val
    ),
    "Decision Tree": (
        joblib.load(os.path.join(output_dir, "decision_tree.joblib")),
        X_tree_val
    ),
    "Random Forest": (
        joblib.load(os.path.join(output_dir, "random_forest.joblib")),
        X_tree_val
    ),
    "LightGBM": (
        joblib.load(os.path.join(output_dir, "lightgbm.joblib")),
        X_gbdt_val
    ),
    "XGBoost": (
        joblib.load(os.path.join(output_dir, "xgboost.joblib")),
        X_gbdt_val
    )
}

# Calculate confusion matrices
confusion_results = {}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (name, (model, X_val_model)) in enumerate(models.items()):
    predictions = model.predict(X_val_model)

    cm = confusion_matrix(y_val, predictions)
    confusion_results[name] = cm

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Legitimate", "Fraud"],
        yticklabels=["Legitimate", "Fraud"],
        ax=axes[i]
    )

    axes[i].set_title(name)
    axes[i].set_xlabel("Predicted Label")
    axes[i].set_ylabel("Actual Label")

# Remove unused sixth subplot
fig.delaxes(axes[5])

plt.suptitle("Confusion Matrices of Classification Models", fontsize=16)
plt.tight_layout()

cm_path = os.path.join(output_dir, "confusion_matrices_detailed.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

# Display numerical results
cm_table = pd.DataFrame(
    [
        {
            "Model": name,
            "TN": cm[0, 0],
            "FP": cm[0, 1],
            "FN": cm[1, 0],
            "TP": cm[1, 1]
        }
        for name, cm in confusion_results.items()
    ]
)

print(cm_table.to_string(index=False))
print("\nSaved:", cm_path)


## Cell 82: Review tree-based feature importance

**Explanation:** Checks feature-importance output files and extracts the top-ranked features from Random Forest, LightGBM, and XGBoost.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Review tree-based feature importance.

import os
import joblib
import pandas as pd

output_dir = "/content/drive/MyDrive/IEEE_CIS_Project/outputs"

models = {
    "Random Forest": ("random_forest.joblib", X_tree_val),
    "LightGBM": ("lightgbm.joblib", X_gbdt_val),
    "XGBoost": ("xgboost.joblib", X_gbdt_val)
}

# Check existing feature importance figures
print("===== FEATURE IMPORTANCE FIGURES =====")

for filename in [
    "random_forest_feature_importance.png",
    "lightgbm_feature_importance.png",
    "xgboost_feature_importance.png"
]:
    path = os.path.join(output_dir, filename)
    print(f"{filename}: {'Exists' if os.path.exists(path) else 'Missing'}")

# Extract and display top 15 features
print("\n===== TOP 15 IMPORTANT FEATURES =====")

for name, (filename, X_val_model) in models.items():
    model = joblib.load(os.path.join(output_dir, filename))

    # Support both standalone estimators and sklearn pipelines
    estimator = model
    if hasattr(model, "named_steps"):
        estimator = list(model.named_steps.values())[-1]

    if not hasattr(estimator, "feature_importances_"):
        print(f"\n{name}: Feature importance is unavailable.")
        continue

    importances = estimator.feature_importances_

    if hasattr(X_val_model, "columns"):
        feature_names = list(X_val_model.columns)
    else:
        feature_names = [f"Feature_{i}" for i in range(len(importances))]

    if len(feature_names) != len(importances):
        print(f"\n{name}: Feature-name count does not match importance count.")
        continue

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })

    importance_df = importance_df.sort_values(
        by="Importance", ascending=False
    ).head(15)

    print(f"\n===== {name} =====")
    print(importance_df.to_string(index=False))


## Cell 83: Inspect available feature-name variables

**Explanation:** Searches the current runtime variables for lists, data frames, or series that may contain feature names. This is a diagnostic inspection step.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect available feature-name variables.

print("===== POSSIBLE FEATURE NAME VARIABLES =====")

for name, value in globals().copy().items():
    if any(key in name.lower() for key in [
        "feature", "column", "selected", "predictor"
    ]):
        try:
            if isinstance(value, (list, tuple, pd.Index)):
                print(f"\n{name}: {len(value)} items")
                print(list(value[:10]))

            elif isinstance(value, pd.DataFrame):
                print(f"\n{name}: DataFrame {value.shape}")
                print(list(value.columns[:10]))

            elif isinstance(value, pd.Series):
                print(f"\n{name}: Series {len(value)} items")
                print(value.head(10).tolist())

        except Exception:
            pass


## Cell 84: Inspect validation matrix dimensions and feature names

**Explanation:** Displays the type, shape, and available column names of the validation matrices, then checks the selected-feature and feature-name collections.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect validation matrix dimensions and feature names.

print("===== INPUT MATRIX INFORMATION =====")

for name, X in [
    ("X_tree_val", X_tree_val),
    ("X_gbdt_val", X_gbdt_val),
    ("X_linear_val", X_linear_val)
]:
    print(f"\n{name}")
    print("Type:", type(X))
    print("Shape:", X.shape)

    if hasattr(X, "columns"):
        print("First 20 columns:", list(X.columns[:20]))
    else:
        print("No column names available.")

print("\n===== SELECTED FEATURES =====")
print("Number of selected features:", len(selected))
print("First 30 selected features:")
print(selected[:30])

print("\n===== FEATURE NAMES =====")
print("Number of feature_names:", len(feature_names))
print("First 30 feature_names:")
print(feature_names[:30])


## Cell 85: Inspect preprocessing objects and related files

**Explanation:** Searches runtime objects and the output directory for preprocessors, encoders, transformers, and related artifacts. This is a diagnostic inspection step.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect preprocessing objects and related files.

print("===== PREPROCESSING OBJECTS =====")

for name, obj in globals().copy().items():
    if any(key in name.lower() for key in [
        "encoder", "preprocess", "transformer",
        "column", "categorical", "numeric", "mapper"
    ]):
        print(
            name,
            "| Type:", type(obj),
            "| Has get_feature_names_out:",
            hasattr(obj, "get_feature_names_out")
        )

print("\n===== OUTPUT DIRECTORY FILES =====")

for filename in sorted(os.listdir(output_dir)):
    if any(key in filename.lower() for key in [
        "feature", "preprocess", "encoder",
        "column", "transform", "select"
    ]):
        print(filename)


## Cell 86: Retrieve transformed feature names

**Explanation:** Uses each fitted preprocessor to retrieve transformed feature names and, when available, displays the corresponding saved feature-importance table.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Retrieve transformed feature names.

import os
import joblib
import pandas as pd

output_dir = "/content/drive/MyDrive/IEEE_CIS_Project/outputs"

preprocessor_map = {
    "Random Forest": preprocessor_tree,
    "LightGBM": preprocessor_gbdt,
    "XGBoost": preprocessor_gbdt
}

importance_files = {
    "Random Forest": "random_forest_feature_importance.csv",
    "LightGBM": "lightgbm_feature_importance.csv",
    "XGBoost": "xgboost_feature_importance.csv"
}

for name in preprocessor_map:
    print(f"\n===== {name} =====")

    prep = preprocessor_map[name]

    try:
        names = prep.get_feature_names_out()
        print("Number of transformed features:", len(names))
        print("First 20 names:", list(names[:20]))

        csv_path = os.path.join(output_dir, importance_files[name])

        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            print("\nTop 15 feature importance:")
            print(df.head(15).to_string(index=False))

    except Exception as e:
        print("Error:", repr(e))


## Cell 87: Inspect one transformed feature name

**Explanation:** Prints a single transformed feature name by index to verify how the preprocessing pipeline names its output columns.

**Code overview:** The code below performs this step using the variables and artifacts prepared in the preceding cells. Review the displayed output to confirm that the operation completed as expected.

In [ ]:
# Purpose: Inspect one transformed feature name.
print(preprocessor_tree.get_feature_names_out()[21])